In [1]:
!pip install /kaggle/input/cmi-detect-behavior/polars-1.30.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install /kaggle/input/cmi-detect-behavior/scikit_learn-1.5.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

Processing /kaggle/input/cmi-detect-behavior/polars-1.30.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
  Attempting uninstall: polars
    Found existing installation: polars 1.21.0
    Uninstalling polars-1.21.0:
      Successfully uninstalled polars-1.21.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-polars-cu12 25.2.2 requires polars<1.22,>=1.20, but you have polars 1.30.0 which is incompatible.
Processing /kaggle/input/cmi-detect-behavior/scikit_learn-1.5.2-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following

In [2]:
import os, json, joblib, numpy as np, pandas as pd
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from cmi_2025_metric_copy_for_import import CompetitionMetric

import polars as pl
from scipy.spatial.transform import Rotation as R
import math
import sys
sys.path.append('/kaggle/input/cmi-utils-top1/')
from cmi_utils import *

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 1. Configuration

In [3]:
RAW_DIR = Path("/kaggle/input/cmi-detect-behavior-with-sensor-data")

all_features_lstm_dir = Path("/kaggle/input/ensemble-55/exp_334_comb")
all_features_lstm_dir2 = Path("/kaggle/input/ensemble-55/exp_337_comb") 
all_features_attention_dir = Path("/kaggle/input/ensemble-55/exp_338_comb") 
all_features_cnn_dir = Path("/kaggle/input/ensemble-55/exp_339_comb")

all_features_lstm_dir_old = Path("/kaggle/input/ensemble-32/exp_317_comb")
all_features_lstm_dir2_old = Path("/kaggle/input/ensemble-32/exp_289") 
all_features_attention_dir_old = Path("/kaggle/input/ensemble-32/exp_252_comb") 
all_features_cnn_dir_old = Path("/kaggle/input/ensemble-32/exp_315_comb")

imu_features_lstm_dir = Path("/kaggle/input/ensemble-55/exp_230") 
imu_features_lstm_dir2 = Path("/kaggle/input/ensemble-55/exp_272_comb") 
imu_features_attention_dir = Path("/kaggle/input/ensemble-55/exp_254_comb")
imu_features_cnn_dir = Path("/kaggle/input/ensemble-55/exp_267") 

model_all_features_path = "three_branch_mixup_best_f1_fold"
model_imu_features_path = "one_branch_mixup_best_f1_fold"

ACC_COLS = ["acc_x", "acc_y", "acc_z"]
ROT_COLS = ["rot_w", "rot_x", "rot_y", "rot_z"]

# NEW
import timm
imu_specs_dir1 = Path("/kaggle/input/cmi-specs-models/exp_343")
imu_specs_dir2 = Path("/kaggle/input/cmi-specs-models/exp_342")
# /NEW

# 2. Data Helpers

In [4]:
def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
    """
    Handle missing values in quaternion data intelligently
    
    Key insight: Quaternions must have unit length |q| = 1
    If one component is missing, we can reconstruct it from the others
    """
    rot_cleaned = rot_data.copy()
    
    for i in range(len(rot_data)):
        row = rot_data[i]
        missing_count = np.isnan(row).sum()
        
        if missing_count == 0:
            # No missing values, normalize to unit quaternion
            norm = np.linalg.norm(row)
            if norm > 1e-8:
                rot_cleaned[i] = row / norm
            else:
                rot_cleaned[i] = [1.0, 0.0, 0.0, 0.0]  # Identity quaternion
                
        elif missing_count == 1:
            # One missing value, reconstruct using unit quaternion constraint
            # |w|² + |x|² + |y|² + |z|² = 1
            missing_idx = np.where(np.isnan(row))[0][0]
            valid_values = row[~np.isnan(row)]
            
            sum_squares = np.sum(valid_values**2)
            if sum_squares <= 1.0:
                missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                # Choose sign for continuity with previous quaternion
                if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                    if rot_cleaned[i-1, missing_idx] < 0:
                        missing_value = -missing_value
                rot_cleaned[i, missing_idx] = missing_value
                rot_cleaned[i, ~np.isnan(row)] = valid_values
            else:
                rot_cleaned[i] = [1.0, 0.0, 0.0, 0.0]
        else:
            # More than one missing value, use identity quaternion
            rot_cleaned[i] = [1.0, 0.0, 0.0, 0.0]
    
    return rot_cleaned

def compute_world_acceleration(acc: np.ndarray, rot: np.ndarray) -> np.ndarray:
    """
    Convert acceleration from device coordinates to world coordinates
    
    This is the key innovation: normalizing for device orientation
    
    Args:
        acc: acceleration in device coordinates, shape (time_steps, 3) [x, y, z]
        rot: rotation quaternion, shape (time_steps, 4) [w, x, y, z] (normalized)
    
    Returns:
        acc_world: acceleration in world coordinates, shape (time_steps, 3)
        
    Why this matters:
    - Device acceleration depends on how the watch is oriented on the wrist
    - World acceleration is independent of device orientation
    - This helps the model focus on actual hand motion rather than wrist rotation
    """
    try:
        # Convert quaternion format from [w, x, y, z] to [x, y, z, w] for scipy
        rot_scipy = rot[:, [1, 2, 3, 0]]
        
        # Verify quaternions are valid (non-zero norm)
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        # Create rotation object and apply transformation
        r = R.from_quat(rot_scipy)
        acc_world = r.apply(acc)
        
    except Exception:
        # Fallback to original acceleration if transformation fails
        print("Warning: World coordinate transformation failed, using device coordinates")
        acc_world = acc.copy()
    
    return acc_world

In [5]:
def calculate_angular_velocity_from_quat(rot_data, time_delta=1/200): # Assuming 200Hz sampling rate
    if isinstance(rot_data, pd.DataFrame):
        quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
    else:
        quat_values = rot_data

    num_samples = quat_values.shape[0]
    angular_vel = np.zeros((num_samples, 3))

    for i in range(num_samples - 1):
        q_t = quat_values[i]
        q_t_plus_dt = quat_values[i+1]

        if np.all(np.isnan(q_t)) or np.all(np.isclose(q_t, 0)) or \
           np.all(np.isnan(q_t_plus_dt)) or np.all(np.isclose(q_t_plus_dt, 0)):
            continue

        try:
            rot_t = R.from_quat(q_t)
            rot_t_plus_dt = R.from_quat(q_t_plus_dt)

            # Calculate the relative rotation
            delta_rot = rot_t.inv() * rot_t_plus_dt
            
            # Convert delta rotation to angular velocity vector
            # The rotation vector (Euler axis * angle) scaled by 1/dt
            # is a good approximation for small delta_rot
            angular_vel[i, :] = delta_rot.as_rotvec() / time_delta
        except ValueError:
            # If quaternion is invalid, angular velocity remains zero
            pass
            
    return angular_vel

def calculate_angular_distance(rot_data):
    if isinstance(rot_data, pd.DataFrame):
        quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
    else:
        quat_values = rot_data

    num_samples = quat_values.shape[0]
    angular_dist = np.zeros(num_samples)

    for i in range(num_samples - 1):
        q1 = quat_values[i]
        q2 = quat_values[i+1]

        if np.all(np.isnan(q1)) or np.all(np.isclose(q1, 0)) or \
           np.all(np.isnan(q2)) or np.all(np.isclose(q2, 0)):
            angular_dist[i] = 0 # Или np.nan, в зависимости от желаемого поведения
            continue
        try:
            # Преобразование кватернионов в объекты Rotation
            r1 = R.from_quat(q1)
            r2 = R.from_quat(q2)

            # Вычисление углового расстояния: 2 * arccos(|real(p * q*)|)
            # где p* - сопряженный кватернион q
            # В scipy.spatial.transform.Rotation, r1.inv() * r2 дает относительное вращение.
            # Угол этого относительного вращения - это и есть угловое расстояние.
            relative_rotation = r1.inv() * r2
            
            # Угол rotation vector соответствует угловому расстоянию
            # Норма rotation vector - это угол в радианах
            angle = np.linalg.norm(relative_rotation.as_rotvec())
            angular_dist[i] = angle
        except ValueError:
            angular_dist[i] = 0 # В случае недействительных кватернионов
            pass
            
    return angular_dist

In [6]:
def remove_gravity_from_acc(acc_data, rot_data):

    if isinstance(acc_data, pd.DataFrame):
        acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
    else:
        acc_values = acc_data

    if isinstance(rot_data, pd.DataFrame):
        quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
    else:
        quat_values = rot_data

    num_samples = acc_values.shape[0]
    linear_accel = np.zeros_like(acc_values)
    
    gravity_world = np.array([0, 0, 9.81])

    for i in range(num_samples):
        if np.all(np.isnan(quat_values[i])) or np.all(np.isclose(quat_values[i], 0)):
            linear_accel[i, :] = acc_values[i, :] 
            continue

        try:
            rotation = R.from_quat(quat_values[i])
            gravity_sensor_frame = rotation.apply(gravity_world, inverse=True)
            linear_accel[i, :] = acc_values[i, :] - gravity_sensor_frame
        except ValueError:
             linear_accel[i, :] = acc_values[i, :]
             
    return linear_accel

In [7]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:

    """
    Add world-frame acceleration features to the dataframe.

    This function computes acceleration in world coordinates using quaternion orientation,
    and appends these as new columns to the dataframe.

    Args:
        df (pd.DataFrame): Input dataframe containing sensor data and metadata.

    Returns:
        pd.DataFrame: Dataframe with added world-frame acceleration columns.
    """

    meta_cols = [
        "sequence_id"
    ]
    feature_cols_all = [c for c in df.columns if c not in meta_cols]
    other_cols = [c for c in feature_cols_all if (c.startswith('thm_') or c.startswith('tof_'))]
    
    # Add features https://www.kaggle.com/code/rktqwe/lb-0-77-linear-accel-tf-bilstm-gru-attention
    df['acc_mag'] = np.sqrt(df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2)
    df['acc_mag_jerk'] = df['acc_mag'].diff().fillna(0)

    # Jerk magnitude (rate of change of acceleration)
    df['jerk_x'] = np.gradient(df['acc_x'])
    df['jerk_y'] = np.gradient(df['acc_y']) 
    df['jerk_z'] = np.gradient(df['acc_z'])
    df['jerk_magnitude'] = np.sqrt(df['jerk_x']**2 + df['jerk_y']**2 + df['jerk_z']**2)
      
    # Correlation between axes (rolling correlation)
    cor_list = []
    window = 20
    for _, group in df.groupby('sequence_id'):
        group['acc_xy_corr'] = group['acc_x'].rolling(window).corr(group['acc_y']).fillna(0)
        group['acc_xz_corr'] = group['acc_x'].rolling(window).corr(group['acc_z']).fillna(0)
        group['acc_yz_corr'] = group['acc_y'].rolling(window).corr(group['acc_z']).fillna(0)
        cor_list.append(group[['acc_xy_corr', 'acc_xz_corr', 'acc_yz_corr']].to_numpy())
    cor_list = np.concatenate(cor_list, axis = 0)
    df[['acc_xy_corr', 'acc_xz_corr', 'acc_yz_corr']] = cor_list

    df['rot_angle'] = 2 * np.arccos(df['rot_w'].clip(-1, 1))
    df['rot_angle_vel'] = df['rot_angle'].diff().fillna(0)   

    print("Calculating angular velocity from quaternion derivatives...")
    angular_vel_list = []
    for _, group in df.groupby('sequence_id'):
        rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']]
        angular_vel_group = calculate_angular_velocity_from_quat(rot_data_group)
        angular_vel_list.append(angular_vel_group)
    angular_vel_list = np.concatenate(angular_vel_list, axis = 0)
    df[['angular_vel_x', 'angular_vel_y', 'angular_vel_z']] = angular_vel_list
    
    df['angular_vel_magnitude'] = np.sqrt(
        df['angular_vel_x']**2 + df['angular_vel_y']**2 + df['angular_vel_x']**2
    )

    print("Calculating angular distance between successive quaternions...")
    angular_distance_list = []
    for _, group in df.groupby('sequence_id'):
        rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']]
        angular_dist_group = calculate_angular_distance(rot_data_group)
        angular_distance_list.append(angular_dist_group)
    angular_distance_list = np.concatenate(angular_distance_list, axis = 0)
    df['angular_distance'] = angular_distance_list
    
    ############################################

    ACC_COLS2 = ["acc_x2", "acc_y2", "acc_z2"]

    linear_accel_list = []
    for _, group in df.groupby('sequence_id'):
        acc_data_group = group[ACC_COLS]
        rot_data_group = group[ROT_COLS]
        linear_accel_group = remove_gravity_from_acc(acc_data_group, rot_data_group)
        linear_accel_list.append(linear_accel_group)
    linear_accel_list  = np.concatenate(linear_accel_list , axis = 0)
    df[ACC_COLS2] = linear_accel_list

    # Add features https://www.kaggle.com/code/rktqwe/lb-0-77-linear-accel-tf-bilstm-gru-attention
    df['acc_mag2'] = np.sqrt(df['acc_x2']**2 + df['acc_y2']**2 + df['acc_z2']**2)
    df['acc_mag_jerk2'] = df['acc_mag2'].diff().fillna(0)

    # Jerk magnitude (rate of change of acceleration)
    df['jerk_x2'] = np.gradient(df['acc_x2'])
    df['jerk_y2'] = np.gradient(df['acc_y2']) 
    df['jerk_z2'] = np.gradient(df['acc_z2'])
    df['jerk_magnitude2'] = np.sqrt(df['jerk_x2']**2 + df['jerk_y2']**2 + df['jerk_z2']**2)
        
    # Correlation between axes (rolling correlation)
    cor_list = []
    window = 20
    for _, group in df.groupby('sequence_id'):
        group['acc_xy_corr2'] = group['acc_x2'].rolling(window).corr(group['acc_y2']).fillna(0)
        group['acc_xz_corr2'] = group['acc_x2'].rolling(window).corr(group['acc_z2']).fillna(0)
        group['acc_yz_corr2'] = group['acc_y2'].rolling(window).corr(group['acc_z2']).fillna(0)
        cor_list.append(group[['acc_xy_corr2', 'acc_xz_corr2', 'acc_yz_corr2']].to_numpy())
    cor_list = np.concatenate(cor_list, axis = 0)
    df[['acc_xy_corr2', 'acc_xz_corr2', 'acc_yz_corr2']] = cor_list

    df = df[meta_cols 
            + ACC_COLS + ['acc_mag', 'acc_mag_jerk'] 
            + ['jerk_x', 'jerk_y', 'jerk_z', 'jerk_magnitude']
            + ["acc_xy_corr", "acc_xz_corr", "acc_yz_corr"]
            + ROT_COLS + ['rot_angle', 'rot_angle_vel'] 
            + ['angular_vel_x', 'angular_vel_y', 'angular_vel_z'] 
            + ['angular_vel_magnitude', 'angular_distance'] 
            + ACC_COLS2 + ['acc_mag2', 'acc_mag_jerk2'] 
            + ['jerk_x2', 'jerk_y2', 'jerk_z2', 'jerk_magnitude2']
            + ["acc_xy_corr2", "acc_xz_corr2", "acc_yz_corr2"] + other_cols]
    # x = x.ffill().bfill().fillna(0).to_numpy(dtype = dtype)

    df.replace(-np.inf, -1, inplace=True)
    df.replace(np.inf, 1, inplace=True)

    return df


def pad_sequences(
    sequences: list[np.ndarray], 
    maxlen: int, 
    padding: str = "pre", 
    truncating: str = "pre", 
    dtype: str = "float32"
) -> np.ndarray:
    
    """
    Pad sequences to the same length.

    Parameters:
    -----------
    sequences : list of numpy.ndarray
        List of sequences (each sequence is a numpy array) to pad
    maxlen : int
        Maximum length of all sequences. Sequences longer than maxlen will be truncated
    padding : str, optional (default="pre")
        'pre' or 'post', pad either before or after each sequence
    truncating : str, optional (default="pre")
        'pre' or 'post', remove values from sequences larger than maxlen either at the
        beginning or at the end
    dtype : str, optional (default="float32")
        Type of the output array

    Returns:
    --------
    numpy.ndarray
        Padded sequences array of shape (len(sequences), maxlen, ...)
    """
    
    if padding not in ["pre", "post"]: 
        raise NotImplementedError("Invalid padding")
    if truncating not in ["pre", "post"]: 
        raise NotImplementedError("Invalid truncating")
    
    n_samples = len(sequences)
    if maxlen is None:
        maxlen = max(len(s) for s in sequences)
    
    # Sample shape from first non empty sequence
    # If no non-empty sequences, return array of shape (0, maxlen)
    sample_shape = tuple()
    for s in sequences:
        if len(s) > 0:
            sample_shape = np.asarray(s).shape[1:]
            break
    
    x = np.zeros((n_samples, maxlen) + sample_shape, dtype = dtype)
    
    for idx, s in enumerate(sequences):
        if len(s) == 0:
            continue
        
        if truncating == "pre":
            s = s[-maxlen:] 
        elif truncating == "post":
            s[:maxlen]

        trunc = s.shape[0]
        
        if padding == "pre":
            x[idx, -trunc:] = s
        elif padding == "post":
            x[idx, :trunc] = s
            
    return x

def to_categorical(y, num_classes = None):
    y = np.array(y, dtype = "int")
    if not num_classes:
        num_classes = np.max(y) + 1
    return np.eye(num_classes)[y]

In [8]:
def mirror_quaternion(quat):
    """
    Mirror a single quaternion through the YZ plane.

    Args:
        quat (array-like of shape (4,)): [w, x, y, z]

    Returns:
        mirrored (np.ndarray of shape (4,)): mirrored quaternion [w, x, y, z]
    """
    P = np.diag([-1, 1, 1])  # reflection through YZ
    rot = R.from_quat([quat[1], quat[2], quat[3], quat[0]])  # SciPy uses [x, y, z, w]
    R_mat = rot.as_matrix()
    R_flipped = P @ R_mat @ P
    flipped = R.from_matrix(R_flipped).as_quat()
    return np.array([flipped[3], flipped[0], flipped[1], flipped[2]])  # back to [w, x, y, z]


def mirror_data(data):
    """
    Mirror left-handed samples to match right-handed frame.

    Args:
        data (np.ndarray of shape (N, 7)): sensor data
    
    Prints:
        A new array with mirrored left-handed samples.

    """
    mirrored = data.copy()
    
    for i in range(data.shape[0]):
        acc = mirrored[i, :3]
        mirrored[i, :3] = np.array([-acc[0], acc[1], acc[2]])

        quat = mirrored[i, 3:]  # [w, x, y, z]
        mirrored[i, 3:] = mirror_quaternion(quat)

    return mirrored

In [9]:
def preprocess_left_handed_pd(df: pd.DataFrame, demo: pd.DataFrame) -> pd.DataFrame:
    """Note this may sort your dataframe by 'row_id', 
       Also removes nans from rotation, and probably converts to unit"""
    def handle_missing(rot_data: np.ndarray) -> np.ndarray:
        rot_cleaned = rot_data.copy()
        
        for i in range(len(rot_data)):
            row = rot_data[i]
            missing_count = np.isnan(row).sum()
            
            if missing_count == 0:
                # No missing values, normalize to unit quaternion
                norm = np.linalg.norm(row)
                if norm > 1e-8:
                    rot_cleaned[i] = row / norm
                else:
                    rot_cleaned[i] = [1.0, 0.0, 0.0, 0.0]  # Identity quaternion
                    
            elif missing_count == 1:
                # One missing value, reconstruct using unit quaternion constraint
                # |w|² + |x|² + |y|² + |z|² = 1
                missing_idx = np.where(np.isnan(row))[0][0]
                valid_values = row[~np.isnan(row)]
                
                sum_squares = np.sum(valid_values**2)
                if sum_squares <= 1.0:
                    missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                    # Choose sign for continuity with previous quaternion
                    if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                        if rot_cleaned[i-1, missing_idx] < 0:
                            missing_value = -missing_value
                    rot_cleaned[i, missing_idx] = missing_value
                    rot_cleaned[i, ~np.isnan(row)] = valid_values
                else:
                    rot_cleaned[i] = [1.0, 0.0, 0.0, 0.0]
            else:
                # More than one missing value, use identity quaternion
                rot_cleaned[i] = [1.0, 0.0, 0.0, 0.0]
        
        return rot_cleaned

    import polars as pl
    pl_df = pl.DataFrame(df)
    pl_demo = pl.DataFrame(demo)
    r_subjs = pl_demo.filter(pl.col("handedness") == 1)["subject"].to_list()
    l_subjs = pl_demo.filter(pl.col("handedness") == 0)["subject"].to_list()
    
    r_tr = pl_df.filter(pl.col("subject").is_in(r_subjs))
    l_tr = pl_df.filter(pl.col("subject").is_in(l_subjs))

    if l_tr.shape[0] == 0:
        return r_tr.to_pandas()

    rot_cleaned = handle_missing(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
    rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
    
    norms = np.linalg.norm(rot_scipy, axis=1)
    if np.any(norms < 1e-8):
        # Replace problematic quaternions with identity
        mask = norms < 1e-8
        rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        print("yes")
    
    r = R.from_quat(rot_scipy)
    tmp = r.as_euler("xyz")
    tmp[:,1] = - tmp[:,1]
    tmp[:,2] = - tmp[:,2] # maybe who knows but you could give it a try 
    r = R.from_euler("xyz", tmp)
    tmp = r.as_quat()
    l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
    l_tr = l_tr.with_columns(-pl.col("acc_x"))
    
    tmp = l_tr[["thm_3", "thm_5"]]
    tmp.columns = ["thm_5", "thm_3"]
    l_tr = l_tr.with_columns(tmp)
    
    swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
    swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
    
    swap_1_2_4 = list()
    for i in range(0,64,8):
        ll = list()
        for (k,l) in swap_1_2_4_base:
            ll.append([k+i, l+i])
        swap_1_2_4 += ll
    
    swap_3_5 = list()
    for i in range(8):
        ll = list()
        for (k,l) in swap_3_5_base:
            ll.append([k+i, l+i])
        swap_3_5 += ll
    
    l_df = l_tr
    
    for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
        l_tr = l_tr.with_columns(l_df[k].alias(l))
    
    for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
        l_tr = l_tr.with_columns(l_df[l].alias(k))
    
    l_df = l_tr
    
    for i in [1,2,4]:
        for (k, l) in swap_1_2_4:
            l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
    
    for i in [3,5]:
        for (k, l) in swap_3_5:
            l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))

    if r_tr.shape[0] == 0:
        return l_tr.to_pandas()
    else:
        pl_df = pl.concat([r_tr, l_tr])
        pl_df = pl_df.sort(by="row_id")

# 3. All features models

## 3.1. All features - LSTM

In [10]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class ResidualSEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, drop=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.se = SEBlock(out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(drop)
        self.pool = nn.MaxPool1d(2)
        
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, padding='same', bias = False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        
        out += identity
        out = self.relu(out)
        out = self.pool(out)
        out = self.dropout(out)
        return out

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Tanh()
        )
        
    def forward(self, x):
        weights = self.attention(x).squeeze(-1)
        weights = F.softmax(weights, dim=1).unsqueeze(-1)
        context = torch.sum(x * weights, dim=1)
        return context
        
class CrossAttention(nn.Module):
    """
    Cross-attention module for multi-branch feature fusion.
    Allows each branch to attend to features from other branches.
    """
    def __init__(self, feature_dim, num_heads=8, dropout=0.1):
        super(CrossAttention, self).__init__()
        self.feature_dim = feature_dim
        self.num_heads = num_heads
        self.head_dim = feature_dim // num_heads
        
        assert feature_dim % num_heads == 0, "feature_dim must be divisible by num_heads"
        
        # Linear projections for queries, keys, and values for each branch
        self.q_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.k_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.v_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        
        # Output projection
        self.out_linear = nn.Linear(feature_dim, feature_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Layer normalization for residual connection
        self.layer_norm = nn.LayerNorm(feature_dim)
        
    def forward(self, query_branch, key_value_branches):
        """
        Args:
            query_branch: (B, T, C) - the branch that will be updated
            key_value_branches: List of (B, T, C) tensors - branches to attend to
        Returns:
            Updated query_branch with cross-attention: (B, T, C)
        """
        B, T, C = query_branch.shape
        
        # Create queries from the query branch
        Q = self.q_linear(query_branch)  # (B, T, C)
        
        # Concatenate all key-value branches for attention
        all_kv = torch.stack(key_value_branches, dim=1)  # (B, num_branches, T, C)
        num_branches = all_kv.shape[1]
        
        # Reshape for multi-head attention
        all_kv = all_kv.reshape(B, num_branches * T, C)  # (B, num_branches*T, C)
        
        K = self.k_linear(all_kv)  # (B, num_branches*T, C)
        V = self.v_linear(all_kv)  # (B, num_branches*T, C)
        
        # Reshape for multi-head attention
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, T, head_dim)
        K = K.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        V = V.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, num_heads, T, num_branches*T)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention to values
        attn_output = torch.matmul(attn_weights, V)  # (B, num_heads, T, head_dim)
        
        # Concatenate heads and put through final linear layer
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        output = self.out_linear(attn_output)
        
        # Residual connection and layer normalization
        output = self.layer_norm(query_branch + output)
        
        return output

class IMUCrossAttentionFusion(nn.Module):
    """
    Cross-attention fusion module for three IMU branches.
    Each branch attends to the other two branches.
    """
    def __init__(self, feature_dim=256, num_heads=8, dropout=0.1):
        super(IMUCrossAttentionFusion, self).__init__()
        
        # Cross-attention modules for each branch
        self.cross_attn1 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn2 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn3 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn4 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn5 = CrossAttention(feature_dim, num_heads, dropout)
        
    def forward(self, imu1, imu2, imu3, thm, tof):
        """
        Args:
            imu1, imu2, imu3, thm, tof: (B, T, C) tensors from each branch
        Returns:
            Tuple of enhanced features: (enhanced_imu1, enhanced_imu2, enhanced_imu3, enhanced_thm, enhanced_tof)
        """
        # Each branch attends to the other two branches
        enhanced_imu1 = self.cross_attn1(imu1, [imu2, imu3, thm, tof])
        enhanced_imu2 = self.cross_attn2(imu2, [imu1, imu3, thm, tof])
        enhanced_imu3 = self.cross_attn3(imu3, [imu1, imu2, thm, tof])
        enhanced_thm = self.cross_attn4(thm, [imu1, imu2, imu3, tof])
        enhanced_tof = self.cross_attn5(tof, [imu1, imu2, imu3, thm])
        
        return enhanced_imu1, enhanced_imu2, enhanced_imu3, enhanced_thm, enhanced_tof


class ImuModelLSTM(nn.Module):
    def __init__(self, imu_dim):
        super().__init__()
        self.imu_dim = imu_dim
        
        # IMU branchs
        self.imu_branch1 = nn.Sequential(
            ResidualSEBlock(12, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        self.imu_branch2 = nn.Sequential(
            ResidualSEBlock(11, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        self.imu_branch3 = nn.Sequential(
            ResidualSEBlock(12, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        

    def forward(self, x):
        imu = x[:, :, :self.imu_dim]

        imu1 = self.imu_branch1(imu[:, :, :12].transpose(1, 2)).transpose(1, 2)

        imu2 = self.imu_branch2(imu[:, :, 12:23].transpose(1, 2)).transpose(1, 2)

        imu3 = self.imu_branch3(imu[:, :, 23:].transpose(1, 2)).transpose(1, 2)
 
        return imu1, imu2, imu3

class ThreeBranchModelLSTM(nn.Module):
    def __init__(self, imu_dim, thm_dim, tof_dim, n_classes):
        super().__init__()
        self.imu_dim = imu_dim
        self.thm_dim = thm_dim
        self.tof_dim = tof_dim
        
        # IMU branch
        self.imu_branch = ImuModelLSTM(self.imu_dim)

        # THM branch
        self.thm_branch = nn.Sequential(
            ResidualSEBlock(thm_dim, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        
        # TOF branch
        self.tof_branch = nn.Sequential(
            ResidualSEBlock(tof_dim, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        
        self.cross_attention_fusion = IMUCrossAttentionFusion(
            feature_dim=256, num_heads=8, dropout=0.1
        )
        
        # BiLSTM
        self.bilstm = nn.LSTM(256*5, 512, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.4)
  
        # Attention
        self.attention = AttentionLayer(1024)
        
        # Dense layers
        self.fc_layers = nn.Sequential(
            nn.Linear(256*4, 512, bias = False),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256, bias = False),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        # Split input
        imu = x[:, :, :self.imu_dim]
        thm = x[:, :, self.imu_dim:(self.imu_dim+self.thm_dim)]
        tof = x[:, :, (self.imu_dim+self.thm_dim):]
        # Handle missing values (-1) by replacing with 500
        tof = torch.where(tof == -1, torch.full_like(tof, 500), tof)
        # Normalize to [0, 1] range
        tof = tof / 500.0
        
        # Process branches
        imu1, imu2, imu3 = self.imu_branch(imu)
        thm = self.thm_branch(thm.transpose(1, 2)).transpose(1, 2)
        tof = self.tof_branch(tof.transpose(1, 2)).transpose(1, 2)

        imu1, imu2, imu3, thm, tof = self.cross_attention_fusion(imu1, imu2, imu3, thm, tof)

        merged = torch.cat((imu1, imu2, imu3, thm, tof), dim=2)
        
        # BiLSTM
        lstm_out, _ = self.bilstm(merged)
        lstm_out = self.dropout(lstm_out)
        
        # Attention
        attended = self.attention(lstm_out)
        
        # Dense layers
        out = self.fc_layers(attended)
        return out 

In [11]:
feature_cols = np.load(all_features_lstm_dir / "feature_cols.npy", allow_pickle=True).tolist()
gesture_classes = np.load(all_features_lstm_dir / "gesture_classes.npy", allow_pickle=True)

imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
thm_cols = [c for c in feature_cols if c.startswith('thm_')]
tof_cols = [c for c in feature_cols if c.startswith('tof_')]

models_all_features_lstm = []
for fold in range(5):
    model = ThreeBranchModelLSTM(len(imu_cols), len(thm_cols), 
                                 len(tof_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(all_features_lstm_dir / f"{model_all_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_all_features_lstm.append(model)

In [12]:
feature_cols = np.load(all_features_lstm_dir_old / "feature_cols.npy", allow_pickle=True).tolist()
gesture_classes = np.load(all_features_lstm_dir_old / "gesture_classes.npy", allow_pickle=True)

imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
thm_cols = [c for c in feature_cols if c.startswith('thm_')]
tof_cols = [c for c in feature_cols if c.startswith('tof_')]

models_all_features_lstm_old = []
for fold in range(5):
    model = ThreeBranchModelLSTM(len(imu_cols), len(thm_cols), 
                                 len(tof_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(all_features_lstm_dir_old / f"{model_all_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_all_features_lstm_old.append(model)

In [13]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class ResidualSEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, drop=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.se = SEBlock(out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(drop)
        self.pool = nn.MaxPool1d(2)
        
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, padding='same', bias = False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        
        out += identity
        out = self.relu(out)
        out = self.pool(out)
        out = self.dropout(out)
        return out

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Tanh()
        )
        
    def forward(self, x):
        weights = self.attention(x).squeeze(-1)
        weights = F.softmax(weights, dim=1).unsqueeze(-1)
        context = torch.sum(x * weights, dim=1)
        return context
        
class CrossAttention(nn.Module):
    """
    Cross-attention module for multi-branch feature fusion.
    Allows each branch to attend to features from other branches.
    """
    def __init__(self, feature_dim, num_heads=8, dropout=0.1):
        super(CrossAttention, self).__init__()
        self.feature_dim = feature_dim
        self.num_heads = num_heads
        self.head_dim = feature_dim // num_heads
        
        assert feature_dim % num_heads == 0, "feature_dim must be divisible by num_heads"
        
        # Linear projections for queries, keys, and values for each branch
        self.q_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.k_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.v_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        
        # Output projection
        self.out_linear = nn.Linear(feature_dim, feature_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Layer normalization for residual connection
        self.layer_norm = nn.LayerNorm(feature_dim)
        
    def forward(self, query_branch, key_value_branches):
        """
        Args:
            query_branch: (B, T, C) - the branch that will be updated
            key_value_branches: List of (B, T, C) tensors - branches to attend to
        Returns:
            Updated query_branch with cross-attention: (B, T, C)
        """
        B, T, C = query_branch.shape
        
        # Create queries from the query branch
        Q = self.q_linear(query_branch)  # (B, T, C)
        
        # Concatenate all key-value branches for attention
        all_kv = torch.stack(key_value_branches, dim=1)  # (B, num_branches, T, C)
        num_branches = all_kv.shape[1]
        
        # Reshape for multi-head attention
        all_kv = all_kv.reshape(B, num_branches * T, C)  # (B, num_branches*T, C)
        
        K = self.k_linear(all_kv)  # (B, num_branches*T, C)
        V = self.v_linear(all_kv)  # (B, num_branches*T, C)
        
        # Reshape for multi-head attention
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, T, head_dim)
        K = K.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        V = V.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, num_heads, T, num_branches*T)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention to values
        attn_output = torch.matmul(attn_weights, V)  # (B, num_heads, T, head_dim)
        
        # Concatenate heads and put through final linear layer
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        output = self.out_linear(attn_output)
        
        # Residual connection and layer normalization
        output = self.layer_norm(query_branch + output)
        
        return output

class IMUCrossAttentionFusion(nn.Module):
    """
    Cross-attention fusion module for three IMU branches.
    Each branch attends to the other two branches.
    """
    def __init__(self, feature_dim=256, num_heads=8, dropout=0.1):
        super(IMUCrossAttentionFusion, self).__init__()
        
        # Cross-attention modules for each branch
        self.cross_attn1 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn2 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn3 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn4 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn5 = CrossAttention(feature_dim, num_heads, dropout)
        
    def forward(self, imu1, imu2, imu3, thm, tof):
        """
        Args:
            imu1, imu2, imu3, thm, tof: (B, T, C) tensors from each branch
        Returns:
            Tuple of enhanced features: (enhanced_imu1, enhanced_imu2, enhanced_imu3, enhanced_thm, enhanced_tof)
        """
        # Each branch attends to the other two branches
        enhanced_imu1 = self.cross_attn1(imu1, [imu2, imu3, thm, tof])
        enhanced_imu2 = self.cross_attn2(imu2, [imu1, imu3, thm, tof])
        enhanced_imu3 = self.cross_attn3(imu3, [imu1, imu2, thm, tof])
        enhanced_thm = self.cross_attn3(thm, [imu1, imu2, imu3, tof])
        enhanced_tof = self.cross_attn3(tof, [imu1, imu2, imu3, thm])
        
        return enhanced_imu1, enhanced_imu2, enhanced_imu3, enhanced_thm, enhanced_tof


class ImuModelLSTM2(nn.Module):
    def __init__(self, imu_dim):
        super().__init__()
        self.imu_dim = imu_dim
        
        # IMU branchs
        self.imu_branch1 = nn.Sequential(
            ResidualSEBlock(12, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        self.imu_branch2 = nn.Sequential(
            ResidualSEBlock(11, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        self.imu_branch3 = nn.Sequential(
            ResidualSEBlock(12, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        

    def forward(self, x):
        imu = x[:, :, :self.imu_dim]

        imu1 = self.imu_branch1(imu[:, :, :12].transpose(1, 2)).transpose(1, 2)

        imu2 = self.imu_branch2(imu[:, :, 12:23].transpose(1, 2)).transpose(1, 2)

        imu3 = self.imu_branch3(imu[:, :, 23:].transpose(1, 2)).transpose(1, 2)
 
        return imu1, imu2, imu3

class ThreeBranchModelLSTM2(nn.Module):
    def __init__(self, imu_dim, thm_dim, tof_dim, n_classes):
        super().__init__()
        self.imu_dim = imu_dim
        self.thm_dim = thm_dim
        self.tof_dim = tof_dim
        
        # IMU branch
        self.imu_branch = ImuModelLSTM2(self.imu_dim)

        # THM branch
        self.thm_branch = nn.Sequential(
            ResidualSEBlock(thm_dim, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        
        # TOF branch
        self.tof_branch = nn.Sequential(
            ResidualSEBlock(tof_dim, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        
        self.cross_attention_fusion = IMUCrossAttentionFusion(
            feature_dim=256, num_heads=8, dropout=0.1
        )
        
        # BiLSTM
        self.bilstm = nn.LSTM(256*5, 512, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.4)
  
        # Attention
        self.attention = AttentionLayer(1024)
        
        # Dense layers
        self.fc_layers = nn.Sequential(
            nn.Linear(256*4, 512, bias = False),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256, bias = False),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        # Split input
        imu = x[:, :, :self.imu_dim]
        thm = x[:, :, self.imu_dim:(self.imu_dim+self.thm_dim)]
        tof = x[:, :, (self.imu_dim+self.thm_dim):]
        # Handle missing values (-1) by replacing with 500
        tof = torch.where(tof == -1, torch.full_like(tof, 500), tof)
        # Normalize to [0, 1] range
        tof = tof / 500.0
        
        # Process branches
        imu1, imu2, imu3 = self.imu_branch(imu)
        thm = self.thm_branch(thm.transpose(1, 2)).transpose(1, 2)
        tof = self.tof_branch(tof.transpose(1, 2)).transpose(1, 2)

        imu1, imu2, imu3, thm, tof = self.cross_attention_fusion(imu1, imu2, imu3, thm, tof)

        merged = torch.cat((imu1, imu2, imu3, thm, tof), dim=2)
        
        # BiLSTM
        lstm_out, _ = self.bilstm(merged)
        lstm_out = self.dropout(lstm_out)
        
        # Attention
        attended = self.attention(lstm_out)
        
        # Dense layers
        out = self.fc_layers(attended)
        return out 


In [14]:
# Use ThreeBranchModelLSTM for model with bug only
feature_cols = np.load(all_features_lstm_dir2 / "feature_cols.npy", allow_pickle=True).tolist()
gesture_classes = np.load(all_features_lstm_dir2 / "gesture_classes.npy", allow_pickle=True)

imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
thm_cols = [c for c in feature_cols if c.startswith('thm_')]
tof_cols = [c for c in feature_cols if c.startswith('tof_')]

models_all_features_lstm2 = []
for fold in range(5):
    model = ThreeBranchModelLSTM(len(imu_cols), len(thm_cols), 
                                 len(tof_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(all_features_lstm_dir2 / f"{model_all_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_all_features_lstm2.append(model)

In [15]:
feature_cols = np.load(all_features_lstm_dir2_old / "feature_cols.npy", allow_pickle=True).tolist()
gesture_classes = np.load(all_features_lstm_dir2_old / "gesture_classes.npy", allow_pickle=True)

imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
thm_cols = [c for c in feature_cols if c.startswith('thm_')]
tof_cols = [c for c in feature_cols if c.startswith('tof_')]

models_all_features_lstm2_old = []
for fold in range(5):
    model = ThreeBranchModelLSTM2(len(imu_cols), len(thm_cols), 
                                 len(tof_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(all_features_lstm_dir2_old / f"{model_all_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_all_features_lstm2_old.append(model)

## 3.2. All features - attention

In [16]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class ResidualSEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, drop=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.se = SEBlock(out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(drop)
        self.pool = nn.MaxPool1d(2)
        
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, padding='same', bias = False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        
        out += identity
        out = self.relu(out)
        out = self.pool(out)
        out = self.dropout(out)
        return out

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, V)
        return output, attention_weights
    
    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()
        
        # Residual connection
        residual = x
        
        # Linear transformations and reshape
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply attention
        attn_output, _ = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, d_model)
        
        # Final linear layer
        output = self.W_o(attn_output)
        
        # Add residual connection and layer norm
        output = self.layer_norm(output + residual)
        
        return output

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length=5000):
        super().__init__()
        
        pe = torch.zeros(max_length, d_model)
        position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
        
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Tanh()
        )
        
    def forward(self, x):
        weights = self.attention(x).squeeze(-1)
        weights = F.softmax(weights, dim=1).unsqueeze(-1)
        context = torch.sum(x * weights, dim=1)
        return context

class ImuModelAtt(nn.Module):
    def __init__(self, imu_dim):
        super().__init__()
        self.imu_dim = imu_dim
        
        # IMU branches
        self.imu_branch1 = nn.Sequential(
            ResidualSEBlock(12, 64, 3, drop=0.3),
            ResidualSEBlock(64, 128, 5, drop=0.3)
        )
        self.imu_branch2 = nn.Sequential(
            ResidualSEBlock(11, 64, 3, drop=0.3),
            ResidualSEBlock(64, 128, 5, drop=0.3)
        )
        self.imu_branch3 = nn.Sequential(
            ResidualSEBlock(12, 64, 3, drop=0.3),
            ResidualSEBlock(64, 128, 5, drop=0.3)
        )
        
        # Positional encoding for attention layers
        self.pos_encoding1 = PositionalEncoding(128)
        self.pos_encoding2 = PositionalEncoding(128)
        self.pos_encoding3 = PositionalEncoding(128)
        
        # Multi-head self-attention layers (replacing BiLSTM)
        self.self_attention1 = MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        self.self_attention2 = MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        self.self_attention3 = MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        
        # Additional attention layers for better feature learning
        self.attention_block1 = nn.Sequential(
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        )
        self.attention_block2 = nn.Sequential(
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        )
        self.attention_block3 = nn.Sequential(
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        )
        
        # Final attention for aggregation
        self.attention1 = AttentionLayer(128)
        self.attention2 = AttentionLayer(128)
        self.attention3 = AttentionLayer(128)

    def forward(self, x):
        imu = x[:, :, :self.imu_dim]

        # Branch 1: Process first 3 IMU dimensions
        imu1 = self.imu_branch1(imu[:, :, :12].transpose(1, 2)).transpose(1, 2)
        imu1 = self.pos_encoding1(imu1)
        attn_out1 = self.self_attention1(imu1)
        attn_out1 = self.attention_block1[0](attn_out1)
        attn_out1 = self.attention_block1[1](attn_out1)
        attended1 = self.attention1(attn_out1)

        # Branch 2: Process dimensions 3-15
        imu2 = self.imu_branch2(imu[:, :, 12:23].transpose(1, 2)).transpose(1, 2)
        imu2 = self.pos_encoding2(imu2)
        attn_out2 = self.self_attention2(imu2)
        attn_out2 = self.attention_block2[0](attn_out2)
        attn_out2 = self.attention_block2[1](attn_out2)
        attended2 = self.attention2(attn_out2)

        # Branch 3: Process remaining dimensions
        imu3 = self.imu_branch3(imu[:, :, 23:].transpose(1, 2)).transpose(1, 2)
        imu3 = self.pos_encoding3(imu3)
        attn_out3 = self.self_attention3(imu3)
        attn_out3 = self.attention_block3[0](attn_out3)
        attn_out3 = self.attention_block3[1](attn_out3)
        attended3 = self.attention3(attn_out3)

        # Merge all branches
        merged = torch.cat((attended1, attended2, attended3), dim=1)

        return merged
    
class ThreeBranchModelAtt(nn.Module):
    def __init__(self, imu_dim, thm_dim, tof_dim, n_classes):
        super().__init__()
        self.imu_dim = imu_dim
        self.thm_dim = thm_dim
        self.tof_dim = tof_dim
        
        # IMU branch
        self.imu_branch = ImuModelAtt(self.imu_dim)

        self.thm_branch = nn.Sequential(
            ResidualSEBlock(self.thm_dim, 64, 3, drop=0.3),
            ResidualSEBlock(64, 128, 5, drop=0.3)
        )
        self.tof_branch = nn.Sequential(
            ResidualSEBlock(self.tof_dim, 64, 3, drop=0.3),
            ResidualSEBlock(64, 128, 5, drop=0.3)
        )
        
        # Positional encoding for attention layers
        self.pos_encoding1 = PositionalEncoding(128)
        self.pos_encoding2 = PositionalEncoding(128)
        
        # Multi-head self-attention layers (replacing BiLSTM)
        self.self_attention1 = MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        self.self_attention2 = MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        
        # Additional attention layers for better feature learning
        self.attention_block1 = nn.Sequential(
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        )
        self.attention_block2 = nn.Sequential(
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(128, num_heads=8, dropout=0.4)
        )
        
        # Final attention for aggregation
        self.attention1 = AttentionLayer(128)
        self.attention2 = AttentionLayer(128)
        
        # Dense layers
        self.fc_layers = nn.Sequential(
            nn.Linear(128*5, 512, bias = False),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256, bias = False),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        # Split input
        imu = x[:, :, :self.imu_dim]
        thm = x[:, :, self.imu_dim:(self.imu_dim+self.thm_dim)]
        tof = x[:, :, (self.imu_dim+self.thm_dim):]
        # Handle missing values (-1) by replacing with 500
        tof = torch.where(tof == -1, torch.full_like(tof, 500), tof)
        # Normalize to [0, 1] range
        tof = tof / 500.0
        
        # Process branches
        imu = self.imu_branch(imu)

        # Branch THM
        thm = self.thm_branch(thm.transpose(1, 2)).transpose(1, 2)
        thm = self.pos_encoding1(thm)
        attn_out1 = self.self_attention1(thm)
        attn_out1 = self.attention_block1[0](attn_out1)
        attn_out1 = self.attention_block1[1](attn_out1)
        attended1 = self.attention1(attn_out1)

        # Branch TOF
        tof = self.tof_branch(tof.transpose(1, 2)).transpose(1, 2)
        tof = self.pos_encoding2(tof)
        attn_out2 = self.self_attention2(tof)
        attn_out2 = self.attention_block2[0](attn_out2)
        attn_out2 = self.attention_block2[1](attn_out2)
        attended2 = self.attention2(attn_out2)

        # Merge branches
        merged = torch.cat((imu, attended1, attended2), dim=1)
        
        # Dense layers
        out = self.fc_layers(merged)
        return out # F.softmax(out, dim=1)
    

In [17]:
feature_cols = np.load(all_features_attention_dir / "feature_cols.npy", allow_pickle=True).tolist()
gesture_classes = np.load(all_features_attention_dir / "gesture_classes.npy", allow_pickle=True)

imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
thm_cols = [c for c in feature_cols if c.startswith('thm_')]
tof_cols = [c for c in feature_cols if c.startswith('tof_')]

models_all_features_attention = []
for fold in range(5):
    model = ThreeBranchModelAtt(len(imu_cols), len(thm_cols), 
                                len(tof_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(all_features_attention_dir / f"{model_all_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_all_features_attention.append(model)

In [18]:
feature_cols = np.load(all_features_attention_dir_old / "feature_cols.npy", allow_pickle=True).tolist()
gesture_classes = np.load(all_features_attention_dir_old / "gesture_classes.npy", allow_pickle=True)

imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
thm_cols = [c for c in feature_cols if c.startswith('thm_')]
tof_cols = [c for c in feature_cols if c.startswith('tof_')]

models_all_features_attention_old = []
for fold in range(5):
    model = ThreeBranchModelAtt(len(imu_cols), len(thm_cols), 
                                len(tof_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(all_features_attention_dir_old / f"{model_all_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_all_features_attention_old.append(model)

## 3.3. All features - CNN

In [19]:
class CoordAttention(nn.Module):
    """
    Coordinate Attention for Sequences.
    Input Dimension: (B, T, C)
    Output Dimension: (B, T, C)
    """
    def __init__(self, channels, reduction=4):
        super(CoordAttention, self).__init__()
        self.mid_channels = max(8, channels // reduction)

        self.compression = nn.Sequential(
            nn.Conv1d(channels, self.mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(self.mid_channels),
            nn.SiLU(inplace=True)
        )

        # Attention branches
        self.time_conv = nn.Conv1d(1, 1, kernel_size=5, padding=2, bias=False)  
        self.channel_conv = nn.Conv1d(self.mid_channels, channels, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, T, C)
        x_p = x.permute(0, 2, 1)  # (B, C, T)
        f   = self.compression(x_p)  # (B, rC, T)

        ## Time Attention (B, 1, T)
        f_t = f.mean(dim=1, keepdim=True)      
        time_attn = self.sigmoid(self.time_conv(f_t))  

        ## Channel Attention (B, C, 1)
        f_c = f.mean(dim=2, keepdim=True)      
        channel_attn = self.sigmoid(self.channel_conv(f_c)) 

        ## (B, T, C)
        out = (x_p * time_attn * channel_attn).permute(0,2,1)
        return out
    
class ResCNNBlock(nn.Module):
    """
    Residual CNN Block with Coordinate Attention. 
    Input Dimension: (B, T_in, C_in)
    Output Dimension: (B, T_out, C_out)
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, pool_size=2, drop=0.2):
        super(ResCNNBlock, self).__init__()

        ## CNN model
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding='same', bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size, padding='same', bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )
        ## Attention module
        self.attention = CoordAttention(out_channels)
        ## Projection path for skip connection
        self.shortcut_proj = None
        if in_channels != out_channels:
            self.shortcut_proj = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, padding='same', bias=False),
                nn.BatchNorm1d(out_channels)
            )
        ## Final processing components
        self.relu_final = nn.ReLU(inplace=True)
        if pool_size is not None: 
            self.max_pool = nn.MaxPool1d(pool_size)
        else: 
            self.max_pool = None
        self.dropout = nn.Dropout(drop)

    def forward(self, x):
        # (B, C_in, T)
        shortcut = x.permute(0,2,1)
        x_p = x.permute(0,2,1)
        
        out = self.cnn(x_p)                         # (B, C_out, T)
        out = self.attention(out.permute(0, 2, 1))  # (B, T, C_out)
        out = out.permute(0, 2, 1)                  # (B, C_out, T)
        
        ## Handle shortcut connection -> (B, C_out, T)
        if self.shortcut_proj:
            shortcut = self.shortcut_proj(shortcut)

        ## Residual connection
        out += shortcut
        out = self.relu_final(out)
        if self.max_pool is not None:
            ## (B, C_out, T) -> (B, C_out, T//pool_size)
            out = self.max_pool(out)
        out = self.dropout(out)
        ## (B, T_out, C_out)
        out = out.permute(0, 2, 1)
        return out
    
class TempCNNBlock(nn.Module):
    """
    Temporal CNN Block with Dilated Convolution, Attention, and Skip Connection.
    Input Dimension: (B, T, C_in)
    Output Dimension: (B, T, C_out)
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, drop=0.25):
        super(TempCNNBlock, self).__init__()
        ## Padding computation
        padding = (kernel_size - 1) * dilation // 2
        ## Temporal Convolution block
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size,
                      dilation=dilation, padding=padding, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )
        ## Attention module
        self.attention = CoordAttention(out_channels)
        ## Dropout
        self.dropout = nn.Dropout(drop)
        
        ## Shortcut for skip connection
        self.shortcut_proj = None
        if in_channels != out_channels:
            self.shortcut_proj = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, padding='same', bias=False),
                nn.BatchNorm1d(out_channels)
            )
        self.relu_final = nn.ReLU()
        
    def forward(self, x):
        ## Processes skip connection
        # (B, C_in, T)
        x_p      = x.permute(0,2,1)
        shortcut = x.permute(0,2,1)
        # (B, C_out, T)
        if self.shortcut_proj:
            shortcut = self.shortcut_proj(shortcut)
        ## Processes output
        out = self.conv(x_p)                        # (B, C_out, T)
        out = self.attention(out.permute(0,2,1))    # (B, T, C_out)
        out = out.permute(0,2,1)                    # (B, C_out, T)
        ## Skip connection
        out = self.relu_final(out + shortcut) # (B, C_out, T)
        out = self.dropout(out)
        return out.permute(0,2,1)  # (B, T, C_out)
    
class MLPAttention(nn.Module):
    def __init__(self, feature_dim):
        super(MLPAttention, self).__init__()
        self.attn = nn.Sequential(
            nn.Linear(feature_dim, feature_dim//8),
            nn.SiLU(inplace=True),
            nn.Linear(feature_dim//8, 1)
        )

    def forward(self, x):
        # inputs shape: (B, T, C)
        weights = self.attn(x)  # (B, T, 1)
        weights = F.softmax(weights, dim=1)  # (B, T, 1)
        context = (x * weights).sum(dim=1)  # (B, C)
        return context


class CrossAttention(nn.Module):
    """
    Cross-attention module for multi-branch feature fusion.
    Allows each branch to attend to features from other branches.
    """
    def __init__(self, feature_dim, num_heads=8, dropout=0.1):
        super(CrossAttention, self).__init__()
        self.feature_dim = feature_dim
        self.num_heads = num_heads
        self.head_dim = feature_dim // num_heads
        
        assert feature_dim % num_heads == 0, "feature_dim must be divisible by num_heads"
        
        # Linear projections for queries, keys, and values for each branch
        self.q_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.k_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.v_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        
        # Output projection
        self.out_linear = nn.Linear(feature_dim, feature_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Layer normalization for residual connection
        self.layer_norm = nn.LayerNorm(feature_dim)
        
    def forward(self, query_branch, key_value_branches):
        """
        Args:
            query_branch: (B, T, C) - the branch that will be updated
            key_value_branches: List of (B, T, C) tensors - branches to attend to
        Returns:
            Updated query_branch with cross-attention: (B, T, C)
        """
        B, T, C = query_branch.shape
        
        # Create queries from the query branch
        Q = self.q_linear(query_branch)  # (B, T, C)
        
        # Concatenate all key-value branches for attention
        all_kv = torch.stack(key_value_branches, dim=1)  # (B, num_branches, T, C)
        num_branches = all_kv.shape[1]
        
        # Reshape for multi-head attention
        all_kv = all_kv.reshape(B, num_branches * T, C)  # (B, num_branches*T, C)
        
        K = self.k_linear(all_kv)  # (B, num_branches*T, C)
        V = self.v_linear(all_kv)  # (B, num_branches*T, C)
        
        # Reshape for multi-head attention
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, T, head_dim)
        K = K.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        V = V.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, num_heads, T, num_branches*T)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention to values
        attn_output = torch.matmul(attn_weights, V)  # (B, num_heads, T, head_dim)
        
        # Concatenate heads and put through final linear layer
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        output = self.out_linear(attn_output)
        
        # Residual connection and layer normalization
        output = self.layer_norm(query_branch + output)
        
        return output

class IMUCrossAttentionFusion(nn.Module):
    """
    Cross-attention fusion module for three IMU branches.
    Each branch attends to the other two branches.
    """
    def __init__(self, feature_dim=256, num_heads=8, dropout=0.1):
        super(IMUCrossAttentionFusion, self).__init__()
        
        # Cross-attention modules for each branch
        self.cross_attn1 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn2 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn3 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn4 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn5 = CrossAttention(feature_dim, num_heads, dropout)
        
    def forward(self, imu1, imu2, imu3, thm, tof):
        """
        Args:
            imu1, imu2, imu3, thm, tof: (B, T, C) tensors from each branch
        Returns:
            Tuple of enhanced features: (enhanced_imu1, enhanced_imu2, enhanced_imu3, enhanced_thm, enhanced_tof)
        """
        # Each branch attends to the other two branches
        enhanced_imu1 = self.cross_attn1(imu1, [imu2, imu3, thm, tof])
        enhanced_imu2 = self.cross_attn2(imu2, [imu1, imu3, thm, tof])
        enhanced_imu3 = self.cross_attn3(imu3, [imu1, imu2, thm, tof])
        enhanced_thm = self.cross_attn4(thm, [imu1, imu2, imu3, tof])
        enhanced_tof = self.cross_attn5(tof, [imu1, imu2, imu3, thm])
        
        return enhanced_imu1, enhanced_imu2, enhanced_imu3, enhanced_thm, enhanced_tof
        
class ImuModelCNN(nn.Module):
    def __init__(self, imu_dim):
        super().__init__()
        self.imu_dim = imu_dim
        
        # IMU branches
        self.imu_branch1 = nn.Sequential(
            ResCNNBlock(self.imu_dim[0], 128, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(128, 256, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(256, 256, kernel_size=5, pool_size=None, drop=0.2),
        )
        self.imu_branch2 = nn.Sequential(
            ResCNNBlock(self.imu_dim[1], 128, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(128, 256, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(256, 256, kernel_size=5, pool_size=None, drop=0.2),
        )
        self.imu_branch3 = nn.Sequential(
            ResCNNBlock(self.imu_dim[2], 128, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(128, 256, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(256, 256, kernel_size=5, pool_size=None, drop=0.2),
        )

    def forward(self, x):
        imu = x[:, :, :sum(self.imu_dim)]

        # Branch 1
        imu1 = self.imu_branch1(imu[:, :, :self.imu_dim[0]])

        # Branch 2
        imu2 = self.imu_branch2(imu[:, :, self.imu_dim[0]:(self.imu_dim[0]+self.imu_dim[1])])

        # Branch 3
        imu3 = self.imu_branch3(imu[:, :, (self.imu_dim[0]+self.imu_dim[1]):])

        return imu1, imu2, imu3
    
class ThreeBranchModelCNN(nn.Module):
    def __init__(self, imu_dim, thm_dim, tof_dim, n_classes):
        super().__init__()
        self.imu_dim = imu_dim
        self.thm_dim = thm_dim
        self.tof_dim = tof_dim
        self.n_classes = n_classes
        
        # IMU branch
        self.imu_branch = ImuModelCNN(self.imu_dim)

        self.thm_branch = nn.Sequential(
            ResCNNBlock(self.thm_dim, 128, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(128, 256, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(256, 256, kernel_size=5, pool_size=None, drop=0.2),
        )

        self.tof_branch = nn.Sequential(
            ResCNNBlock(self.tof_dim, 128, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(128, 256, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(256, 256, kernel_size=5, pool_size=None, drop=0.2),
        )

        #self.cross_attention_fusion = IMUCrossAttentionFusion(
        #    feature_dim=256, num_heads=8, dropout=0.1
        #)
        
        self.merged_attention = CoordAttention(256*5)
        self.merged_dropout   = nn.Dropout(1/3)
        self.temporalCNN      = nn.Sequential(
            TempCNNBlock(256*5, 512, kernel_size=3, dilation=1, drop=0.2),
            TempCNNBlock(512, 512, kernel_size=3, dilation=2, drop=0.2),
            TempCNNBlock(512, 256, kernel_size=3, dilation=4, drop=0.2),
            TempCNNBlock(256, 256, kernel_size=3, dilation=8, drop=0.2),
        )

        self.classifier_attention = MLPAttention(256)
        self.classifier_dropout   = nn.Dropout(0.2)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, self.n_classes)
        )

    def forward(self, x):
        # Split input
        imu = x[:, :, :sum(self.imu_dim)]
        thm = x[:, :, sum(self.imu_dim):(sum(self.imu_dim)+self.thm_dim)]
        tof = x[:, :, (sum(self.imu_dim)+self.thm_dim):]
        
        # Handle missing values (-1) by replacing with 500
        tof = torch.where(tof == -1, torch.full_like(tof, 500), tof)
        # Normalize to [0, 1] range
        tof = tof / 500.0
        
        # Process branches
        imu1, imu2, imu3 = self.imu_branch(imu)

        # Branch THM
        thm = self.thm_branch(thm)

        # Branch TOF
        tof = self.tof_branch(tof)

        #imu1, imu2, imu3, thm, tof = self.cross_attention_fusion(imu1, imu2, imu3, thm, tof)

        # Merge branches
        merged = torch.cat((imu1, imu2, imu3, thm, tof), dim=-1)

        merged = self.merged_attention(merged)
        merged = self.merged_dropout(merged)
        out = self.temporalCNN(merged) 

        ## Provides output to classifier
        out = self.classifier_attention(out) 
        out = self.classifier_dropout(out)
        out = self.classifier(out)          
        return out

In [20]:
feature_cols = np.load(all_features_cnn_dir / "feature_cols.npy", allow_pickle=True).tolist()
gesture_classes = np.load(all_features_cnn_dir / "gesture_classes.npy", allow_pickle=True)

imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
thm_cols = [c for c in feature_cols if c.startswith('thm_')]
tof_cols = [c for c in feature_cols if c.startswith('tof_')]

models_all_features_cnn = []
for fold in range(5):
    model = ThreeBranchModelCNN([12, 11, 12], len(thm_cols), 
                                len(tof_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(all_features_cnn_dir / f"{model_all_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_all_features_cnn.append(model)

In [21]:
feature_cols = np.load(all_features_cnn_dir_old / "feature_cols.npy", allow_pickle=True).tolist()
gesture_classes = np.load(all_features_cnn_dir_old / "gesture_classes.npy", allow_pickle=True)

imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
thm_cols = [c for c in feature_cols if c.startswith('thm_')]
tof_cols = [c for c in feature_cols if c.startswith('tof_')]

models_all_features_cnn_old = []
for fold in range(5):
    model = ThreeBranchModelCNN([12, 11, 12], len(thm_cols), 
                                len(tof_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(all_features_cnn_dir_old / f"{model_all_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_all_features_cnn_old.append(model)

In [22]:
models_all_features = models_all_features_lstm + models_all_features_lstm2 + \
    models_all_features_attention + models_all_features_cnn
len(models_all_features)

20

In [23]:
models_all_features_old = models_all_features_lstm_old + models_all_features_lstm2_old + \
    models_all_features_attention_old + models_all_features_cnn_old
len(models_all_features_old)

20

# 4. IMU features models

## 4.1. IMU features - LSTM

In [24]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class ResidualSEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, drop=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.se = SEBlock(out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(drop)
        self.pool = nn.MaxPool1d(2)
        
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, padding='same', bias = False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        
        out += identity
        out = self.relu(out)
        out = self.pool(out)
        out = self.dropout(out)
        return out

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Tanh()
        )
        
    def forward(self, x):
        weights = self.attention(x).squeeze(-1)
        weights = F.softmax(weights, dim=1).unsqueeze(-1)
        context = torch.sum(x * weights, dim=1)
        return context
    
class OneBranchModelLSTM(nn.Module):
    def __init__(self, imu_dim, n_classes):
        super().__init__()
        self.imu_dim = imu_dim
        
        # IMU branchs
        self.imu_branch1 = nn.Sequential(
            ResidualSEBlock(12, 64, 3, drop=0.3),
            ResidualSEBlock(64, 128, 5, drop=0.3)
        )
        self.imu_branch2 = nn.Sequential(
            ResidualSEBlock(11, 64, 3, drop=0.3),
            ResidualSEBlock(64, 128, 5, drop=0.3)
        )
        self.imu_branch3 = nn.Sequential(
            ResidualSEBlock(12, 64, 3, drop=0.3),
            ResidualSEBlock(64, 128, 5, drop=0.3)
        )
        
        # BiLSTM
        self.bilstm1 = nn.LSTM(128, 128, bidirectional=True, batch_first=True)
        self.dropout1 = nn.Dropout(0.4)
        self.bilstm2 = nn.LSTM(128, 128, bidirectional=True, batch_first=True)
        self.dropout2 = nn.Dropout(0.4)
        self.bilstm3 = nn.LSTM(128, 128, bidirectional=True, batch_first=True)
        self.dropout3 = nn.Dropout(0.4)       
        
        # Attention
        self.attention1 = AttentionLayer(256)
        self.attention2 = AttentionLayer(256)
        self.attention3 = AttentionLayer(256)
        
        # Dense layers
        self.fc_layers = nn.Sequential(
            nn.Linear(256*3, 512, bias = False),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256, bias = False),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        imu = x[:, :, :self.imu_dim]

        imu1 = self.imu_branch1(imu[:, :, :12].transpose(1, 2)).transpose(1, 2)
        lstm_out1, _ = self.bilstm1(imu1)
        lstm_out1 = self.dropout1(lstm_out1)
        attended1 = self.attention1(lstm_out1)

        imu2 = self.imu_branch2(imu[:, :, 12:23].transpose(1, 2)).transpose(1, 2)
        lstm_out2, _ = self.bilstm2(imu2)
        lstm_out2 = self.dropout2(lstm_out2)
        attended2 = self.attention2(lstm_out2)

        imu3 = self.imu_branch3(imu[:, :, 23:].transpose(1, 2)).transpose(1, 2)
        lstm_out3, _ = self.bilstm3(imu3)
        lstm_out3 = self.dropout3(lstm_out3)
        attended3 = self.attention3(lstm_out3)

        merged = torch.cat((attended1, attended2, attended3), dim=1)

        out = self.fc_layers(merged)
        return out 

In [25]:
imu_cols = np.load(imu_features_lstm_dir / "feature_cols_imu.npy", allow_pickle=True).tolist()
gesture_classes = np.load(imu_features_lstm_dir / "gesture_classes.npy", allow_pickle=True)

models_imu_features_lstm = []
for fold in range(5):
    model = OneBranchModelLSTM(len(imu_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(imu_features_lstm_dir / f"{model_imu_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_imu_features_lstm.append(model)

In [26]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class ResidualSEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, drop=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.se = SEBlock(out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(drop)
        self.pool = nn.MaxPool1d(2)
        
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, padding='same', bias = False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        
        out += identity
        out = self.relu(out)
        out = self.pool(out)
        out = self.dropout(out)
        return out

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Tanh()
        )
        
    def forward(self, x):
        weights = self.attention(x).squeeze(-1)
        weights = F.softmax(weights, dim=1).unsqueeze(-1)
        context = torch.sum(x * weights, dim=1)
        return context
    
class CrossAttention(nn.Module):
    """
    Cross-attention module for multi-branch feature fusion.
    Allows each branch to attend to features from other branches.
    """
    def __init__(self, feature_dim, num_heads=8, dropout=0.1):
        super(CrossAttention, self).__init__()
        self.feature_dim = feature_dim
        self.num_heads = num_heads
        self.head_dim = feature_dim // num_heads
        
        assert feature_dim % num_heads == 0, "feature_dim must be divisible by num_heads"
        
        # Linear projections for queries, keys, and values for each branch
        self.q_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.k_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.v_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        
        # Output projection
        self.out_linear = nn.Linear(feature_dim, feature_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Layer normalization for residual connection
        self.layer_norm = nn.LayerNorm(feature_dim)
        
    def forward(self, query_branch, key_value_branches):
        """
        Args:
            query_branch: (B, T, C) - the branch that will be updated
            key_value_branches: List of (B, T, C) tensors - branches to attend to
        Returns:
            Updated query_branch with cross-attention: (B, T, C)
        """
        B, T, C = query_branch.shape
        
        # Create queries from the query branch
        Q = self.q_linear(query_branch)  # (B, T, C)
        
        # Concatenate all key-value branches for attention
        all_kv = torch.stack(key_value_branches, dim=1)  # (B, num_branches, T, C)
        num_branches = all_kv.shape[1]
        
        # Reshape for multi-head attention
        all_kv = all_kv.reshape(B, num_branches * T, C)  # (B, num_branches*T, C)
        
        K = self.k_linear(all_kv)  # (B, num_branches*T, C)
        V = self.v_linear(all_kv)  # (B, num_branches*T, C)
        
        # Reshape for multi-head attention
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, T, head_dim)
        K = K.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        V = V.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, num_heads, T, num_branches*T)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention to values
        attn_output = torch.matmul(attn_weights, V)  # (B, num_heads, T, head_dim)
        
        # Concatenate heads and put through final linear layer
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        output = self.out_linear(attn_output)
        
        # Residual connection and layer normalization
        output = self.layer_norm(query_branch + output)
        
        return output

class IMUCrossAttentionFusion(nn.Module):
    """
    Cross-attention fusion module for three IMU branches.
    Each branch attends to the other two branches.
    """
    def __init__(self, feature_dim=256, num_heads=8, dropout=0.1):
        super(IMUCrossAttentionFusion, self).__init__()
        
        # Cross-attention modules for each branch
        self.cross_attn1 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn2 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn3 = CrossAttention(feature_dim, num_heads, dropout)
        
    def forward(self, imu1, imu2, imu3):
        """
        Args:
            imu1, imu2, imu3: (B, T, C) tensors from each IMU branch
        Returns:
            Tuple of enhanced features: (enhanced_imu1, enhanced_imu2, enhanced_imu3)
        """
        # Each branch attends to the other two branches
        enhanced_imu1 = self.cross_attn1(imu1, [imu2, imu3])
        enhanced_imu2 = self.cross_attn2(imu2, [imu1, imu3])
        enhanced_imu3 = self.cross_attn3(imu3, [imu1, imu2])
        
        return enhanced_imu1, enhanced_imu2, enhanced_imu3
    
class OneBranchModelLSTM2(nn.Module):
    def __init__(self, imu_dim, n_classes):
        super().__init__()
        self.imu_dim = imu_dim
        
        # IMU branchs
        self.imu_branch1 = nn.Sequential(
            ResidualSEBlock(12, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        self.imu_branch2 = nn.Sequential(
            ResidualSEBlock(11, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        self.imu_branch3 = nn.Sequential(
            ResidualSEBlock(12, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        
        self.cross_attention_fusion = IMUCrossAttentionFusion(
            feature_dim=256, num_heads=8, dropout=0.1
        )

        # BiLSTM
        self.bilstm = nn.LSTM(256*3, 512, num_layers=1, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(0.4)   

        # Attention
        self.attention = AttentionLayer(1024)
        
        # Dense layers
        self.fc_layers = nn.Sequential(
            nn.Linear(1024, 512, bias = False),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256, bias = False),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        imu = x[:, :, :self.imu_dim]

        imu1 = self.imu_branch1(imu[:, :, :12].transpose(1, 2)).transpose(1, 2)
        imu2 = self.imu_branch2(imu[:, :, 12:23].transpose(1, 2)).transpose(1, 2)
        imu3 = self.imu_branch3(imu[:, :, 23:].transpose(1, 2)).transpose(1, 2)

        # Apply cross-attention fusion between branches
        imu1, imu2, imu3 = self.cross_attention_fusion(imu1, imu2, imu3)

        merged = torch.cat((imu1, imu2, imu3), dim=2)

        lstm_out, _ = self.bilstm(merged)
        lstm_out = self.dropout(lstm_out)

        attended = self.attention(lstm_out)
 
        out = self.fc_layers(attended)
        return out 

In [27]:
imu_cols = np.load(imu_features_lstm_dir2 / "feature_cols_imu.npy", allow_pickle=True).tolist()
gesture_classes = np.load(imu_features_lstm_dir2 / "gesture_classes.npy", allow_pickle=True)

models_imu_features_lstm2 = []
for fold in range(5):
    model = OneBranchModelLSTM2(len(imu_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(imu_features_lstm_dir2 / f"{model_imu_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_imu_features_lstm2.append(model)

## 4.2. IMU features - attention

In [28]:
class SEBlock(nn.Module):
    def __init__(self, channel, reduction=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Sequential(
            nn.Linear(channel, channel // reduction, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(channel // reduction, channel, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1)
        return x * y.expand_as(x)

class ResidualSEBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, drop=0.3):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding='same', bias = False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        self.se = SEBlock(out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(drop)
        self.pool = nn.MaxPool1d(2)
        
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, padding='same', bias = False),
                nn.BatchNorm1d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.se(out)
        
        out += identity
        out = self.relu(out)
        out = self.pool(out)
        out = self.dropout(out)
        return out

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
            
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        output = torch.matmul(attention_weights, V)
        return output, attention_weights
    
    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()
        
        # Residual connection
        residual = x
        
        # Linear transformations and reshape
        Q = self.W_q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Apply attention
        attn_output, _ = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, seq_len, d_model)
        
        # Final linear layer
        output = self.W_o(attn_output)
        
        # Add residual connection and layer norm
        output = self.layer_norm(output + residual)
        
        return output

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_length=5000):
        super().__init__()
        
        pe = torch.zeros(max_length, d_model)
        position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
        
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                           (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Tanh()
        )
        
    def forward(self, x):
        weights = self.attention(x).squeeze(-1)
        weights = F.softmax(weights, dim=1).unsqueeze(-1)
        context = torch.sum(x * weights, dim=1)
        return context

class OneBranchModelAtt(nn.Module):
    def __init__(self, imu_dim, n_classes):
        super().__init__()
        self.imu_dim = imu_dim
        
        # IMU branches
        self.imu_branch1 = nn.Sequential(
            ResidualSEBlock(12, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        self.imu_branch2 = nn.Sequential(
            ResidualSEBlock(11, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        self.imu_branch3 = nn.Sequential(
            ResidualSEBlock(12, 128, 3, drop=0.3),
            ResidualSEBlock(128, 256, 5, drop=0.3)
        )
        
        # Positional encoding for attention layers
        self.pos_encoding1 = PositionalEncoding(256)
        self.pos_encoding2 = PositionalEncoding(256)
        self.pos_encoding3 = PositionalEncoding(256)
        
        # Multi-head self-attention layers (replacing BiLSTM)
        self.attention_block1 = nn.Sequential(
            MultiHeadSelfAttention(256, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(256, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(256, num_heads=8, dropout=0.4)
        )
        self.attention_block2 = nn.Sequential(
            MultiHeadSelfAttention(256, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(256, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(256, num_heads=8, dropout=0.4)
        )
        self.attention_block3 = nn.Sequential(
            MultiHeadSelfAttention(256, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(256, num_heads=8, dropout=0.4),
            MultiHeadSelfAttention(256, num_heads=8, dropout=0.4)
        )
        
        # Final attention for aggregation
        self.attention1 = AttentionLayer(256)
        self.attention2 = AttentionLayer(256)
        self.attention3 = AttentionLayer(256)
        
        # Dense layers
        self.fc_layers = nn.Sequential(
            nn.Linear(256*3, 512, bias = False),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256, bias = False),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, n_classes)
        )

    def forward(self, x):
        imu = x[:, :, :self.imu_dim]

        # Branch 1
        imu1 = self.imu_branch1(imu[:, :, :12].transpose(1, 2)).transpose(1, 2)
        imu1 = self.pos_encoding1(imu1)
        attn_out1 = self.attention_block1(imu1)
        attended1 = self.attention1(attn_out1)

        # Branch 2
        imu2 = self.imu_branch2(imu[:, :, 12:23].transpose(1, 2)).transpose(1, 2)
        imu2 = self.pos_encoding2(imu2)
        attn_out2 = self.attention_block2(imu2)
        attended2 = self.attention2(attn_out2)

        # Branch 3
        imu3 = self.imu_branch3(imu[:, :, 23:].transpose(1, 2)).transpose(1, 2)
        imu3 = self.pos_encoding3(imu3)
        attn_out3 = self.attention_block3(imu3)
        attended3 = self.attention3(attn_out3)

        # Merge all branches
        merged = torch.cat((attended1, attended2, attended3), dim=1)

        # Final classification
        out = self.fc_layers(merged)
        return out


In [29]:
imu_cols = np.load(imu_features_attention_dir / "feature_cols_imu.npy", allow_pickle=True).tolist()
gesture_classes = np.load(imu_features_attention_dir / "gesture_classes.npy", allow_pickle=True)

models_imu_features_attention = []
for fold in range(5):
    model = OneBranchModelAtt(len(imu_cols), len(gesture_classes)).to(device)
    state_dict = torch.load(imu_features_attention_dir / f"{model_imu_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_imu_features_attention.append(model)

## 4.3. IMU features - CNN

In [30]:
class CoordAttention(nn.Module):
    """
    Coordinate Attention for Sequences.
    Input Dimension: (B, T, C)
    Output Dimension: (B, T, C)
    """
    def __init__(self, channels, reduction=4):
        super(CoordAttention, self).__init__()
        self.mid_channels = max(8, channels // reduction)

        self.compression = nn.Sequential(
            nn.Conv1d(channels, self.mid_channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(self.mid_channels),
            nn.SiLU(inplace=True)
        )

        # Attention branches
        self.time_conv = nn.Conv1d(1, 1, kernel_size=5, padding=2, bias=False)  
        self.channel_conv = nn.Conv1d(self.mid_channels, channels, kernel_size=1, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: (B, T, C)
        x_p = x.permute(0, 2, 1)  # (B, C, T)
        f   = self.compression(x_p)  # (B, rC, T)

        ## Time Attention (B, 1, T)
        f_t = f.mean(dim=1, keepdim=True)      
        time_attn = self.sigmoid(self.time_conv(f_t))  

        ## Channel Attention (B, C, 1)
        f_c = f.mean(dim=2, keepdim=True)      
        channel_attn = self.sigmoid(self.channel_conv(f_c)) 

        ## (B, T, C)
        out = (x_p * time_attn * channel_attn).permute(0,2,1)
        return out
    
class ResCNNBlock(nn.Module):
    """
    Residual CNN Block with Coordinate Attention. 
    Input Dimension: (B, T_in, C_in)
    Output Dimension: (B, T_out, C_out)
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, pool_size=2, drop=0.2):
        super(ResCNNBlock, self).__init__()

        ## CNN model
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size, padding='same', bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size, padding='same', bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )
        ## Attention module
        self.attention = CoordAttention(out_channels)
        ## Projection path for skip connection
        self.shortcut_proj = None
        if in_channels != out_channels:
            self.shortcut_proj = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, padding='same', bias=False),
                nn.BatchNorm1d(out_channels)
            )
        ## Final processing components
        self.relu_final = nn.ReLU(inplace=True)
        if pool_size is not None: 
            self.max_pool = nn.MaxPool1d(pool_size)
        else: 
            self.max_pool = None
        self.dropout = nn.Dropout(drop)

    def forward(self, x):
        # (B, C_in, T)
        shortcut = x.permute(0,2,1)
        x_p = x.permute(0,2,1)
        
        out = self.cnn(x_p)                         # (B, C_out, T)
        out = self.attention(out.permute(0, 2, 1))  # (B, T, C_out)
        out = out.permute(0, 2, 1)                  # (B, C_out, T)
        
        ## Handle shortcut connection -> (B, C_out, T)
        if self.shortcut_proj:
            shortcut = self.shortcut_proj(shortcut)

        ## Residual connection
        out += shortcut
        out = self.relu_final(out)
        if self.max_pool is not None:
            ## (B, C_out, T) -> (B, C_out, T//pool_size)
            out = self.max_pool(out)
        out = self.dropout(out)
        ## (B, T_out, C_out)
        out = out.permute(0, 2, 1)
        return out
    
class TempCNNBlock(nn.Module):
    """
    Temporal CNN Block with Dilated Convolution, Attention, and Skip Connection.
    Input Dimension: (B, T, C_in)
    Output Dimension: (B, T, C_out)
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, drop=0.25):
        super(TempCNNBlock, self).__init__()
        ## Padding computation
        padding = (kernel_size - 1) * dilation // 2
        ## Temporal Convolution block
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size,
                      dilation=dilation, padding=padding, bias=False),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )
        ## Attention module
        self.attention = CoordAttention(out_channels)
        ## Dropout
        self.dropout = nn.Dropout(drop)
        
        ## Shortcut for skip connection
        self.shortcut_proj = None
        if in_channels != out_channels:
            self.shortcut_proj = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, padding='same', bias=False),
                nn.BatchNorm1d(out_channels)
            )
        self.relu_final = nn.ReLU()
        
    def forward(self, x):
        ## Processes skip connection
        # (B, C_in, T)
        x_p      = x.permute(0,2,1)
        shortcut = x.permute(0,2,1)
        # (B, C_out, T)
        if self.shortcut_proj:
            shortcut = self.shortcut_proj(shortcut)
        ## Processes output
        out = self.conv(x_p)                        # (B, C_out, T)
        out = self.attention(out.permute(0,2,1))    # (B, T, C_out)
        out = out.permute(0,2,1)                    # (B, C_out, T)
        ## Skip connection
        out = self.relu_final(out + shortcut) # (B, C_out, T)
        out = self.dropout(out)
        return out.permute(0,2,1)  # (B, T, C_out)
    
class MLPAttention(nn.Module):
    def __init__(self, feature_dim):
        super(MLPAttention, self).__init__()
        self.attn = nn.Sequential(
            nn.Linear(feature_dim, feature_dim//8),
            nn.SiLU(inplace=True),
            nn.Linear(feature_dim//8, 1)
        )

    def forward(self, x):
        # inputs shape: (B, T, C)
        weights = self.attn(x)  # (B, T, 1)
        weights = F.softmax(weights, dim=1)  # (B, T, 1)
        context = (x * weights).sum(dim=1)  # (B, C)
        return context


class CrossAttention(nn.Module):
    """
    Cross-attention module for multi-branch feature fusion.
    Allows each branch to attend to features from other branches.
    """
    def __init__(self, feature_dim, num_heads=8, dropout=0.1):
        super(CrossAttention, self).__init__()
        self.feature_dim = feature_dim
        self.num_heads = num_heads
        self.head_dim = feature_dim // num_heads
        
        assert feature_dim % num_heads == 0, "feature_dim must be divisible by num_heads"
        
        # Linear projections for queries, keys, and values for each branch
        self.q_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.k_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        self.v_linear = nn.Linear(feature_dim, feature_dim, bias=False)
        
        # Output projection
        self.out_linear = nn.Linear(feature_dim, feature_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Layer normalization for residual connection
        self.layer_norm = nn.LayerNorm(feature_dim)
        
    def forward(self, query_branch, key_value_branches):
        """
        Args:
            query_branch: (B, T, C) - the branch that will be updated
            key_value_branches: List of (B, T, C) tensors - branches to attend to
        Returns:
            Updated query_branch with cross-attention: (B, T, C)
        """
        B, T, C = query_branch.shape
        
        # Create queries from the query branch
        Q = self.q_linear(query_branch)  # (B, T, C)
        
        # Concatenate all key-value branches for attention
        all_kv = torch.stack(key_value_branches, dim=1)  # (B, num_branches, T, C)
        num_branches = all_kv.shape[1]
        
        # Reshape for multi-head attention
        all_kv = all_kv.reshape(B, num_branches * T, C)  # (B, num_branches*T, C)
        
        K = self.k_linear(all_kv)  # (B, num_branches*T, C)
        V = self.v_linear(all_kv)  # (B, num_branches*T, C)
        
        # Reshape for multi-head attention
        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, T, head_dim)
        K = K.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        V = V.view(B, num_branches * T, self.num_heads, self.head_dim).transpose(1, 2)  # (B, num_heads, num_branches*T, head_dim)
        
        # Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, num_heads, T, num_branches*T)
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        # Apply attention to values
        attn_output = torch.matmul(attn_weights, V)  # (B, num_heads, T, head_dim)
        
        # Concatenate heads and put through final linear layer
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, C)  # (B, T, C)
        output = self.out_linear(attn_output)
        
        # Residual connection and layer normalization
        output = self.layer_norm(query_branch + output)
        
        return output

class IMUCrossAttentionFusion(nn.Module):
    """
    Cross-attention fusion module for three IMU branches.
    Each branch attends to the other two branches.
    """
    def __init__(self, feature_dim=256, num_heads=8, dropout=0.1):
        super(IMUCrossAttentionFusion, self).__init__()
        
        # Cross-attention modules for each branch
        self.cross_attn1 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn2 = CrossAttention(feature_dim, num_heads, dropout)
        self.cross_attn3 = CrossAttention(feature_dim, num_heads, dropout)
        
    def forward(self, imu1, imu2, imu3):
        """
        Args:
            imu1, imu2, imu3: (B, T, C) tensors from each IMU branch
        Returns:
            Tuple of enhanced features: (enhanced_imu1, enhanced_imu2, enhanced_imu3)
        """
        # Each branch attends to the other two branches
        enhanced_imu1 = self.cross_attn1(imu1, [imu2, imu3])
        enhanced_imu2 = self.cross_attn2(imu2, [imu1, imu3])
        enhanced_imu3 = self.cross_attn3(imu3, [imu1, imu2])
        
        return enhanced_imu1, enhanced_imu2, enhanced_imu3

class OneBranchModelCNN(nn.Module):
    def __init__(self, imu_dim, n_classes):
        super().__init__()
        self.imu_dim = imu_dim
        self.n_classes = n_classes
        
        # IMU branches
        self.imu_branch1 = nn.Sequential(
            ResCNNBlock(imu_dim[0], 128, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(128, 256, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(256, 256, kernel_size=5, pool_size=None, drop=0.2),
        )
        self.imu_branch2 = nn.Sequential(
            ResCNNBlock(imu_dim[1], 128, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(128, 256, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(256, 256, kernel_size=5, pool_size=None, drop=0.2),
        )
        self.imu_branch3 = nn.Sequential(
            ResCNNBlock(imu_dim[2], 128, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(128, 256, kernel_size=3, pool_size=2, drop=0.2),
            ResCNNBlock(256, 256, kernel_size=5, pool_size=None, drop=0.2),
        )

        self.cross_attention_fusion = IMUCrossAttentionFusion(
            feature_dim=256, num_heads=8, dropout=0.1
        )

        self.merged_attention = CoordAttention(256*3)
        self.merged_dropout   = nn.Dropout(1/3)
        self.temporalCNN      = nn.Sequential(
            TempCNNBlock(256*3, 512, kernel_size=3, dilation=1, drop=0.2),
            TempCNNBlock(512, 512, kernel_size=3, dilation=2, drop=0.2),
            TempCNNBlock(512, 256, kernel_size=3, dilation=4, drop=0.2),
            TempCNNBlock(256, 256, kernel_size=3, dilation=8, drop=0.2),
        )

        self.classifier_attention = MLPAttention(256)
        self.classifier_dropout   = nn.Dropout(0.2)
        self.classifier = nn.Sequential(
            nn.Linear(256, 128, bias=False),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, self.n_classes)
        )

    def forward(self, x):
        imu = x[:, :, :sum(self.imu_dim)]

        # Branch 1
        imu1 = self.imu_branch1(imu[:, :, :self.imu_dim[0]])

        # Branch 2
        imu2 = self.imu_branch2(imu[:, :, self.imu_dim[0]:(self.imu_dim[0]+self.imu_dim[1])])

        # Branch 3
        imu3 = self.imu_branch3(imu[:, :, (self.imu_dim[0]+self.imu_dim[1]):])

        # Apply cross-attention fusion between branches
        enhanced_imu1, enhanced_imu2, enhanced_imu3 = self.cross_attention_fusion(imu1, imu2, imu3)

        # Merge all branches
        x_merged = torch.cat((enhanced_imu1, enhanced_imu2, enhanced_imu3), dim=2)
        x_merged = self.merged_attention(x_merged)
        x_merged = self.merged_dropout(x_merged)
        out = self.temporalCNN(x_merged) # (B, 37, 128)

        ## Provides output to classifier
        out = self.classifier_attention(out) # (B, 128)
        out = self.classifier_dropout(out)
        out = self.classifier(out)           # (B, 18)
        return out

In [31]:
imu_cols = np.load(imu_features_cnn_dir / "feature_cols_imu.npy", allow_pickle=True).tolist()
gesture_classes = np.load(imu_features_cnn_dir / "gesture_classes.npy", allow_pickle=True)

models_imu_features_cnn = []
for fold in range(5):
    model = OneBranchModelCNN([12, 11, 12], len(gesture_classes)).to(device)
    state_dict = torch.load(imu_features_cnn_dir / f"{model_imu_features_path}{fold}.pt", 
                            map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model = model.eval()
    models_imu_features_cnn.append(model)

In [32]:
models_imu_features = models_imu_features_lstm + models_imu_features_lstm2 + \
    models_imu_features_attention + models_imu_features_cnn
len(models_imu_features)

20

# Devin's models

In [33]:
import polars as pl
import numpy as np
import glob
import os

from pickle import load

import torch
import torch.nn as nn
import torch.nn.functional as F

import kaggle_evaluation.cmi_inference_server

from scipy.spatial.transform import Rotation as R

In [34]:
device2 = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

In [35]:
imu_cols = ["acc_x","acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
diff_cols = ["diff_x", "diff_y", "diff_z"]
extra_imu_cols = ["mag", "angle"]
extra_extra_imu = ["ang_vel_x", "ang_vel_y", "ang_vel_z", "ang_dist"]
thm_cols = ["thm_" + str(x) for x in range(1, 6)]
thm_diff_cols = ["thm_" + str(x) +"_diff" for x in range(1, 6)] 
tof_cols = ["tof_agg" + str(x)  for x in range(0,45)]
tof_diff_cols = ["tof_agg" + str(x) + "_diff" for x in range(0,45)]

all_model_cols = imu_cols+diff_cols+extra_imu_cols+extra_extra_imu+thm_cols+tof_cols
all_model_diff_cols = imu_cols+diff_cols+extra_imu_cols+extra_extra_imu+thm_cols+tof_cols+thm_diff_cols+tof_diff_cols
imu_model_cols = imu_cols+diff_cols+extra_imu_cols+extra_extra_imu

In [36]:
# only issue with this is its either all missing or none, 
def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
    """
    Handle missing values in quaternion data intelligently
    
    Key insight: Quaternions must have unit length |q| = 1
    If one component is missing, we can reconstruct it from the others
    """
    rot_cleaned = rot_data.copy()
    
    for i in range(len(rot_data)):
        row = rot_data[i]
        missing_count = np.isnan(row).sum()
        
        if missing_count == 0:
            # No missing values, normalize to unit quaternion
            norm = np.linalg.norm(row)
            if norm > 1e-8:
                rot_cleaned[i] = row / norm
            else:
                rot_cleaned[i] = [1.0, 0.0, 0.0, 0.0]  # Identity quaternion
                
        elif missing_count == 1:
            # One missing value, reconstruct using unit quaternion constraint
            # |w|² + |x|² + |y|² + |z|² = 1
            missing_idx = np.where(np.isnan(row))[0][0]
            valid_values = row[~np.isnan(row)]
            
            sum_squares = np.sum(valid_values**2)
            if sum_squares <= 1.0:
                missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                # Choose sign for continuity with previous quaternion
                if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                    if rot_cleaned[i-1, missing_idx] < 0:
                        missing_value = -missing_value
                rot_cleaned[i, missing_idx] = missing_value
                rot_cleaned[i, ~np.isnan(row)] = valid_values
            else:
                rot_cleaned[i] = [1.0, 0.0, 0.0, 0.0]
        else:
            # More than one missing value, use identity quaternion
            rot_cleaned[i] = [1.0, 0.0, 0.0, 0.0]
    
    return rot_cleaned

# thanks: https://www.kaggle.com/code/nksusth/lb-0-78-quaternions-tf-bilstm-gru-attention
def calculate_angular_velocity_from_quat2(rot_data, time_delta=1/10): # Assuming 10Hz sampling rate
    if isinstance(rot_data, pl.DataFrame):
        quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].to_numpy()
    else:
        quat_values = rot_data

    num_samples = quat_values.shape[0]
    angular_vel = np.zeros((num_samples, 3))

    for i in range(num_samples - 1):
        q_t = quat_values[i]
        q_t_plus_dt = quat_values[i+1]

        if np.all(np.isnan(q_t)) or np.all(np.isclose(q_t, 0)) or \
           np.all(np.isnan(q_t_plus_dt)) or np.all(np.isclose(q_t_plus_dt, 0)):
            continue

        try:
            rot_t = R.from_quat(q_t)
            rot_t_plus_dt = R.from_quat(q_t_plus_dt)

            # Calculate the relative rotation
            delta_rot = rot_t.inv() * rot_t_plus_dt
            
            # Convert delta rotation to angular velocity vector
            # The rotation vector (Euler axis * angle) scaled by 1/dt
            # is a good approximation for small delta_rot
            angular_vel[i, :] = delta_rot.as_rotvec() / time_delta
        except ValueError:
            # If quaternion is invalid, angular velocity remains zero
            pass
            
    return angular_vel

# thanks: https://www.kaggle.com/code/nksusth/lb-0-78-quaternions-tf-bilstm-gru-attention
def calculate_angular_distance2(rot_data):
    if isinstance(rot_data, pl.DataFrame):
        quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].to_numpy()
    else:
        quat_values = rot_data

    num_samples = quat_values.shape[0]
    angular_dist = np.zeros(num_samples)

    for i in range(num_samples - 1):
        q1 = quat_values[i]
        q2 = quat_values[i+1]

        if np.all(np.isnan(q1)) or np.all(np.isclose(q1, 0)) or \
           np.all(np.isnan(q2)) or np.all(np.isclose(q2, 0)):
            angular_dist[i] = 0 # Или np.nan, в зависимости от желаемого поведения
            continue
        try:
            # Преобразование кватернионов в объекты Rotation
            r1 = R.from_quat(q1)
            r2 = R.from_quat(q2)

            # Вычисление углового расстояния: 2 * arccos(|real(p * q*)|)
            # где p* - сопряженный кватернион q
            # В scipy.spatial.transform.Rotation, r1.inv() * r2 дает относительное вращение.
            # Угол этого относительного вращения - это и есть угловое расстояние.
            relative_rotation = r1.inv() * r2
            
            # Угол rotation vector соответствует угловому расстоянию
            # Норма rotation vector - это угол в радианах
            angle = np.linalg.norm(relative_rotation.as_rotvec())
            angular_dist[i] = angle
        except ValueError:
            angular_dist[i] = 0 # В случае недействительных кватернионов
            pass
            
    return angular_dist

def preprocess_left_handed(l_tr):
    rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
    rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
    
    norms = np.linalg.norm(rot_scipy, axis=1)
    if np.any(norms < 1e-8):
        # Replace problematic quaternions with identity
        mask = norms < 1e-8
        rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        print("yes")
    
    r = R.from_quat(rot_scipy)
    tmp = r.as_euler("xyz")
    tmp[:,1] = - tmp[:,1]
    tmp[:,2] = - tmp[:,2] 
    r = R.from_euler("xyz", tmp)
    tmp = r.as_quat()
    l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
    l_tr = l_tr.with_columns(-pl.col("acc_x"))
    
    tmp = l_tr[["thm_3", "thm_5"]]
    tmp.columns = ["thm_5", "thm_3"]
    l_tr = l_tr.with_columns(tmp)
    
    swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
    swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
    
    swap_1_2_4 = list()
    for i in range(0,64,8):
        ll = list()
        for (k,l) in swap_1_2_4_base:
            ll.append([k+i, l+i])
        swap_1_2_4 += ll
    
    swap_3_5 = list()
    for i in range(8):
        ll = list()
        for (k,l) in swap_3_5_base:
            ll.append([k+i, l+i])
        swap_3_5 += ll
    
    l_df = l_tr
    
    for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
        l_tr = l_tr.with_columns(l_df[k].alias(l))
    
    for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
        l_tr = l_tr.with_columns(l_df[l].alias(k))
    
    l_df = l_tr
    
    for i in [1,2,4]:
        for (k, l) in swap_1_2_4:
            l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
    
    for i in [3,5]:
        for (k, l) in swap_3_5:
            l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
    return l_tr

In [37]:
def prep_data(df, target_cols, scaler):

    df = df.with_columns(pl.col("acc_x").diff().alias("diff_x"))
    df = df.with_columns(pl.col("acc_y").diff().alias("diff_y"))
    df = df.with_columns(pl.col("acc_z").diff().alias("diff_z"))
    df = df.with_columns((pl.col("acc_x").pow(2) + pl.col("acc_y").pow(2) + pl.col("acc_z").pow(2)).sqrt().alias("mag"))
    df = df.with_columns((2*pl.col("rot_w").arccos()).alias("angle"))

    ang_vel = calculate_angular_velocity_from_quat2(df)
    ang_dist = calculate_angular_distance2(df)
    df = df.with_columns(pl.DataFrame(ang_vel, schema=["ang_vel_x", "ang_vel_y", "ang_vel_z"]))
    df = df.with_columns(pl.Series(ang_dist).alias("ang_dist"))


    
    if len(target_cols) > 20:
        groups1 = [[0,1,8,9],[3,4,11,12],[6,7,14,15],[24,25,32,33],[27,28,35,36],
                   [30,31,38,39],[48,49,56,57], [51,52,59,60], [54,55,62,63]]
        
        cols1 = []
        for i in groups1:
            for k in range(1, 6):
                c = list()
                for j in i:
                    c.append("tof_"+str(k)+"_v"+str(j))
                cols1.append(c)               
    
        for e, cs in enumerate(cols1):
            t = df.select(pl.col(cs))
            t = t.with_columns(pl.all().replace(-1, None))
            t = t.with_columns(pl.mean_horizontal(pl.all()).alias("tof_agg" + str(e)))
            df = df.hstack(pl.DataFrame(t["tof_agg" + str(e)]))

        for i in range(1,6):
            df = df.with_columns(pl.col(f"thm_{i}").diff().alias(f"thm_{i}_diff"))
        for i in range(45):
            df = df.with_columns(pl.col(f"tof_agg{i}").diff().alias(f"tof_agg{i}_diff"))
    
    df = df.with_columns(pl.col(target_cols).replace([float("inf"), float("-inf"), float("nan")], None))
    
    scaled = pl.DataFrame(scaler.transform(df[target_cols]), schema=target_cols)
    df = df.with_columns(scaled)
    
    
    
    df = df.with_columns(pl.col(target_cols).replace([float("inf"), float("-inf"), float("nan")], None))
    df = df.fill_null(0)
    
    seq_length = 128+5
    x_ts = np.zeros((1,len(target_cols), seq_length))
    
    x_ts[0,:,-df.shape[0]:] = df[target_cols].to_numpy()[-seq_length:,:].T
    return x_ts

In [38]:
with open("/kaggle/input/cmi-detect-behavior/encoder.pkl", "rb") as f:
    encoder = load(f)

In [39]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, padding="same")
        self.bn1 = nn.BatchNorm1d(num_features=out_channels)
        self.conv2 = nn.Conv1d(in_channels=out_channels, out_channels=out_channels, kernel_size=kernel_size, padding="same")
        self.bn2 = nn.BatchNorm1d(num_features=out_channels)
        
        if in_channels != out_channels:
            self.res_layer = torch.nn.Conv1d(in_channels, out_channels, kernel_size=1, padding="same")
        else:
            self.res_layer = None
            
    def forward(self, x):
        res = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        if self.res_layer is None:
            x = torch.add(x, res)
        else: 
            x = torch.add(x, self.res_layer(res))
        return x 

class SEBlock(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.squeeze = nn.AdaptiveAvgPool1d(1)
        self.excitation = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        b, c, _ = x.size()
        y = self.squeeze(x).view(b, c)
        y = self.excitation(y).view(b, c, 1)
        return x * y.expand_as(x)

class ResidualBlockSE(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, padding="same")
        self.bn1 = nn.BatchNorm1d(num_features=out_channels)
        self.conv2 = nn.Conv1d(in_channels=out_channels, out_channels=out_channels, kernel_size=kernel_size, padding="same")
        self.bn2 = nn.BatchNorm1d(num_features=out_channels)
        self.se = SEBlock(out_channels)
        
        if in_channels != out_channels:
            self.res_layer = nn.Conv1d(in_channels, out_channels, kernel_size=1, padding="same")
            self.res_bn = nn.BatchNorm1d(out_channels)
        else:
            self.res_layer = None
            self.res_bn = None
            
    def forward(self, x):
        res = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = self.se(x)
        if self.res_layer is None:
            x = torch.add(x, res)
        else: 
            x = torch.add(x, self.res_bn(self.res_layer(res)))
        x = F.relu(x)
        return x  


class RNNetAllBig(nn.Module):
    def __init__(self, chan=128,n_all_feat=50, classes=18):
        super().__init__()

        self.block0_0 = ResidualBlock(4, chan//2, 3)
        self.block0_1 = ResidualBlock(chan//2, chan//2, 3)
        self.block0_2 = ResidualBlock(chan//2, chan//2, 3)

        self.block1_0 = ResidualBlock(5, chan//2, 3)
        self.block1_1 = ResidualBlock(chan//2, chan//2, 3)
        self.block1_2 = ResidualBlock(chan//2, chan//2, 3)

        self.block2_0 = ResidualBlock(3, chan//2, 3)
        self.block2_1 = ResidualBlock(chan//2, chan//2, 3)
        self.block2_2 = ResidualBlock(chan//2, chan//2, 3)

        self.block3_0 = ResidualBlock(4, chan//2,3)
        self.block3_1 = ResidualBlock(chan//2, chan//2,3)
        self.block3_2 = ResidualBlock(chan//2, chan//2, 3)

        self.block4_0 = ResidualBlock(n_all_feat, 2*chan,3)
        self.block4_1 = ResidualBlock(2*chan, 2*chan,5)

        self.block5_0 = ResidualBlock(16, chan,3)
        self.block5_1 = ResidualBlock(chan, chan,3)
        self.block5_2 = ResidualBlock(chan, chan, 3)
        
        self.rnn = nn.GRU(5*chan, 512,
            num_layers = 2, 
            batch_first =True,
            bidirectional=True)
        self.fc0 = nn.Linear(1024, 256)
        self.drop = nn.Dropout(0.2)
        self.fc1 = nn.Linear(256, classes)
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:16,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        x4 = self.block4_1(self.block4_0(x[:,16:,:]))
        x5 = self.block5_2(self.block5_1(self.block5_0(x[:,:16,:])))
        
        x = torch.cat([x0,x1,x2, x3, x4, x5], dim=1)
        
        x = x.permute(0, 2, 1)
        x,_ = self.rnn(x)
        x = self.drop(F.relu(self.fc0(x)))
        x = self.fc1(x)
        x = x[:,-30:,:].mean(axis=1)
        return x 

class RNNetAllBigSE(nn.Module):
    def __init__(self, chan=128, n_all_feat=50):
        super().__init__()

        self.block0_0 = ResidualBlockSE(4, chan//2, 3)
        self.block0_1 = ResidualBlockSE(chan//2, chan//2, 3)
        self.block0_2 = ResidualBlockSE(chan//2, chan//2, 3)

        self.block1_0 = ResidualBlockSE(5, chan//2, 3)
        self.block1_1 = ResidualBlockSE(chan//2, chan//2, 3)
        self.block1_2 = ResidualBlockSE(chan//2, chan//2, 3)

        self.block2_0 = ResidualBlockSE(3, chan//2, 3)
        self.block2_1 = ResidualBlockSE(chan//2, chan//2, 3)
        self.block2_2 = ResidualBlockSE(chan//2, chan//2, 3)

        self.block3_0 = ResidualBlockSE(4, chan//2,3)
        self.block3_1 = ResidualBlockSE(chan//2, chan//2,3)
        self.block3_2 = ResidualBlockSE(chan//2, chan//2, 3)

        self.block4_0 = ResidualBlockSE(n_all_feat, 2*chan,3)
        self.block4_1 = ResidualBlockSE(2*chan, 2*chan,5)

        self.block5_0 = ResidualBlockSE(16, chan,3)
        self.block5_1 = ResidualBlockSE(chan, chan,3)
        self.block5_2 = ResidualBlockSE(chan, chan, 3)
        
        self.rnn = nn.GRU(5*chan, 512,
            num_layers = 2, 
            batch_first =True,
            bidirectional=True)
        self.fc0 = nn.Linear(1024, 256)
        self.drop = nn.Dropout(0.2)
        self.fc1 = nn.Linear(256, 18)
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:16,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        x4 = self.block4_1(self.block4_0(x[:,16:,:]))
        x5 = self.block5_2(self.block5_1(self.block5_0(x[:,:16,:])))
        
        x = torch.cat([x0,x1,x2, x3, x4, x5], dim=1)
        
        x = x.permute(0, 2, 1)
        x,_ = self.rnn(x)
        x = self.drop(F.relu(self.fc0(x)))
        x = self.fc1(x)
        x = x[:,-30:,:].mean(axis=1)
        return x 


# https://www.kaggle.com/code/takoihiraokazu/cv-ensemble-sub-0607-1
class RnnModelAlldata(nn.Module):
    def __init__(
        self, dropout=0.2,
        input_numerical_size=66,
        numeraical_linear_size = 128,
        model_size = 128,
        linear_out = 128,
        out_size=18):
        super(RnnModelAlldata, self).__init__()
        self.numerical_linear  = nn.Sequential(
                nn.Linear(input_numerical_size, numeraical_linear_size),
                nn.LayerNorm(numeraical_linear_size)
            )
        
        self.rnn = nn.GRU(numeraical_linear_size, model_size,
                            num_layers = 2, 
                            bidirectional=True,
                            batch_first=True)
        self.linear_out  = nn.Sequential(
                nn.Linear(model_size*2, 
                          linear_out),
                nn.LayerNorm(linear_out),
                nn.Dropout(dropout),
                nn.Linear(linear_out, 
                          out_size))
        self._reinitialize()
        
    def _reinitialize(self):
        """
        Tensorflow/Keras-like initialization
        """
        for name, p in self.named_parameters():
            if 'rnn' in name:
                if 'weight_ih' in name:
                    nn.init.xavier_uniform_(p.data)
                elif 'weight_hh' in name:
                    nn.init.orthogonal_(p.data)
                elif 'bias_ih' in name:
                    p.data.fill_(0)
                    # Set forget-gate bias to 1
                    n = p.size(0)
                    p.data[(n // 4):(n // 2)].fill_(1)
                elif 'bias_hh' in name:
                    p.data.fill_(0)
    
    def forward(self, numerical_array):
        numerical_array = numerical_array.permute(0, 2, 1)
        numerical_embedding = self.numerical_linear(numerical_array)
        output,_ = self.rnn(numerical_embedding)
        output = self.linear_out(output)
        return output[:,-30:,:].mean(axis=1)

class NetAll(nn.Module):
    def __init__(self,n_blocks=2, chan=64, kern=5, kern_increase=4,n_all_feat=50):
        super().__init__()
        
        self.block0_0 = ResidualBlock(4, chan,3)
        self.block0_1 = ResidualBlock(chan, chan,3)
        self.block0_2 = ResidualBlock(chan, chan, 3)

        self.block1_0 = ResidualBlock(5, chan,3)
        self.block1_1 = ResidualBlock(chan, chan,3)
        self.block1_2 = ResidualBlock(chan, chan, 3)

        self.block2_0 = ResidualBlock(3, chan,3)
        self.block2_1 = ResidualBlock(chan, chan,3)
        self.block2_2 = ResidualBlock(chan, chan, 3)

        self.block3_0 = ResidualBlock(4, chan,3)
        self.block3_1 = ResidualBlock(chan, chan,3)
        self.block3_2 = ResidualBlock(chan, chan, 3)

        self.block4_0 = ResidualBlock(n_all_feat, 2*chan,3)
        self.block4_1 = ResidualBlock(2*chan, 2*chan,5)
        
        self.layers = nn.ModuleList()
        for i in range(n_blocks+1):
            if i == 0:
                self.layers.append(ResidualBlock(6*chan, 4*chan,kern))
            else:
                self.layers.append(ResidualBlock((4*chan)*2**((i-1)//2), (4*chan)*2**(i//2),kern+kern_increase*i))
        self.pool = nn.MaxPool1d(2,2)  
        self.fc0 = nn.Linear((4*chan)*2**(i//2), 18)
        
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:16,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        x4 = self.block4_1(self.block4_0(x[:,16:,:]))
        
        x = torch.cat([x0,x1,x2, x3, x4], dim=1)
        
        for i in range(len(self.layers)-1):
            x = self.pool(self.layers[i](x))
        x = self.layers[-1](x)
        x = torch.mean(x, dim=-1)
        x = self.fc0(x)
        return x 

class NetAllSE(nn.Module):
    def __init__(self,n_blocks=2, chan=64, kern=5, kern_increase=4,n_all_feat=50):
        super().__init__()
        
        self.block0_0 = ResidualBlockSE(4, chan,3)
        self.block0_1 = ResidualBlockSE(chan, chan,3)
        self.block0_2 = ResidualBlockSE(chan, chan, 3)

        self.block1_0 = ResidualBlockSE(5, chan,3)
        self.block1_1 = ResidualBlockSE(chan, chan,3)
        self.block1_2 = ResidualBlockSE(chan, chan, 3)

        self.block2_0 = ResidualBlockSE(3, chan,3)
        self.block2_1 = ResidualBlockSE(chan, chan,3)
        self.block2_2 = ResidualBlock(chan, chan, 3)

        self.block3_0 = ResidualBlockSE(4, chan,3)
        self.block3_1 = ResidualBlockSE(chan, chan,3)
        self.block3_2 = ResidualBlockSE(chan, chan, 3)

        self.block4_0 = ResidualBlockSE(n_all_feat, 2*chan,3)
        self.block4_1 = ResidualBlockSE(2*chan, 2*chan,5)
        
        self.layers = nn.ModuleList()
        for i in range(n_blocks+1):
            if i == 0:
                self.layers.append(ResidualBlockSE(6*chan, 4*chan,kern))
            else:
                self.layers.append(ResidualBlockSE((4*chan)*2**((i-1)//2), (4*chan)*2**(i//2),kern+kern_increase*i))
        self.pool = nn.MaxPool1d(2,2)  
        self.fc0 = nn.Linear((4*chan)*2**(i//2), 18)
        
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:16,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        x4 = self.block4_1(self.block4_0(x[:,16:,:]))
        
        x = torch.cat([x0,x1,x2, x3, x4], dim=1)
        
        for i in range(len(self.layers)-1):
            x = self.pool(self.layers[i](x))
        x = self.layers[-1](x)
        x = torch.mean(x, dim=-1)
        x = self.fc0(x)
        return x 






class RNNetTOFTHM(nn.Module):
    def __init__(self, chan=128):
        super().__init__()

        self.block4_0 = ResidualBlock(50, 2*chan,3)
        self.block4_1 = ResidualBlock(2*chan, 2*chan,5)
        
        self.rnn = nn.GRU(2*chan, 128,
            num_layers = 2, 
            batch_first =True,
            bidirectional=True)
        self.fc0 = nn.Linear(256, 256)
        self.drop = nn.Dropout(0.2)
        self.fc1 = nn.Linear(256, 18)
    def forward(self, x):
        x = self.block4_1(self.block4_0(x[:,16:,:]))
        
        x = x.permute(0, 2, 1)
        x,_ = self.rnn(x)
        x = self.drop(F.relu(self.fc0(x)))
        x = self.fc1(x)
        x = x[:,-30:,:].mean(axis=1)
        return x 


class RNNetImu(nn.Module):
    def __init__(self, chan=64):
        super().__init__()

        self.block0_0 = ResidualBlock(4, chan,3)
        self.block0_1 = ResidualBlock(chan, chan,3)
        self.block0_2 = ResidualBlock(chan, chan, 3)

        self.block1_0 = ResidualBlock(5, chan,3)
        self.block1_1 = ResidualBlock(chan, chan,3)
        self.block1_2 = ResidualBlock(chan, chan, 3)

        self.block2_0 = ResidualBlock(3, chan,3)
        self.block2_1 = ResidualBlock(chan, chan,3)
        self.block2_2 = ResidualBlock(chan, chan, 3)

        self.block3_0 = ResidualBlock(4, chan,3)
        self.block3_1 = ResidualBlock(chan, chan,3)
        self.block3_2 = ResidualBlock(chan, chan, 3)
        
        self.rnn = nn.GRU(4*chan, 128,
            num_layers = 2, 
            batch_first =True,
            bidirectional=True)

        self.fc0 = nn.Linear(256, 256)
        self.drop = nn.Dropout(0.2)
        self.fc1 = nn.Linear(256, 18)
                    
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        
        x = torch.cat([x0,x1,x2,x3], dim=1)
        
        x = x.permute(0, 2, 1)
        x,_ = self.rnn(x)
        x = self.drop(F.relu(self.fc0(x)))
        x = self.fc1(x)
        x = x[:,-30:,:].mean(axis=1)
        return x 



class NetImu(nn.Module):
    def __init__(self,n_blocks=2, chan=64, kern=5, kern_increase=4):
        super().__init__()
        
        self.block0_0 = ResidualBlock(4, chan,3)
        self.block0_1 = ResidualBlock(chan, chan,3)
        self.block0_2 = ResidualBlock(chan, chan, 3)

        self.block1_0 = ResidualBlock(5, chan,3)
        self.block1_1 = ResidualBlock(chan, chan,3)
        self.block1_2 = ResidualBlock(chan, chan, 3)

        self.block2_0 = ResidualBlock(3, chan,3)
        self.block2_1 = ResidualBlock(chan, chan,3)
        self.block2_2 = ResidualBlock(chan, chan, 3)

        self.block3_0 = ResidualBlock(4, chan,3)
        self.block3_1 = ResidualBlock(chan, chan,3)
        self.block3_2 = ResidualBlock(chan, chan, 3)
        
        self.layers = nn.ModuleList()
        for i in range(n_blocks+1):
            if i == 0:
                self.layers.append(ResidualBlock(4*chan, 4*chan,kern))
            else:
                self.layers.append(ResidualBlock((4*chan)*2**((i-1)//2), (4*chan)*2**(i//2),kern+kern_increase*i))
        self.pool = nn.MaxPool1d(2,2)  
        self.fc0 = nn.Linear((4*chan)*2**(i//2), 18)
        
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        
        x = torch.cat([x0,x1,x2,x3], dim=1)
        
        for i in range(len(self.layers)-1):
            x = self.pool(self.layers[i](x))
        x = self.layers[-1](x)
        x = torch.mean(x, dim=-1)
        x = self.fc0(x)
        return x 

class RNNetImuSE(nn.Module):
    def __init__(self, chan=64):
        super().__init__()

        self.block0_0 = ResidualBlockSE(4, chan,3)
        self.block0_1 = ResidualBlockSE(chan, chan,3)
        self.block0_2 = ResidualBlockSE(chan, chan, 3)

        self.block1_0 = ResidualBlockSE(5, chan,3)
        self.block1_1 = ResidualBlockSE(chan, chan,3)
        self.block1_2 = ResidualBlockSE(chan, chan, 3)

        self.block2_0 = ResidualBlockSE(3, chan,3)
        self.block2_1 = ResidualBlockSE(chan, chan,3)
        self.block2_2 = ResidualBlockSE(chan, chan, 3)

        self.block3_0 = ResidualBlockSE(4, chan,3)
        self.block3_1 = ResidualBlockSE(chan, chan,3)
        self.block3_2 = ResidualBlockSE(chan, chan, 3)
        
        self.rnn = nn.GRU(4*chan, 128,
            num_layers = 2, 
            batch_first =True,
            bidirectional=True)

        self.fc0 = nn.Linear(256, 256)
        self.drop = nn.Dropout(0.2)
        self.fc1 = nn.Linear(256, 18)
                    
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        
        x = torch.cat([x0,x1,x2,x3], dim=1)
        
        x = x.permute(0, 2, 1)
        x,_ = self.rnn(x)
        x = self.drop(F.relu(self.fc0(x)))
        x = self.fc1(x)
        x = x[:,-30:,:].mean(axis=1)
        return x 
class NetImuSE(nn.Module):
    def __init__(self,n_blocks=2, chan=64, kern=5, kern_increase=4, classes=18):
        super().__init__()
        
        self.block0_0 = ResidualBlockSE(4, chan,3)
        self.block0_1 = ResidualBlockSE(chan, chan,3)
        self.block0_2 = ResidualBlockSE(chan, chan, 3)

        self.block1_0 = ResidualBlockSE(5, chan,3)
        self.block1_1 = ResidualBlockSE(chan, chan,3)
        self.block1_2 = ResidualBlockSE(chan, chan, 3)

        self.block2_0 = ResidualBlockSE(3, chan,3)
        self.block2_1 = ResidualBlockSE(chan, chan,3)
        self.block2_2 = ResidualBlockSE(chan, chan, 3)

        self.block3_0 = ResidualBlockSE(4, chan,3)
        self.block3_1 = ResidualBlockSE(chan, chan,3)
        self.block3_2 = ResidualBlockSE(chan, chan, 3)
        
        self.layers = nn.ModuleList()
        for i in range(n_blocks+1):
            if i == 0:
                self.layers.append(ResidualBlockSE(4*chan, 4*chan,kern))
            else:
                self.layers.append(ResidualBlockSE((4*chan)*2**((i-1)//2), (4*chan)*2**(i//2),kern+kern_increase*i))
        self.pool = nn.MaxPool1d(2,2)  
        self.fc0 = nn.Linear((4*chan)*2**(i//2), classes)
        
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        
        x = torch.cat([x0,x1,x2,x3], dim=1)
        
        for i in range(len(self.layers)-1):
            x = self.pool(self.layers[i](x))
        x = self.layers[-1](x)
        x = torch.mean(x, dim=-1)
        x = self.fc0(x)
        return x 
# https://www.kaggle.com/code/takoihiraokazu/cv-ensemble-sub-0607-1
class RnnModelImu(nn.Module):
    def __init__(
        self, dropout=0.2,
        input_numerical_size=16,
        numeraical_linear_size = 128,
        model_size = 128,
        linear_out = 128,
        out_size=18):
        super(RnnModelImu, self).__init__()
        self.numerical_linear  = nn.Sequential(
                nn.Linear(input_numerical_size, numeraical_linear_size),
                nn.LayerNorm(numeraical_linear_size)
            )
        
        self.rnn = nn.GRU(numeraical_linear_size, model_size,
                            num_layers = 2, 
                            bidirectional=True,
                            batch_first=True)
        self.linear_out  = nn.Sequential(
                nn.Linear(model_size*2, 
                          linear_out),
                nn.LayerNorm(linear_out),
                nn.Dropout(dropout),
                nn.Linear(linear_out, 
                          out_size))
        self._reinitialize()
        
    def _reinitialize(self):
        """
        Tensorflow/Keras-like initialization
        """
        for name, p in self.named_parameters():
            if 'rnn' in name:
                if 'weight_ih' in name:
                    nn.init.xavier_uniform_(p.data)
                elif 'weight_hh' in name:
                    nn.init.orthogonal_(p.data)
                elif 'bias_ih' in name:
                    p.data.fill_(0)
                    # Set forget-gate bias to 1
                    n = p.size(0)
                    p.data[(n // 4):(n // 2)].fill_(1)
                elif 'bias_hh' in name:
                    p.data.fill_(0)
    
    def forward(self, numerical_array):
        numerical_array = numerical_array.permute(0, 2, 1)
        numerical_embedding = self.numerical_linear(numerical_array)
        output,_ = self.rnn(numerical_embedding)
        output = self.linear_out(output)
        return output[:,-30:,:].mean(axis=1)


class ResidualSECNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, pool_size=1, dropout=0.0, weight_decay=1e-4):
        super().__init__()
        
        # First conv block
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        
        # Second conv block
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        # SE block
        self.se = SEBlock(out_channels)
        
        # Shortcut connection
        self.shortcut = nn.Sequential()
        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        self.pool_size = pool_size
        self.pool = nn.MaxPool1d(pool_size)
        self.dropout = nn.Dropout(dropout)
    
        self.act = nn.ReLU()
    
        self.gru = nn.GRU(out_channels, out_channels//2, batch_first=True, bidirectional=True)
        
    def forward(self, x):
        shortcut = self.shortcut(x)
        
        # First conv
        out = self.act(self.bn1(self.conv1(x)))
        # Second conv
        out = self.bn2(self.conv2(out))
        
        # SE block
        out = self.se(out)
        
        # Add shortcut
        
        out += shortcut
        
        out = self.act(out)
    
        out = out + self.gru(out.transpose(1, 2))[0].transpose(1, 2)
        
        # Pool and dropout
        if self.pool_size>1:
            out = self.pool(out)
        out = self.dropout(out)
        
        return out  


class NetAllResGru(nn.Module):
    def __init__(self,n_blocks=2, chan=64, kern=5, kern_increase=4):
        super().__init__()
        
        self.block0_0 = ResidualSECNNBlock(4, chan,3)
        self.block0_1 = ResidualSECNNBlock(chan, chan,3)
        self.block0_2 = ResidualSECNNBlock(chan, chan, 3)

        self.block1_0 = ResidualSECNNBlock(5, chan,3)
        self.block1_1 = ResidualSECNNBlock(chan, chan,3)
        self.block1_2 = ResidualSECNNBlock(chan, chan, 3)

        self.block2_0 = ResidualSECNNBlock(3, chan,3)
        self.block2_1 = ResidualSECNNBlock(chan, chan,3)
        self.block2_2 = ResidualSECNNBlock(chan, chan, 3)

        self.block3_0 = ResidualSECNNBlock(4, chan,3)
        self.block3_1 = ResidualSECNNBlock(chan, chan,3)
        self.block3_2 = ResidualSECNNBlock(chan, chan, 3)

        self.block4_0 = ResidualSECNNBlock(50, 2*chan,3)
        self.block4_1 = ResidualSECNNBlock(2*chan, 2*chan,5)
        
        self.layers = nn.ModuleList()
        for i in range(n_blocks+1):
            if i == 0:
                self.layers.append(ResidualSECNNBlock(6*chan, 4*chan,kern))
            else:
                self.layers.append(ResidualSECNNBlock((4*chan)*2**((i-1)//2), (4*chan)*2**(i//2),kern+kern_increase*i))
        self.pool = nn.MaxPool1d(2,2)  
        self.fc0 = nn.Linear((4*chan)*2**(i//2), 18)
        
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:16,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        x4 = self.block4_1(self.block4_0(x[:,16:,:]))
        
        x = torch.cat([x0,x1,x2, x3, x4], dim=1)
        
        for i in range(len(self.layers)-1):
            x = self.pool(self.layers[i](x))
        x = self.layers[-1](x)
        x = torch.mean(x, dim=-1)
        x = self.fc0(x)
        return x 


class NetImuResGru(nn.Module):
    def __init__(self,n_blocks=2, chan=64, kern=5, kern_increase=4):
        super().__init__()
        
        self.block0_0 = ResidualSECNNBlock(4, chan,3)
        self.block0_1 = ResidualSECNNBlock(chan, chan,3)
        self.block0_2 = ResidualSECNNBlock(chan, chan, 3)

        self.block1_0 = ResidualSECNNBlock(5, chan,3)
        self.block1_1 = ResidualSECNNBlock(chan, chan,3)
        self.block1_2 = ResidualSECNNBlock(chan, chan, 3)

        self.block2_0 = ResidualSECNNBlock(3, chan,3)
        self.block2_1 = ResidualSECNNBlock(chan, chan,3)
        self.block2_2 = ResidualSECNNBlock(chan, chan, 3)

        self.block3_0 = ResidualSECNNBlock(4, chan,3)
        self.block3_1 = ResidualSECNNBlock(chan, chan,3)
        self.block3_2 = ResidualSECNNBlock(chan, chan, 3)
        
        self.layers = nn.ModuleList()
        for i in range(n_blocks+1):
            if i == 0:
                self.layers.append(ResidualSECNNBlock(4*chan, 4*chan,kern))
            else:
                self.layers.append(ResidualSECNNBlock((4*chan)*2**((i-1)//2), (4*chan)*2**(i//2),kern+kern_increase*i))
        self.pool = nn.MaxPool1d(2,2)  
        self.fc0 = nn.Linear((4*chan)*2**(i//2), 18)
        
    def forward(self, x):
        in1 = torch.cat([x[:,:3,:], x[:,10:11,:]], dim=1)
        in2 = torch.cat([x[:,3:7,:], x[:,11:12,:]], dim=1)
        in3 = x[:,7:10,:]
        in4 = x[:,12:,:]
        x0 = self.block0_2(self.block0_1(self.block0_0(in1)))
        x1 = self.block1_2(self.block1_1(self.block1_0(in2)))
        x2 = self.block2_2(self.block2_1(self.block2_0(in3)))
        x3 = self.block3_2(self.block3_1(self.block3_0(in4)))
        
        x = torch.cat([x0,x1,x2,x3], dim=1)
        
        for i in range(len(self.layers)-1):
            x = self.pool(self.layers[i](x))
        x = self.layers[-1](x)
        x = torch.mean(x, dim=-1)
        x = self.fc0(x)
        return x 

In [40]:
def standard_scale_3d_transform(data, mean, std):
    return (data - mean) / std

def prep_data_tof3d(tof_arr,mean, std):
    tr_tof = standard_scale_3d_transform(tof_arr, mean, std)
    tr_tof = np.nan_to_num(tr_tof, 0)
    return tr_tof

class ResidualBlock3d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size):
        super().__init__()
        self.conv1 = nn.Conv3d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, padding="same")
        self.bn1 = nn.BatchNorm3d(num_features=out_channels)
        self.conv2 = nn.Conv3d(in_channels=out_channels, out_channels=out_channels, kernel_size=kernel_size, padding="same")
        self.bn2 = nn.BatchNorm3d(num_features=out_channels)
        
        if in_channels != out_channels:
            self.res_layer = torch.nn.Conv3d(in_channels, out_channels, kernel_size=1, padding="same")
        else:
            self.res_layer = None
            
    def forward(self, x):
        res = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        if self.res_layer is None:
            x = torch.add(x, res)
        else: 
            x = torch.add(x, self.res_layer(res)) # should the res layer have a relu? or batch norm?
            # TODO experiment with the residual block
        return x 


class RNNet3d(nn.Module):
    def __init__(self, input_chan, chan=128):
        super().__init__()
        self.block0 = ResidualBlock3d(input_chan, chan, 3)
        self.block1 = ResidualBlock3d(chan, chan, 3)
        self.block2 = ResidualBlock3d(chan, chan, 3)

        self.rnn = nn.GRU(chan, 128,
            num_layers = 2, 
            batch_first =True,
            bidirectional=True)
        
        self.pool = nn.MaxPool3d((1,2,2))
        self.fc0 = nn.Linear(256, 256)
        self.drop = nn.Dropout(0.2)
        self.fc1 = nn.Linear(256, 18)
            
        
    def forward(self, x):
        x = self.block0(x)
        x = self.pool(x)
        x = self.block1(x)
        x = self.pool(x)
        x = self.block2(x)
        x = self.pool(x)
        
        x = x.squeeze(dim=[3,4])
        x = x.permute(0, 2, 1)
        x,_ = self.rnn(x)
        x = self.drop(F.relu(self.fc0(x)))
        x = self.fc1(x)
        x = x[:,-30:,:].mean(axis=1)
        return x

In [41]:
model_folder_path = "/kaggle/input/cmi-detect-behavior/twelfth_nn_submission/twelfth_nn_submission/"
imu_files_cnngru = glob.glob(os.path.join(model_folder_path, "imuonly_cnngru128_itr*"))
imu_files_cnn = glob.glob(os.path.join(model_folder_path, "imuonly_cnn64chan_itr*"))
imu_files_densegru = glob.glob(os.path.join(model_folder_path, "imuonly_densegru_itr*"))
imu_files_cnngruSE = glob.glob(os.path.join(model_folder_path, "imuonly_cnngru128SE_itr*"))
imu_files_cnnSE = glob.glob(os.path.join(model_folder_path, "imuonly_cnn64chanSE_itr*"))

all_files_cnngru = glob.glob(os.path.join(model_folder_path, "alldata_cnngru512_itr*"))
all_files_cnn = glob.glob(os.path.join(model_folder_path, "alldata_cnn64chan_itr*"))
all_files_densegru = glob.glob(os.path.join(model_folder_path, "alldata_densegru_itr*"))
all_files_cnngruSE = glob.glob(os.path.join(model_folder_path, "alldata_cnngru512SE_itr*"))
all_files_cnnSE = glob.glob(os.path.join(model_folder_path, "alldata_cnn64chanSE_itr*"))
#all_files_tofthm = glob.glob(os.path.join(model_folder_path, "alldata_cnngru128tofthm*"))

model_folder_path = "/kaggle/input/cmi-detect-behavior/thirteenth_nn_submission/thirteenth_nn_submission/"
all_files_cnngru_diff = glob.glob(os.path.join(model_folder_path, "alldata_cnngru512_diff*"))
all_files_cnn_diff = glob.glob(os.path.join(model_folder_path, "alldata_cnn64chan_diff*"))
all_files_densegru_diff = glob.glob(os.path.join(model_folder_path, "alldata_densegru_diff*"))
all_files_cnngruSE_diff = glob.glob(os.path.join(model_folder_path, "alldata_cnngru512SE_diff*"))
all_files_cnnSE_diff = glob.glob(os.path.join(model_folder_path, "alldata_cnn64chanSE_diff*"))

model_folder_path = "/kaggle/input/cmi-detect-behavior/fourteenth_nn_submission/fourteenth_nn_submission/"
all_files_tof3dnet = glob.glob(os.path.join(model_folder_path, "alldata_tof3dnet_itr*"))
all_files_tof3dnet_diff = glob.glob(os.path.join(model_folder_path, "alldata_tof3dnet_diff*"))

model_folder_path = "/kaggle/input/cmi-detect-behavior/seventeenth_nn_submission/seventeenth_nn_submission/"
imu_files_cnngru_resgru = glob.glob(os.path.join(model_folder_path, "imuonly_cnngru128_resgru*"))
all_files_cnngru_resgru = glob.glob(os.path.join(model_folder_path, "alldata_cnngru512_resgru*"))

model_folder_path = "/kaggle/input/cmi-detect-orientation-models/nineteenth_nn_submission"
imu_files_orientation = glob.glob(os.path.join(model_folder_path, "imuonly_cnn64chan*"))
all_files_orientation = glob.glob(os.path.join(model_folder_path, "alldata_cnngru512*"))

if True:
    models_imu_cnngru = list()
    for mf in imu_files_cnngru:
        model = RNNetImu().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_imu_cnngru.append(model)
    print(len(models_imu_cnngru))
    
    models_imu_cnn = list()
    for mf in imu_files_cnn:
        model = NetImu().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        models_imu_cnn.append(model)
    print(len(models_imu_cnn))
    
    models_imu_densegru = list()
    for mf in imu_files_densegru:
        model = RnnModelImu().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_imu_densegru.append(model)
    print(len(models_imu_densegru))
    
    models_imu_cnngruSE = list()
    for mf in imu_files_cnngruSE:
        model = RNNetImuSE().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_imu_cnngruSE.append(model)
    print(len(models_imu_cnngruSE))
    
    models_imu_cnnSE = list()
    for mf in imu_files_cnnSE:
        model = NetImuSE().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        models_imu_cnnSE.append(model)
    print(len(models_imu_cnnSE))
    
    models_all_cnngru = list()
    for mf in all_files_cnngru:
        model = RNNetAllBig().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_all_cnngru.append(model)
    print(len(models_all_cnngru))
    
    models_all_cnn = list()
    for mf in all_files_cnn:
        model = NetAll().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        models_all_cnn.append(model)
    print(len(models_all_cnn))
    
    models_all_densegru = list()
    for mf in all_files_densegru:
        model = RnnModelAlldata().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_all_densegru.append(model)
    print(len(models_all_densegru))

    models_all_cnngruSE = list()
    for mf in all_files_cnngruSE:
        model = RNNetAllBigSE().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_all_cnngruSE.append(model)
    print(len(models_all_cnngruSE))
    
    models_all_cnnSE = list()
    for mf in all_files_cnnSE:
        model = NetAllSE().to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        models_all_cnnSE.append(model)
    print(len(models_all_cnnSE))
    
    # start adding diff
    models_all_cnngru_diff = list()
    for mf in all_files_cnngru_diff:
        model = RNNetAllBig(n_all_feat=100).to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_all_cnngru_diff.append(model)
    print(len(models_all_cnngru_diff))
    
    models_all_cnn_diff = list()
    for mf in all_files_cnn_diff:
        model = NetAll(n_all_feat=100).to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        models_all_cnn_diff.append(model)
    print(len(models_all_cnn_diff))
    
    models_all_densegru_diff = list()
    for mf in all_files_densegru_diff:
        model = RnnModelAlldata(input_numerical_size=116).to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_all_densegru_diff.append(model)
    print(len(models_all_densegru_diff))
    
    models_all_cnngruSE_diff = list()
    for mf in all_files_cnngruSE_diff:
        model = RNNetAllBigSE(n_all_feat=100).to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_all_cnngruSE_diff.append(model)
    print(len(models_all_cnngruSE_diff))
    
    models_all_cnnSE_diff = list()
    for mf in all_files_cnnSE_diff:
        model = NetAllSE(n_all_feat=100).to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        models_all_cnnSE_diff.append(model)
    print(len(models_all_cnnSE_diff))
    
    models_all_tof3dnet = list()
    for mf in all_files_tof3dnet:
        model = RNNet3d(5).to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_all_tof3dnet.append(model)
    print(len(models_all_tof3dnet))
    
    models_all_tof3dnet_diff = list()
    for mf in all_files_tof3dnet_diff:
        model = RNNet3d(10).to(device2)
        model_state = torch.load(mf, map_location=device2)
        model.load_state_dict(model_state)
        model.rnn.flatten_parameters()
        models_all_tof3dnet_diff.append(model)
    print(len(models_all_tof3dnet_diff))


models_imu_cnngru_resgru = list()
for mf in imu_files_cnngru_resgru:
    model = NetImuResGru().to(device)
    model_state = torch.load(mf, map_location=device)
    model.load_state_dict(model_state)
    models_imu_cnngru_resgru.append(model)
print(len(models_imu_cnngru_resgru))

models_all_cnngru_resgru = list()
for mf in all_files_cnngru_resgru:
    model = NetAllResGru().to(device)
    model_state = torch.load(mf, map_location=device)
    model.load_state_dict(model_state)
    models_all_cnngru_resgru.append(model)
print(len(models_all_cnngru_resgru))

models_imu_orientation = list()
for mf in imu_files_orientation:
    model = NetImuSE(classes=4).to(device)
    model_state = torch.load(mf, map_location=device)
    model.load_state_dict(model_state)
    models_imu_orientation.append(model)
print(len(models_imu_orientation))

models_all_orientation = list()
for mf in all_files_orientation:
    model = RNNetAllBig(classes=4).to(device)
    model_state = torch.load(mf, map_location=device)
    model.load_state_dict(model_state)
    models_all_orientation.append(model)
print(len(models_all_orientation))

40
40
40
40
40
40
40
40
40
40
40
40
40
40
40
20
20
20
20
4
4


In [42]:
with open("/kaggle/input/cmi-detect-behavior/twelfth_nn_submission/twelfth_nn_submission/imuonly_scaler.pkl", "rb") as f:
    scaler_imu = load(f)

with open("/kaggle/input/cmi-detect-behavior/twelfth_nn_submission/twelfth_nn_submission/alldata_scaler.pkl", "rb") as f:
    scaler_all = load(f)

with open("/kaggle/input/cmi-detect-behavior/thirteenth_nn_submission/thirteenth_nn_submission/alldata_diff_scaler.pkl", "rb") as f:
    scaler_all_diff = load(f)

with open("/kaggle/input/cmi-detect-orientation-models/nineteenth_nn_submission/orientation_encoder.pkl", "rb") as f:
    orientation_encoder = load(f)

tof_mean = np.load("/kaggle/input/cmi-detect-behavior/fourteenth_nn_submission/fourteenth_nn_submission/tof_mean.npy")
tof_std = np.load("/kaggle/input/cmi-detect-behavior/fourteenth_nn_submission/fourteenth_nn_submission/tof_std.npy")

tof_diff_mean = np.load("/kaggle/input/cmi-detect-behavior/fourteenth_nn_submission/fourteenth_nn_submission/tof_diff_mean.npy")
tof_diff_std = np.load("/kaggle/input/cmi-detect-behavior/fourteenth_nn_submission/fourteenth_nn_submission/tof_diff_std.npy")

In [43]:
def tof_to_array(df, sensor, t):
    l = list()
    for i in range(8):
        ll = list()
        for j in range(8):
            ll.append(df[t,f"tof_{sensor}_v{i*8+j}"])
        l.append(ll)
    return np.array(l)

def tofdiff_to_array(df, sensor, t):
    l = list()
    for i in range(8):
        ll = list()
        for j in range(8):
            ll.append(df[t,f"tof_{sensor}_v{i*8+j}_diff"])
        l.append(ll)
    return np.array(l)

def prep_tof(df, mean, std):
    tof_arr = np.zeros((1, 5, 128, 8, 8))
    start = max(df.shape[0]-128, 0)
    offset = max(128-df.shape[0], 0)
    for ee, i in enumerate(range(start, df.shape[0])):
        for j in range(1,6):
            tof_image = tof_to_array(df, j, i)
            tof_arr[0,j-1,offset+ee,:,:] = tof_image
    return prep_data_tof3d(tof_arr, mean, std)

def prep_tof_diff(df, mean, std):
    df = df.with_columns(pl.all().replace(-1, None))
    for i in [f"tof_{x}_v{y}" for x in range(1,6) for y in range(64)]:
        df = df.with_columns(pl.col(i).diff().alias(i+"_diff"))
    tof_diff_arr = np.zeros((1, 10, 128, 8, 8))
    
    start = max(df.shape[0]-128, 0)
    offset = max(128-df.shape[0], 0)
    for ee, i in enumerate(range(start, df.shape[0])):
        for j in range(1,6):
            tof_image = tof_to_array(df, j, i)
            tof_diff_arr[0,j-1,offset+ee,:,:] = tof_image
        for j in range(1,6):
            tof_image = tofdiff_to_array(df, j, i)
            tof_diff_arr[0,5+j-1,offset+ee,:,:] = tof_image
    return prep_data_tof3d(tof_diff_arr, mean, std)

In [44]:
def make_checker():
    count = dict()
    def checker(subj_id, acc_y):
        nonlocal count
        if subj_id in count:
            count[subj_id].append(acc_y)
        else:
            count[subj_id] = [acc_y]
        a = pl.Series(count[subj_id])
        m = a.mean()
        l = a.shape[0]
        if m > 0.5 and l > 70:
            return True
        else:
            return False
    return checker

In [45]:
c = make_checker()

In [46]:
def predict_inner(inputs, models):
    for e, model in enumerate(models):
        with torch.no_grad():
            model.eval()
            outputs = model(inputs)
            if e == 0:
                predicted = outputs #.to("cpu").numpy()
            else:
                predicted += outputs #.to("cpu").numpy()
    return predicted

### Predicting

# zhou

In [47]:
!pip install --no-deps /kaggle/input/scikit-learn-1-6-1/scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

#!/usr/bin/env python
# coding: utf-8

# # Importable version of https://www.kaggle.com/code/metric/cmi-2025

# In[ ]:


"""
Hierarchical macro F1 metric for the CMI 2025 Challenge.

This script defines a single entry point `score(solution, submission, row_id_column_name)`
that the Kaggle metrics orchestrator will call.
It performs validation on submission IDs and computes a combined binary & multiclass F1 score.
"""

import pandas as pd
from sklearn.metrics import f1_score


class ParticipantVisibleError(Exception):
    """Errors raised here will be shown directly to the competitor."""
    pass


class CompetitionMetric:
    """Hierarchical macro F1 for the CMI 2025 challenge."""
    def __init__(self):
        self.target_gestures = [
            'Above ear - pull hair',
            'Cheek - pinch skin',
            'Eyebrow - pull hair',
            'Eyelash - pull hair',
            'Forehead - pull hairline',
            'Forehead - scratch',
            'Neck - pinch skin',
            'Neck - scratch',
        ]
        self.non_target_gestures = [
            'Write name on leg',
            'Wave hello',
            'Glasses on/off',
            'Text on phone',
            'Write name in air',
            'Feel around in tray and pull out an object',
            'Scratch knee/leg skin',
            'Pull air toward your face',
            'Drink from bottle/cup',
            'Pinch knee/leg skin'
        ]
        self.all_classes = self.target_gestures + self.non_target_gestures

    def calculate_hierarchical_f1(
        self,
        sol: pd.DataFrame,
        sub: pd.DataFrame
    ) -> float:

        # Validate gestures
        invalid_types = {i for i in sub['gesture'].unique() if i not in self.all_classes}
        if invalid_types:
            raise ParticipantVisibleError(
                f"Invalid gesture values in submission: {invalid_types}"
            )

        # Compute binary F1 (Target vs Non-Target)
        y_true_bin = sol['gesture'].isin(self.target_gestures).values
        y_pred_bin = sub['gesture'].isin(self.target_gestures).values
        f1_binary = f1_score(
            y_true_bin,
            y_pred_bin,
            pos_label=True,
            zero_division=0,
            average='binary'
        )

        # Build multi-class labels for gestures
        y_true_mc = sol['gesture'].apply(lambda x: x if x in self.target_gestures else 'non_target')
        y_pred_mc = sub['gesture'].apply(lambda x: x if x in self.target_gestures else 'non_target')

        # Compute macro F1 over all gesture classes
        f1_macro = f1_score(
            y_true_mc,
            y_pred_mc,
            average='macro',
            zero_division=0
        )

        return 0.5 * f1_binary + 0.5 * f1_macro


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str
) -> float:
    """
    Compute hierarchical macro F1 for the CMI 2025 challenge.

    Expected input:
      - solution and submission as pandas.DataFrame
      - Column 'sequence_id': unique identifier for each sequence
      - 'gesture': one of the eight target gestures or "Non-Target"

    This metric averages:
    1. Binary F1 on SequenceType (Target vs Non-Target)
    2. Macro F1 on gesture (mapping non-targets to "Non-Target")

    Raises ParticipantVisibleError for invalid submissions,
    including invalid SequenceType or gesture values.


    Examples
    --------
    >>> import pandas as pd
    >>> row_id_column_name = "id"
    >>> solution = pd.DataFrame({'id': range(4), 'gesture': ['Eyebrow - pull hair']*4})
    >>> submission = pd.DataFrame({'id': range(4), 'gesture': ['Forehead - pull hairline']*4})
    >>> score(solution, submission, row_id_column_name=row_id_column_name)
    0.5
    >>> submission = pd.DataFrame({'id': range(4), 'gesture': ['Text on phone']*4})
    >>> score(solution, submission, row_id_column_name=row_id_column_name)
    0.0
    >>> score(solution, solution, row_id_column_name=row_id_column_name)
    1.0
    """
    # Validate required columns
    for col in (row_id_column_name, 'gesture'):
        if col not in solution.columns:
            raise ParticipantVisibleError(f"Solution file missing required column: '{col}'")
        if col not in submission.columns:
            raise ParticipantVisibleError(f"Submission file missing required column: '{col}'")

    metric = CompetitionMetric()
    return metric.calculate_hierarchical_f1(solution, submission)


import torch
import torch.distributed as dist


def zeropower_via_newtonschulz5(G, steps: int):
    """
    Newton-Schulz iteration to compute the zeroth power / orthogonalization of G. We opt to use a
    quintic iteration whose coefficients are selected to maximize the slope at zero. For the purpose
    of minimizing steps, it turns out to be empirically effective to keep increasing the slope at
    zero even beyond the point where the iteration no longer converges all the way to one everywhere
    on the interval. This iteration therefore does not produce UV^T but rather something like US'V^T
    where S' is diagonal with S_{ii}' ~ Uniform(0.5, 1.5), which turns out not to hurt model
    performance at all relative to UV^T, where USV^T = G is the SVD.
    """
    assert G.ndim >= 2 # batched Muon implementation by @scottjmaddox, and put into practice in the record by @YouJiacheng
    a, b, c = (3.4445, -4.7750,  2.0315)
    X = G.bfloat16()
    if G.size(-2) > G.size(-1):
        X = X.mT

    # Ensure spectral norm is at most 1
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)
    # Perform the NS iterations
    for _ in range(steps):
        A = X @ X.mT
        B = b * A + c * A @ A # quintic computation strategy adapted from suggestion by @jxbz, @leloykun, and @YouJiacheng
        X = a * X + B @ X
    
    if G.size(-2) > G.size(-1):
        X = X.mT
    return X


def muon_update(grad, momentum, beta=0.95, ns_steps=5, nesterov=True):
    momentum.lerp_(grad, 1 - beta)
    update = grad.lerp_(momentum, beta) if nesterov else momentum
    if update.ndim == 4: # for the case of conv filters
        update = update.view(len(update), -1)
    update = zeropower_via_newtonschulz5(update, steps=ns_steps)
    update *= max(1, grad.size(-2) / grad.size(-1))**0.5
    return update


class Muon(torch.optim.Optimizer):
    """
    Muon - MomentUm Orthogonalized by Newton-schulz

    https://kellerjordan.github.io/posts/muon/

    Muon internally runs standard SGD-momentum, and then performs an orthogonalization post-
    processing step, in which each 2D parameter's update is replaced with the nearest orthogonal
    matrix. For efficient orthogonalization we use a Newton-Schulz iteration, which has the
    advantage that it can be stably run in bfloat16 on the GPU.

    Muon should only be used for hidden weight layers. The input embedding, final output layer,
    and any internal gains or biases should be optimized using a standard method such as AdamW.
    Hidden convolutional weights can be trained using Muon by viewing them as 2D and then
    collapsing their last 3 dimensions.

    Arguments:
        lr: The learning rate, in units of spectral norm per update.
        weight_decay: The AdamW-style weight decay.
        momentum: The momentum. A value of 0.95 here is usually fine.
    """
    def __init__(self, params, lr=0.02, weight_decay=0, momentum=0.95):
        defaults = dict(lr=lr, weight_decay=weight_decay, momentum=momentum)
        assert isinstance(params, list) and len(params) >= 1 and isinstance(params[0], torch.nn.Parameter)
        params = sorted(params, key=lambda x: x.size(), reverse=True)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):

        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            params = group["params"]
            params_pad = params + [torch.empty_like(params[-1])] * (dist.get_world_size() - len(params) % dist.get_world_size())
            for base_i in range(len(params))[::dist.get_world_size()]:
                if base_i + dist.get_rank() < len(params):
                    p = params[base_i + dist.get_rank()]
                    state = self.state[p]
                    if len(state) == 0:
                        state["momentum_buffer"] = torch.zeros_like(p)
                    update = muon_update(p.grad, state["momentum_buffer"], beta=group["momentum"])
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                    p.add_(update.reshape(p.shape), alpha=-group["lr"])
                dist.all_gather(params_pad[base_i:base_i + dist.get_world_size()], params_pad[base_i + dist.get_rank()])

        return loss


class SingleDeviceMuon(torch.optim.Optimizer):
    """
    Muon variant for usage in non-distributed settings.
    """
    def __init__(self, params, lr=0.02, weight_decay=0, momentum=0.95):
        defaults = dict(lr=lr, weight_decay=weight_decay, momentum=momentum)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):

        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            for p in group["params"]:
                state = self.state[p]
                if len(state) == 0:
                    state["momentum_buffer"] = torch.zeros_like(p)
                update = muon_update(p.grad, state["momentum_buffer"], beta=group["momentum"])
                p.mul_(1 - group["lr"] * group["weight_decay"])
                p.add_(update.reshape(p.shape), alpha=-group["lr"])

        return loss


def adam_update(grad, buf1, buf2, step, betas, eps):
    buf1.lerp_(grad, 1 - betas[0])
    buf2.lerp_(grad.square(), 1 - betas[1])
    buf1c = buf1 / (1 - betas[0]**step)
    buf2c = buf2 / (1 - betas[1]**step)
    return buf1c / (buf2c.sqrt() + eps)


class MuonWithAuxAdam(torch.optim.Optimizer):
    """
    Distributed Muon variant that can be used for all parameters in the network, since it runs an
    internal AdamW for the parameters that are not compatible with Muon. The user must manually
    specify which parameters shall be optimized with Muon and which with Adam by passing in a
    list of param_groups with the `use_muon` flag set.

    The point of this class is to allow the user to have a single optimizer in their code, rather
    than having both a Muon and an Adam which each need to be stepped.

    You can see an example usage below:

    https://github.com/KellerJordan/modded-nanogpt/blob/master/records/052525_MuonWithAuxAdamExample/b01550f9-03d8-4a9c-86fe-4ab434f1c5e0.txt#L470
    ```
    hidden_matrix_params = [p for n, p in model.blocks.named_parameters() if p.ndim >= 2 and "embed" not in n]
    embed_params = [p for n, p in model.named_parameters() if "embed" in n]
    scalar_params = [p for p in model.parameters() if p.ndim < 2]
    head_params = [model.lm_head.weight]

    from muon import MuonWithAuxAdam
    adam_groups = [dict(params=head_params, lr=0.22), dict(params=embed_params, lr=0.6), dict(params=scalar_params, lr=0.04)]
    adam_groups = [dict(**g, betas=(0.8, 0.95), eps=1e-10, use_muon=False) for g in adam_groups]
    muon_group = dict(params=hidden_matrix_params, lr=0.05, momentum=0.95, use_muon=True)
    param_groups = [*adam_groups, muon_group]
    optimizer = MuonWithAuxAdam(param_groups)
    ```
    """
    def __init__(self, param_groups):
        for group in param_groups:
            assert "use_muon" in group
            if group["use_muon"]:
                group["params"] = sorted(group["params"], key=lambda x: x.size(), reverse=True)
                # defaults
                group["lr"] = group.get("lr", 0.02)
                group["momentum"] = group.get("momentum", 0.95)
                group["weight_decay"] = group.get("weight_decay", 0)
                assert set(group.keys()) == set(["params", "lr", "momentum", "weight_decay", "use_muon"])
            else:
                # defaults
                group["lr"] = group.get("lr", 3e-4)
                group["betas"] = group.get("betas", (0.9, 0.95))
                group["eps"] = group.get("eps", 1e-10)
                group["weight_decay"] = group.get("weight_decay", 0)
                assert set(group.keys()) == set(["params", "lr", "betas", "eps", "weight_decay", "use_muon"])
        super().__init__(param_groups, dict())

    @torch.no_grad()
    def step(self, closure=None):

        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            if group["use_muon"]:
                params = group["params"]
                params_pad = params + [torch.empty_like(params[-1])] * (dist.get_world_size() - len(params) % dist.get_world_size())
                for base_i in range(len(params))[::dist.get_world_size()]:
                    if base_i + dist.get_rank() < len(params):
                        p = params[base_i + dist.get_rank()]
                        state = self.state[p]
                        if len(state) == 0:
                            state["momentum_buffer"] = torch.zeros_like(p)
                        update = muon_update(p.grad, state["momentum_buffer"], beta=group["momentum"])
                        p.mul_(1 - group["lr"] * group["weight_decay"])
                        p.add_(update.reshape(p.shape), alpha=-group["lr"])
                    dist.all_gather(params_pad[base_i:base_i + dist.get_world_size()], params_pad[base_i + dist.get_rank()])
            else:
                for p in group["params"]:
                    state = self.state[p]
                    if len(state) == 0:
                        state["exp_avg"] = torch.zeros_like(p)
                        state["exp_avg_sq"] = torch.zeros_like(p)
                        state["step"] = 0
                    state["step"] += 1
                    update = adam_update(p.grad, state["exp_avg"], state["exp_avg_sq"],
                                         state["step"], group["betas"], group["eps"])
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                    p.add_(update, alpha=-group["lr"])

        return loss


class SingleDeviceMuonWithAuxAdam(torch.optim.Optimizer):
    """
    Non-distributed variant of MuonWithAuxAdam.
    """
    def __init__(self, param_groups):
        for group in param_groups:
            assert "use_muon" in group
            if group["use_muon"]:
                # defaults
                group["lr"] = group.get("lr", 0.02)
                group["momentum"] = group.get("momentum", 0.95)
                group["weight_decay"] = group.get("weight_decay", 0)
                assert set(group.keys()) == set(["params", "lr", "momentum", "weight_decay", "use_muon"])
            else:
                # defaults
                group["lr"] = group.get("lr", 3e-4)
                group["betas"] = group.get("betas", (0.9, 0.95))
                group["eps"] = group.get("eps", 1e-10)
                group["weight_decay"] = group.get("weight_decay", 0)
                assert set(group.keys()) == set(["params", "lr", "betas", "eps", "weight_decay", "use_muon"])
        super().__init__(param_groups, dict())

    @torch.no_grad()
    def step(self, closure=None):

        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            if group["use_muon"]:
                for p in group["params"]:
                    if p.grad is None:
                        continue
                        
                    state = self.state[p]
                    if len(state) == 0:
                        state["momentum_buffer"] = torch.zeros_like(p)
                    update = muon_update(p.grad, state["momentum_buffer"], beta=group["momentum"])
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                    p.add_(update.reshape(p.shape), alpha=-group["lr"])
            else:
                for p in group["params"]:
                    if p.grad is None:
                        continue
                    state = self.state[p]
                    if len(state) == 0:
                        state["exp_avg"] = torch.zeros_like(p)
                        state["exp_avg_sq"] = torch.zeros_like(p)
                        state["step"] = 0
                    state["step"] += 1
                    update = adam_update(p.grad, state["exp_avg"], state["exp_avg_sq"],
                                         state["step"], group["betas"], group["eps"])
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                    p.add_(update, alpha=-group["lr"])

        return loss

from torch.optim.lr_scheduler import _LRScheduler
class ConstantCosineLR(_LRScheduler):
    """
    Constant learning rate followed by CosineAnnealing.
    """
    def __init__(
        self, 
        optimizer,
        total_steps, 
        pct_cosine, 
        last_epoch=-1,
        ):
        self.total_steps = total_steps
        self.milestone = int(total_steps * (pct_cosine))
        self.cosine_steps = max(total_steps - self.milestone, 1)
        self.min_lr = 0
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        step = self.last_epoch + 1
        if step <= self.milestone:
            factor = 1.
        else:
            s = step - self.milestone
            factor = 0.5 * (1 + math.cos(math.pi * s / self.cosine_steps))
        return [lr * factor for lr in self.base_lrs]

Processing /kaggle/input/scikit-learn-1-6-1/scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.5.2
    Uninstalling scikit-learn-1.5.2:
      Successfully uninstalled scikit-learn-1.5.2


In [48]:
VALIDATION = False

In [49]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 




if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v1:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23"):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path("./")                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path("./")                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path("./")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        self.VALIDATION = False
        self.SEED = 42

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, 
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                out += shortcut
                out = self.act(out)
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context



        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.feature_engineering = feature_engineering
                if feature_engineering:
                    self.imu_fe = ImuFeatureExtractor(**kwargs)
                    imu_dim = 32 + 1 + 14 + 7 + 6 + 6
                else:
                    self.imu_fe = nn.Identity()
                    imu_dim = imu_dim_raw   
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7 + 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (9 + 4 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
        
                self.emb_all = 256 + 256 + 32 + 64
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 256
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.LSTM(self.emb_all, self.dim_lstm, num_layers=1, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 256, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(256)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(256, 5)
                self.classifier = nn.Linear(256, n_classes)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
        
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan//2].transpose(1, 2)  # (batch, imu_dim, seq_len)
                imu2 = x[:, :, self.fir_nchan//2:self.fir_nchan].transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale3[:,:thm.shape[1],:] + self.bias3[:,:thm.shape[1],:]) * mask_all.transpose(1, 2)
                tof = (tof * self.scale4[:,:tof.shape[1],:] + self.bias4[:,:tof.shape[1],:]) * mask_all.transpose(1, 2)
                
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
        
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
        
                x1 = torch.cat((x1, x12, thm, tof), 1)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
        
            def pad_sequences_torch(self, seq, maxlen, padding='post', truncating='post', value=0.0):
        
                if seq.shape[0] >= maxlen:
                    if truncating == 'post':
                        seq = seq[:maxlen]
                    else:  # 'pre'
                        seq = seq[-maxlen:]
                else:
                    pad_len = maxlen - seq.shape[0]
                    if padding == 'post':
                        seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                    else:  # 'pre'
                        seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
                return seq  
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]
        
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {'Seated Lean Non Dom - FACE DOWN': 1,
                         'Lie on Side - Non Dominant': 2,
                         'Seated Straight': 3,
                         'Lie on Back': 4}
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]
            idlistall.append(id_list_valid)
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
    
                    X_imuonly = X[:].clone()
                    X_imuonly[:, :, 14:] = 0.0
                    
                    X, X_imuonly, y = X.float().to(device), X_imuonly.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    
                    optimizer.zero_grad()
                    
                    logits, logits2, logits3 = model(X)
                    # logits_, logits2_, logits3_ = model(X_imuonly)
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2, clipval, 5)  * 0.4
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.4
    
                    # loss += clipped_cross_entropy(logits_, y, clipval, 18) * 0.1
                    # loss += clipped_cross_entropy(logits2_, y2, clipval, 5)  * 0.4 * 0.1
                    # loss += clipped_cross_entropy(logits3_, y3, clipval, 5)  * 0.4 * 0.1
    
                    # loss += 1 * (criterion_rdrop(logits, logits_) + criterion_rdrop(logits2, logits2_) + criterion_rdrop(logits3, logits3_))
                    
                    loss.backward()
                    
                    optimizer.step()
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()
                
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds, val_targets = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_all, val_targets_all = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []

                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 14:] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        
                        logits = model(X)
                        logits_imuonly = model(X_imuonly)
    
                        val_preds.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())

                        val_preds_logits.append(logits.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_preds_all.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_all.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_targets.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_imuonly.append(y.argmax(dim=1).cpu().numpy())
    
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
    
                        loss = F.cross_entropy(logits, y)
                        val_loss += loss.item()
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_split = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_all)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_all)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    val_loss /= len(val_loader)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ALL : ', round(val_acc, 4),  '| SPLIT : ', round(val_acc_split, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall)
        
    def get_new_features(self, df):
        # 优化后的函数
        def remove_gravity_from_acc(acc_data, rot_data):
            if isinstance(acc_data, pd.DataFrame):
                acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
            else:
                acc_values = np.asarray(acc_data)
                
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = acc_values.shape[0]
            linear_accel = np.zeros_like(acc_values)
            gravity_world = np.array([0, 0, 9.81])
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            
            valid_quats = quat_values[valid_mask]
            valid_quats_normalized = valid_quats / quat_norms[valid_mask, np.newaxis]
            try:
                rotations = R.from_quat(valid_quats_normalized)
                    
                gravity_sensor_frame = rotations.apply(gravity_world, inverse=True)
            
                linear_accel[valid_mask] = acc_values[valid_mask] - gravity_sensor_frame
            except:
                linear_accel[valid_mask] = acc_values[valid_mask]
            linear_accel[~valid_mask] = acc_values[~valid_mask]
            
            return linear_accel
        
        
        def calculate_angular_velocity_from_quat(rot_data, time_delta=1/200):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_vel = np.zeros((num_samples, 3))
            
            if num_samples < 2:
                return angular_vel
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q_t = quat_values[:-1][valid_pairs_mask]
                q_t_plus_dt = quat_values[1:][valid_pairs_mask]
                
                q_t_norms = quat_norms[:-1][valid_pairs_mask]
                q_t_plus_dt_norms = quat_norms[1:][valid_pairs_mask]
                
                q_t_norm = q_t / q_t_norms[:, np.newaxis]
                q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms[:, np.newaxis]
                
                rot_t = R.from_quat(q_t_norm)
                rot_t_plus_dt = R.from_quat(q_t_plus_dt_norm)
                delta_rot = rot_t.inv() * rot_t_plus_dt
                
                angular_vel[:-1][valid_pairs_mask] = delta_rot.as_rotvec() / time_delta
            
            angular_vel[-1, :] = angular_vel[-2, :] if num_samples > 1 else 0
                    
            return angular_vel
        
        
        def calculate_angular_distance(rot_data, cumulative=False):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_dist = np.zeros(num_samples)
            
            if num_samples < 2:
                return angular_dist
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q1 = quat_values[:-1][valid_pairs_mask]
                q2 = quat_values[1:][valid_pairs_mask]
                
                q1_norms = quat_norms[:-1][valid_pairs_mask]
                q2_norms = quat_norms[1:][valid_pairs_mask]
                
                q1_norm = q1 / q1_norms[:, np.newaxis]
                q2_norm = q2 / q2_norms[:, np.newaxis]
                
                r1 = R.from_quat(q1_norm)
                r2 = R.from_quat(q2_norm)
                relative_rotation = r1.inv() * r2
                
                angular_dist[:-1][valid_pairs_mask] = np.linalg.norm(relative_rotation.as_rotvec(), axis=1)
            
            if num_samples > 1:
                angular_dist[-1] = angular_dist[-2] if cumulative else 0
                
            if cumulative:
                angular_dist = np.cumsum(angular_dist)
                    
            return angular_dist
    
    
        
        linear_accel_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            acc_data_group = group[['acc_x', 'acc_y', 'acc_z']].values
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            linear_accel_group = remove_gravity_from_acc(acc_data_group, rot_data_group)
            linear_accel_list.append(pd.DataFrame(linear_accel_group, columns=['f1', 'f2', 'f3'], index=group.index))
        df_linear_accel = pd.concat(linear_accel_list)
        df = pd.concat([df, df_linear_accel], axis=1)
    
        angular_vel_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_vel_group = calculate_angular_velocity_from_quat(rot_data_group)
            angular_vel_list.append(pd.DataFrame(angular_vel_group, columns=['f4', 'f5', 'f6'], index=group.index))
        df_angular_vel = pd.concat(angular_vel_list)
        df = pd.concat([df, df_angular_vel], axis=1)
    
        angular_distance_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_dist_group = calculate_angular_distance(rot_data_group)
            angular_distance_list.append(pd.DataFrame(angular_dist_group, columns=['f7'], index=group.index))
        df_angular_distance = pd.concat(angular_distance_list)
        df = pd.concat([df, df_angular_distance], axis=1)
    
        feature_names = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
    
    
        thm_mean = df[['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']].mean(1)
    
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = (np.where(df[col].isna(), thm_mean, df[col]) ) / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in tqdm(range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
            if self.tof_mode == -1:
                for mode in [2, 4, 8, 16, 32]:
                    region_size = 64 // mode
                    for r in tqdm(range(mode)):
                        region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                        new_columns.update({
                            f'tof{mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                            f'tof{mode}_{i}_region_{r}_std': region_data.std(axis=1),
                            f'tof{mode}_{i}_region_{r}_min': region_data.min(axis=1),
                            f'tof{mode}_{i}_region_{r}_max': region_data.max(axis=1)
                        })
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(5) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio=0.):        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            for model in self.models:
                mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
                pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
                x = torch.FloatTensor(pad).to(self.device)
                
                model.eval()
                p = model(x)
                if outputs is None: outputs = p
                else: outputs += p
            outputs /= len(self.models)
                    
        return self.gesture_classes, outputs.cpu().numpy()

In [50]:
model_zhou_inference_v1_1 = model_zhou_v1('/kaggle/input/cmi3v35')

# code for validation score prediction
if VALIDATION:
    # use scikit-learn 1.6.1 for groupkfold split with seed
    MMM = model_zhou_inference_v1_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_1_1, pred_imuonly_1_1, targets, idlist_1_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_1_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_1_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [51]:
model_zhou_inference_v1_2 = model_zhou_v1('/kaggle/input/cmi3v38')

# code for validation score prediction
if VALIDATION:
    # use scikit-learn 1.6.1 for groupkfold split with seed
    MMM = model_zhou_inference_v1_2
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_1_2, pred_imuonly_1_2, targets, idlist_1_2 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_1_2.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_1_2.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [52]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented
        
transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v14:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 8
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4, 5, 6, 7]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                out += shortcut
                out = self.act(out)
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context



        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.feature_engineering = feature_engineering
                if feature_engineering:
                    self.imu_fe = ImuFeatureExtractor(**kwargs)
                    imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7
                else:
                    self.imu_fe = nn.Identity()
                    imu_dim = imu_dim_raw   
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7 + 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (9 + 4 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
        
                self.emb_all = 256 + 256 + 32 + 64
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 256
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.LSTM(self.emb_all, self.dim_lstm, num_layers=1, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 256, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(256)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(256, 5)
                self.classifier = nn.Linear(256, n_classes)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
        
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan//2].transpose(1, 2)  # (batch, imu_dim, seq_len)
                imu2 = x[:, :, self.fir_nchan//2:self.fir_nchan].transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale3[:,:thm.shape[1],:] + self.bias3[:,:thm.shape[1],:]) * mask_all.transpose(1, 2)
                tof = (tof * self.scale4[:,:tof.shape[1],:] + self.bias4[:,:tof.shape[1],:]) * mask_all.transpose(1, 2)
                
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
        
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
        
                x1 = torch.cat((x1, x12, thm, tof), 1)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
    
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':

                    X_ = np.concatenate([X, y2], -1)
    
                    X_ = transforms_custom(X_)
    
                    X = X_[:,:X.shape[-1]]
                    y2 = X_[:,X.shape[-1]:]
        
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {'Seated Lean Non Dom - FACE DOWN': 1,
                         'Lie on Side - Non Dominant': 2,
                         'Seated Straight': 3,
                         'Lie on Back': 4}
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
    
                    X_imuonly = X[:].clone()
                    X_imuonly[:, :, 14:] = 0.0

                    
                    X[BS//8:, :, 14:] = 0.0
                    
                    
                    X, X_imuonly, y = X.float().to(device), X_imuonly.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    
                    optimizer.zero_grad()
                    
                    LEN = 170 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.5
    
                    loss.backward()
                    
                    optimizer.step()
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds, val_targets = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_all, val_targets_all = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 14:] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        
                        logits = model(X)
                        logits_imuonly = model(X_imuonly)
    
                        val_preds.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())

                        val_preds_logits.append(logits.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_preds_all.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_all.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_targets.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_imuonly.append(y.argmax(dim=1).cpu().numpy())
    
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
    
                        loss = F.cross_entropy(logits, y)
                        val_loss += loss.item()
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_split = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_all)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_all)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    val_loss /= len(val_loader)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ALL : ', round(val_acc, 4),  '| SPLIT : ', round(val_acc_split, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc_imuonly # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall)
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall)
        
    def get_new_features(self, df):
        # 优化后的函数
        def remove_gravity_from_acc(acc_data, rot_data):
            if isinstance(acc_data, pd.DataFrame):
                acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
            else:
                acc_values = np.asarray(acc_data)
                
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = acc_values.shape[0]
            linear_accel = np.zeros_like(acc_values)
            gravity_world = np.array([0, 0, 9.81])
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            
            valid_quats = quat_values[valid_mask]
            valid_quats_normalized = valid_quats / quat_norms[valid_mask, np.newaxis]
            try:
                rotations = R.from_quat(valid_quats_normalized)
                    
                gravity_sensor_frame = rotations.apply(gravity_world, inverse=True)
            
                linear_accel[valid_mask] = acc_values[valid_mask] - gravity_sensor_frame
            except:
                linear_accel[valid_mask] = acc_values[valid_mask]
            linear_accel[~valid_mask] = acc_values[~valid_mask]
            
            return linear_accel
        
        
        def calculate_angular_velocity_from_quat(rot_data, time_delta=1/200):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_vel = np.zeros((num_samples, 3))
            
            if num_samples < 2:
                return angular_vel
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q_t = quat_values[:-1][valid_pairs_mask]
                q_t_plus_dt = quat_values[1:][valid_pairs_mask]
                
                q_t_norms = quat_norms[:-1][valid_pairs_mask]
                q_t_plus_dt_norms = quat_norms[1:][valid_pairs_mask]
                
                q_t_norm = q_t / q_t_norms[:, np.newaxis]
                q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms[:, np.newaxis]
                
                rot_t = R.from_quat(q_t_norm)
                rot_t_plus_dt = R.from_quat(q_t_plus_dt_norm)
                delta_rot = rot_t.inv() * rot_t_plus_dt
                
                angular_vel[:-1][valid_pairs_mask] = delta_rot.as_rotvec() / time_delta
            
            angular_vel[-1, :] = angular_vel[-2, :] if num_samples > 1 else 0
                    
            return angular_vel
        
        
        def calculate_angular_distance(rot_data, cumulative=False):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_dist = np.zeros(num_samples)
            
            if num_samples < 2:
                return angular_dist
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q1 = quat_values[:-1][valid_pairs_mask]
                q2 = quat_values[1:][valid_pairs_mask]
                
                q1_norms = quat_norms[:-1][valid_pairs_mask]
                q2_norms = quat_norms[1:][valid_pairs_mask]
                
                q1_norm = q1 / q1_norms[:, np.newaxis]
                q2_norm = q2 / q2_norms[:, np.newaxis]
                
                r1 = R.from_quat(q1_norm)
                r2 = R.from_quat(q2_norm)
                relative_rotation = r1.inv() * r2
                
                angular_dist[:-1][valid_pairs_mask] = np.linalg.norm(relative_rotation.as_rotvec(), axis=1)
            
            if num_samples > 1:
                angular_dist[-1] = angular_dist[-2] if cumulative else 0
                
            if cumulative:
                angular_dist = np.cumsum(angular_dist)
                    
            return angular_dist
    
    
        
        linear_accel_list = []
        angular_vel_list = []
        angular_distance_list = []
        
        for _, group in (df.groupby('sequence_id')):
            acc_data_group = group[['acc_x', 'acc_y', 'acc_z']].values
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            linear_accel_group = remove_gravity_from_acc(acc_data_group, rot_data_group)
            linear_accel_list.append(pd.DataFrame(linear_accel_group, columns=['f1', 'f2', 'f3'], index=group.index))

            ot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_vel_group = calculate_angular_velocity_from_quat(rot_data_group)
            angular_vel_list.append(pd.DataFrame(angular_vel_group, columns=['f4', 'f5', 'f6'], index=group.index))

            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_dist_group = calculate_angular_distance(rot_data_group)
            angular_distance_list.append(pd.DataFrame(angular_dist_group, columns=['f7'], index=group.index))
        
        df_linear_accel = pd.concat(linear_accel_list)    
        df_angular_vel = pd.concat(angular_vel_list)    
        df_angular_distance = pd.concat(angular_distance_list)
        df = pd.concat([df, df_linear_accel, df_angular_vel, df_angular_distance], axis=1)
    
        feature_names = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
    
    
        thm_mean = df[['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']].mean(1)
    
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = (np.where(df[col].isna(), thm_mean, df[col]) ) / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
            if self.tof_mode == -1:
                for mode in [2, 4, 8, 16, 32]:
                    region_size = 64 // mode
                    for r in (range(mode)):
                        region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                        new_columns.update({
                            f'tof{mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                            f'tof{mode}_{i}_region_{r}_std': region_data.std(axis=1),
                            f'tof{mode}_{i}_region_{r}_min': region_data.min(axis=1),
                            f'tof{mode}_{i}_region_{r}_max': region_data.max(axis=1)
                        })
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,14:] = 0.
            
            for model in self.models:
                model.eval()
                p = model(x)
                if outputs is None: outputs = p
                else: outputs += p
            outputs /= len(self.models)

        return self.gesture_classes, outputs.cpu().numpy()

In [53]:
model_zhou_inference_v14_1 = model_zhou_v14('/kaggle/input/cmi3v59/42', 42, './42')
if VALIDATION:
    MMM = model_zhou_inference_v14_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, 5, 6, 7]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_14_1, pred_imuonly_14_1, targets, idlist_14_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_14_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_14_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [54]:
model_zhou_inference_v14_2 = model_zhou_v14('/kaggle/input/cmi3v59/666', 666, './666')
if VALIDATION:
    MMM = model_zhou_inference_v14_2
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, 5, 6, 7]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_14_2, pred_imuonly_14_2, targets, idlist_14_2 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_14_2.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_14_2.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [55]:
model_zhou_inference_v14_3 = model_zhou_v14('/kaggle/input/cmi3v59/8888', 8888, './8888')
if VALIDATION:
    MMM = model_zhou_inference_v14_3
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, 5, 6, 7]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_14_3, pred_imuonly_14_3, targets, idlist_14_3 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_14_3.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_14_3.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [56]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented
        
transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v15:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 8
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4, 5, 6, 7]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                out += shortcut
                out = self.act(out)
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context

        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.feature_engineering = feature_engineering
                if feature_engineering:
                    self.imu_fe = ImuFeatureExtractor(**kwargs)
                    imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7
                else:
                    self.imu_fe = nn.Identity()
                    imu_dim = imu_dim_raw   
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7 + 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (9 + 4 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.emb_all = 256 + 256 + 32 + 64
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 512
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.GRU(self.emb_all, self.dim_lstm, num_layers=1, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 256, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(256)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(256, 5)
                self.classifier = nn.Linear(256, n_classes)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
        
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan//2].transpose(1, 2)  # (batch, imu_dim, seq_len)
                imu2 = x[:, :, self.fir_nchan//2:self.fir_nchan].transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale3[:,:thm.shape[1],:] + self.bias3[:,:thm.shape[1],:]) * mask_all.transpose(1, 2)
                tof = (tof * self.scale4[:,:tof.shape[1],:] + self.bias4[:,:tof.shape[1],:]) * mask_all.transpose(1, 2)
                
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
        
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
        
                x1 = torch.cat((x1, x12, thm, tof), 1)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
    
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':

                    X_ = np.concatenate([X, y2], -1)
    
                    X_ = transforms_custom(X_)
    
                    X = X_[:,:X.shape[-1]]
                    y2 = X_[:,X.shape[-1]:]
        
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {'Seated Lean Non Dom - FACE DOWN': 1,
                         'Lie on Side - Non Dominant': 2,
                         'Seated Straight': 3,
                         'Lie on Back': 4}
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]

                    X[:BS//8,:,14:] = 0.
    
                    X_imuonly = X[:].clone()
                    X_imuonly[:, :, 14:] = 0.0
                    
                    X, X_imuonly, y = X.float().to(device), X_imuonly.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    
                    optimizer.zero_grad()
                    
                    LEN = 170 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.8
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.8
    
                    loss.backward()
                    
                    optimizer.step()
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds, val_targets = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_all, val_targets_all = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 14:] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        
                        logits = model(X)
                        logits_imuonly = model(X_imuonly)
    
                        val_preds.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())

                        val_preds_logits.append(logits.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_preds_all.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_all.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_targets.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_imuonly.append(y.argmax(dim=1).cpu().numpy())
    
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
    
                        loss = F.cross_entropy(logits, y)
                        val_loss += loss.item()
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_split = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_all)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_all)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    val_loss /= len(val_loader)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ALL : ', round(val_acc, 4),  '| SPLIT : ', round(val_acc_split, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall)
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall)
        
    def get_new_features(self, df):
        # 优化后的函数
        def remove_gravity_from_acc(acc_data, rot_data):
            if isinstance(acc_data, pd.DataFrame):
                acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
            else:
                acc_values = np.asarray(acc_data)
                
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = acc_values.shape[0]
            linear_accel = np.zeros_like(acc_values)
            gravity_world = np.array([0, 0, 9.81])
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            
            valid_quats = quat_values[valid_mask]
            valid_quats_normalized = valid_quats / quat_norms[valid_mask, np.newaxis]
            try:
                rotations = R.from_quat(valid_quats_normalized)
                    
                gravity_sensor_frame = rotations.apply(gravity_world, inverse=True)
            
                linear_accel[valid_mask] = acc_values[valid_mask] - gravity_sensor_frame
            except:
                linear_accel[valid_mask] = acc_values[valid_mask]
            linear_accel[~valid_mask] = acc_values[~valid_mask]
            
            return linear_accel
        
        
        def calculate_angular_velocity_from_quat(rot_data, time_delta=1/200):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_vel = np.zeros((num_samples, 3))
            
            if num_samples < 2:
                return angular_vel
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q_t = quat_values[:-1][valid_pairs_mask]
                q_t_plus_dt = quat_values[1:][valid_pairs_mask]
                
                q_t_norms = quat_norms[:-1][valid_pairs_mask]
                q_t_plus_dt_norms = quat_norms[1:][valid_pairs_mask]
                
                q_t_norm = q_t / q_t_norms[:, np.newaxis]
                q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms[:, np.newaxis]
                
                rot_t = R.from_quat(q_t_norm)
                rot_t_plus_dt = R.from_quat(q_t_plus_dt_norm)
                delta_rot = rot_t.inv() * rot_t_plus_dt
                
                angular_vel[:-1][valid_pairs_mask] = delta_rot.as_rotvec() / time_delta
            
            angular_vel[-1, :] = angular_vel[-2, :] if num_samples > 1 else 0
                    
            return angular_vel
        
        
        def calculate_angular_distance(rot_data, cumulative=False):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_dist = np.zeros(num_samples)
            
            if num_samples < 2:
                return angular_dist
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q1 = quat_values[:-1][valid_pairs_mask]
                q2 = quat_values[1:][valid_pairs_mask]
                
                q1_norms = quat_norms[:-1][valid_pairs_mask]
                q2_norms = quat_norms[1:][valid_pairs_mask]
                
                q1_norm = q1 / q1_norms[:, np.newaxis]
                q2_norm = q2 / q2_norms[:, np.newaxis]
                
                r1 = R.from_quat(q1_norm)
                r2 = R.from_quat(q2_norm)
                relative_rotation = r1.inv() * r2
                
                angular_dist[:-1][valid_pairs_mask] = np.linalg.norm(relative_rotation.as_rotvec(), axis=1)
            
            if num_samples > 1:
                angular_dist[-1] = angular_dist[-2] if cumulative else 0
                
            if cumulative:
                angular_dist = np.cumsum(angular_dist)
                    
            return angular_dist
    
    
        
        linear_accel_list = []
        angular_vel_list = []
        angular_distance_list = []
        
        for _, group in (df.groupby('sequence_id')):
            acc_data_group = group[['acc_x', 'acc_y', 'acc_z']].values
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            linear_accel_group = remove_gravity_from_acc(acc_data_group, rot_data_group)
            linear_accel_list.append(pd.DataFrame(linear_accel_group, columns=['f1', 'f2', 'f3'], index=group.index))

            ot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_vel_group = calculate_angular_velocity_from_quat(rot_data_group)
            angular_vel_list.append(pd.DataFrame(angular_vel_group, columns=['f4', 'f5', 'f6'], index=group.index))

            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_dist_group = calculate_angular_distance(rot_data_group)
            angular_distance_list.append(pd.DataFrame(angular_dist_group, columns=['f7'], index=group.index))
        
        df_linear_accel = pd.concat(linear_accel_list)    
        df_angular_vel = pd.concat(angular_vel_list)    
        df_angular_distance = pd.concat(angular_distance_list)
        df = pd.concat([df, df_linear_accel, df_angular_vel, df_angular_distance], axis=1)
    
        feature_names = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
    
    
        thm_mean = df[['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']].mean(1)
    
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = (np.where(df[col].isna(), thm_mean, df[col]) ) / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
            if self.tof_mode == -1:
                for mode in [2, 4, 8, 16, 32]:
                    region_size = 64 // mode
                    for r in (range(mode)):
                        region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                        new_columns.update({
                            f'tof{mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                            f'tof{mode}_{i}_region_{r}_std': region_data.std(axis=1),
                            f'tof{mode}_{i}_region_{r}_min': region_data.min(axis=1),
                            f'tof{mode}_{i}_region_{r}_max': region_data.max(axis=1)
                        })
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,14:] = 0.
            
            for model in self.models:
                model.eval()
                p = model(x)
                if outputs is None: outputs = p
                else: outputs += p
            outputs /= len(self.models)

        return self.gesture_classes, outputs.cpu().numpy()

In [57]:
model_zhou_inference_v15_1 = model_zhou_v15('/kaggle/input/cmi3v58/42', 42, './42')
if VALIDATION:
    MMM = model_zhou_inference_v15_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, 5, 6, 7]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_15_1, pred_imuonly_15_1, targets, idlist_15_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_15_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_15_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [58]:
model_zhou_inference_v15_2 = model_zhou_v15('/kaggle/input/cmi3v58/6665252', 6665252, './6665252')
if VALIDATION:
    MMM = model_zhou_inference_v15_2
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, 5, 6, 7]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_15_2, pred_imuonly_15_2, targets, idlist_15_2 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_15_2.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_15_2.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [59]:
model_zhou_inference_v15_3 = model_zhou_v15('/kaggle/input/cmi3v58/88885252', 88885252, './88885252')
if VALIDATION:
    MMM = model_zhou_inference_v15_3
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, 5, 6, 7]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_15_3, pred_imuonly_15_3, targets, idlist_15_3 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_15_3.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_15_3.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [60]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented
        
transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v16:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 150
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4,]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                out += shortcut
                out = self.act(out)
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context

        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.feature_engineering = feature_engineering
                if feature_engineering:
                    self.imu_fe = ImuFeatureExtractor(**kwargs)
                    imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7
                else:
                    self.imu_fe = nn.Identity()
                    imu_dim = imu_dim_raw   
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7 + 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (9 + 4 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.emb_all = 256 + 256 + 32 + 64
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 256
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.LSTM(self.emb_all, self.dim_lstm, num_layers=1, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 256, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(256)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(256, 5)
                self.classifier = nn.Linear(256, n_classes)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
        
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan//2].transpose(1, 2)  # (batch, imu_dim, seq_len)
                imu2 = x[:, :, self.fir_nchan//2:self.fir_nchan].transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale3[:,:thm.shape[1],:] + self.bias3[:,:thm.shape[1],:]) * mask_all.transpose(1, 2)
                tof = (tof * self.scale4[:,:tof.shape[1],:] + self.bias4[:,:tof.shape[1],:]) * mask_all.transpose(1, 2)
                
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
        
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
        
                x1 = torch.cat((x1, x12, thm, tof), 1)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
    
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':

                    X_ = np.concatenate([X, y2], -1)
    
                    X_ = transforms_custom(X_)
    
                    X = X_[:,:X.shape[-1]]
                    y2 = X_[:,X.shape[-1]:]
        
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))

        # group_list = []
        # for _, group in tqdm(df.groupby('sequence_id')):
        #     index_old = group.index
        #     group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
        #     group.index = index_old
            
        #     group_list.append(group)
        # df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {'Seated Lean Non Dom - FACE DOWN': 1,
                         'Lie on Side - Non Dominant': 2,
                         'Seated Straight': 3,
                         'Lie on Back': 4}
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]

                    # X[:BS//8,:,14:] = 0.
    
                    X_imuonly = X[:].clone()
                    X_imuonly[:, :, 14:] = 0.0
                    
                    X, X_imuonly, y = X.float().to(device), X_imuonly.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    
                    optimizer.zero_grad()
                    
                    LEN = 100 + np.random.randint(100)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.8
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.8
    
                    loss.backward()
                    
                    optimizer.step()
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds, val_targets = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_all, val_targets_all = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 14:] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        
                        logits = model(X)
                        logits_imuonly = model(X_imuonly)
    
                        val_preds.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())

                        val_preds_logits.append(logits.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_preds_all.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_all.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_targets.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_imuonly.append(y.argmax(dim=1).cpu().numpy())
    
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
    
                        loss = F.cross_entropy(logits, y)
                        val_loss += loss.item()
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_split = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_all)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_all)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    val_loss /= len(val_loader)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ALL : ', round(val_acc, 4),  '| SPLIT : ', round(val_acc_split, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall)
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall)
        
    def get_new_features(self, df):
        # 优化后的函数
        def remove_gravity_from_acc(acc_data, rot_data):
            if isinstance(acc_data, pd.DataFrame):
                acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
            else:
                acc_values = np.asarray(acc_data)
                
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = acc_values.shape[0]
            linear_accel = np.zeros_like(acc_values)
            gravity_world = np.array([0, 0, 9.81])
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            
            valid_quats = quat_values[valid_mask]
            valid_quats_normalized = valid_quats / quat_norms[valid_mask, np.newaxis]
            try:
                rotations = R.from_quat(valid_quats_normalized)
                    
                gravity_sensor_frame = rotations.apply(gravity_world, inverse=True)
            
                linear_accel[valid_mask] = acc_values[valid_mask] - gravity_sensor_frame
            except:
                linear_accel[valid_mask] = acc_values[valid_mask]
            linear_accel[~valid_mask] = acc_values[~valid_mask]
            
            return linear_accel
        
        
        def calculate_angular_velocity_from_quat(rot_data, time_delta=1/200):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_vel = np.zeros((num_samples, 3))
            
            if num_samples < 2:
                return angular_vel
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q_t = quat_values[:-1][valid_pairs_mask]
                q_t_plus_dt = quat_values[1:][valid_pairs_mask]
                
                q_t_norms = quat_norms[:-1][valid_pairs_mask]
                q_t_plus_dt_norms = quat_norms[1:][valid_pairs_mask]
                
                q_t_norm = q_t / q_t_norms[:, np.newaxis]
                q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms[:, np.newaxis]
                
                rot_t = R.from_quat(q_t_norm)
                rot_t_plus_dt = R.from_quat(q_t_plus_dt_norm)
                delta_rot = rot_t.inv() * rot_t_plus_dt
                
                angular_vel[:-1][valid_pairs_mask] = delta_rot.as_rotvec() / time_delta
            
            angular_vel[-1, :] = angular_vel[-2, :] if num_samples > 1 else 0
                    
            return angular_vel
        
        
        def calculate_angular_distance(rot_data, cumulative=False):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_dist = np.zeros(num_samples)
            
            if num_samples < 2:
                return angular_dist
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q1 = quat_values[:-1][valid_pairs_mask]
                q2 = quat_values[1:][valid_pairs_mask]
                
                q1_norms = quat_norms[:-1][valid_pairs_mask]
                q2_norms = quat_norms[1:][valid_pairs_mask]
                
                q1_norm = q1 / q1_norms[:, np.newaxis]
                q2_norm = q2 / q2_norms[:, np.newaxis]
                
                r1 = R.from_quat(q1_norm)
                r2 = R.from_quat(q2_norm)
                relative_rotation = r1.inv() * r2
                
                angular_dist[:-1][valid_pairs_mask] = np.linalg.norm(relative_rotation.as_rotvec(), axis=1)
            
            if num_samples > 1:
                angular_dist[-1] = angular_dist[-2] if cumulative else 0
                
            if cumulative:
                angular_dist = np.cumsum(angular_dist)
                    
            return angular_dist
    
    
        
        linear_accel_list = []
        angular_vel_list = []
        angular_distance_list = []
        
        for _, group in (df.groupby('sequence_id')):
            acc_data_group = group[['acc_x', 'acc_y', 'acc_z']].values
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            linear_accel_group = remove_gravity_from_acc(acc_data_group, rot_data_group)
            linear_accel_list.append(pd.DataFrame(linear_accel_group, columns=['f1', 'f2', 'f3'], index=group.index))

            ot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_vel_group = calculate_angular_velocity_from_quat(rot_data_group)
            angular_vel_list.append(pd.DataFrame(angular_vel_group, columns=['f4', 'f5', 'f6'], index=group.index))

            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_dist_group = calculate_angular_distance(rot_data_group)
            angular_distance_list.append(pd.DataFrame(angular_dist_group, columns=['f7'], index=group.index))
        
        df_linear_accel = pd.concat(linear_accel_list)    
        df_angular_vel = pd.concat(angular_vel_list)    
        df_angular_distance = pd.concat(angular_distance_list)
        df = pd.concat([df, df_linear_accel, df_angular_vel, df_angular_distance], axis=1)
    
        feature_names = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
    
    
        thm_mean = df[['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']].mean(1)
    
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = (np.where(df[col].isna(), thm_mean, df[col]) ) / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
            if self.tof_mode == -1:
                for mode in [2, 4, 8, 16, 32]:
                    region_size = 64 // mode
                    for r in (range(mode)):
                        region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                        new_columns.update({
                            f'tof{mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                            f'tof{mode}_{i}_region_{r}_std': region_data.std(axis=1),
                            f'tof{mode}_{i}_region_{r}_min': region_data.min(axis=1),
                            f'tof{mode}_{i}_region_{r}_max': region_data.max(axis=1)
                        })
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        # sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,14:] = 0.
            
            for model in self.models:
                model.eval()
                p = model(x)
                if outputs is None: outputs = p
                else: outputs += p
            outputs /= len(self.models)

        return self.gesture_classes, outputs.cpu().numpy()

In [61]:
model_zhou_inference_v16_1 = model_zhou_v16('/kaggle/input/cmi3v61', 42, './42')
if VALIDATION:
    MMM = model_zhou_inference_v16_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, ]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_16_1, pred_imuonly_16_1, targets, idlist_16_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_16_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_16_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [62]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented
        
transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v17:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 80
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4, ]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                out += shortcut
                out = self.act(out)
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context



        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.feature_engineering = feature_engineering
                if feature_engineering:
                    self.imu_fe = ImuFeatureExtractor(**kwargs)
                    imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7
                else:
                    self.imu_fe = nn.Identity()
                    imu_dim = imu_dim_raw   
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7 + 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (9 + 4 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 160, 3, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(160, 256, 3, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
        
                self.emb_all = 256 + 256 + 32 + 64
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 256
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.LSTM(self.emb_all, self.dim_lstm, num_layers=1, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 256, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(256)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(256, 5)
                self.classifier = nn.Linear(256, n_classes)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
        
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan//2].transpose(1, 2)  # (batch, imu_dim, seq_len)
                imu2 = x[:, :, self.fir_nchan//2:self.fir_nchan].transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale3[:,:thm.shape[1],:] + self.bias3[:,:thm.shape[1],:]) * mask_all.transpose(1, 2)
                tof = (tof * self.scale4[:,:tof.shape[1],:] + self.bias4[:,:tof.shape[1],:]) * mask_all.transpose(1, 2)
                
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
        
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
        
                x1 = torch.cat((x1, x12, thm, tof), 1)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
    
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':

                    X_ = np.concatenate([X, y2], -1)
    
                    X_ = transforms_custom(X_)
    
                    X = X_[:,:X.shape[-1]]
                    y2 = X_[:,X.shape[-1]:]
        
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {'Seated Lean Non Dom - FACE DOWN': 1,
                         'Lie on Side - Non Dominant': 2,
                         'Seated Straight': 3,
                         'Lie on Back': 4}
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
    
                    X_imuonly = X[:].clone()
                    X_imuonly[:, :, 14:] = 0.0

                    
                    X[BS//8:, :, 14:] = 0.0
                    
                    
                    X, X_imuonly, y = X.float().to(device), X_imuonly.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    
                    optimizer.zero_grad()
                    
                    LEN = 170 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.5
    
                    loss.backward()
                    
                    optimizer.step()
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds, val_targets = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_all, val_targets_all = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 14:] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        
                        logits = model(X)
                        logits_imuonly = model(X_imuonly)
    
                        val_preds.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())

                        val_preds_logits.append(logits.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_preds_all.append(logits.argmax(dim=1).cpu().numpy())
                        val_preds_all.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_targets.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_imuonly.append(y.argmax(dim=1).cpu().numpy())
    
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
                        val_targets_all.append(y.argmax(dim=1).cpu().numpy())
    
                        loss = F.cross_entropy(logits, y)
                        val_loss += loss.item()
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_split = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_all)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_all)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    val_loss /= len(val_loader)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ALL : ', round(val_acc, 4),  '| SPLIT : ', round(val_acc_split, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc_imuonly # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall)
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall)
        
    def get_new_features(self, df):
        # 优化后的函数
        def remove_gravity_from_acc(acc_data, rot_data):
            if isinstance(acc_data, pd.DataFrame):
                acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
            else:
                acc_values = np.asarray(acc_data)
                
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = acc_values.shape[0]
            linear_accel = np.zeros_like(acc_values)
            gravity_world = np.array([0, 0, 9.81])
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            
            valid_quats = quat_values[valid_mask]
            valid_quats_normalized = valid_quats / quat_norms[valid_mask, np.newaxis]
            try:
                rotations = R.from_quat(valid_quats_normalized)
                    
                gravity_sensor_frame = rotations.apply(gravity_world, inverse=True)
            
                linear_accel[valid_mask] = acc_values[valid_mask] - gravity_sensor_frame
            except:
                linear_accel[valid_mask] = acc_values[valid_mask]
            linear_accel[~valid_mask] = acc_values[~valid_mask]
            
            return linear_accel
        
        
        def calculate_angular_velocity_from_quat(rot_data, time_delta=1/200):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_vel = np.zeros((num_samples, 3))
            
            if num_samples < 2:
                return angular_vel
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q_t = quat_values[:-1][valid_pairs_mask]
                q_t_plus_dt = quat_values[1:][valid_pairs_mask]
                
                q_t_norms = quat_norms[:-1][valid_pairs_mask]
                q_t_plus_dt_norms = quat_norms[1:][valid_pairs_mask]
                
                q_t_norm = q_t / q_t_norms[:, np.newaxis]
                q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms[:, np.newaxis]
                
                rot_t = R.from_quat(q_t_norm)
                rot_t_plus_dt = R.from_quat(q_t_plus_dt_norm)
                delta_rot = rot_t.inv() * rot_t_plus_dt
                
                angular_vel[:-1][valid_pairs_mask] = delta_rot.as_rotvec() / time_delta
            
            angular_vel[-1, :] = angular_vel[-2, :] if num_samples > 1 else 0
                    
            return angular_vel
        
        
        def calculate_angular_distance(rot_data, cumulative=False):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_dist = np.zeros(num_samples)
            
            if num_samples < 2:
                return angular_dist
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q1 = quat_values[:-1][valid_pairs_mask]
                q2 = quat_values[1:][valid_pairs_mask]
                
                q1_norms = quat_norms[:-1][valid_pairs_mask]
                q2_norms = quat_norms[1:][valid_pairs_mask]
                
                q1_norm = q1 / q1_norms[:, np.newaxis]
                q2_norm = q2 / q2_norms[:, np.newaxis]
                
                r1 = R.from_quat(q1_norm)
                r2 = R.from_quat(q2_norm)
                relative_rotation = r1.inv() * r2
                
                angular_dist[:-1][valid_pairs_mask] = np.linalg.norm(relative_rotation.as_rotvec(), axis=1)
            
            if num_samples > 1:
                angular_dist[-1] = angular_dist[-2] if cumulative else 0
                
            if cumulative:
                angular_dist = np.cumsum(angular_dist)
                    
            return angular_dist
    
    
        
        linear_accel_list = []
        angular_vel_list = []
        angular_distance_list = []
        
        for _, group in (df.groupby('sequence_id')):
            acc_data_group = group[['acc_x', 'acc_y', 'acc_z']].values
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            linear_accel_group = remove_gravity_from_acc(acc_data_group, rot_data_group)
            linear_accel_list.append(pd.DataFrame(linear_accel_group, columns=['f1', 'f2', 'f3'], index=group.index))

            ot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_vel_group = calculate_angular_velocity_from_quat(rot_data_group)
            angular_vel_list.append(pd.DataFrame(angular_vel_group, columns=['f4', 'f5', 'f6'], index=group.index))

            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_dist_group = calculate_angular_distance(rot_data_group)
            angular_distance_list.append(pd.DataFrame(angular_dist_group, columns=['f7'], index=group.index))
        
        df_linear_accel = pd.concat(linear_accel_list)    
        df_angular_vel = pd.concat(angular_vel_list)    
        df_angular_distance = pd.concat(angular_distance_list)
        df = pd.concat([df, df_linear_accel, df_angular_vel, df_angular_distance], axis=1)
    
        feature_names = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
    
    
        thm_mean = df[['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']].mean(1)
    
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = (np.where(df[col].isna(), thm_mean, df[col]) ) / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
            if self.tof_mode == -1:
                for mode in [2, 4, 8, 16, 32]:
                    region_size = 64 // mode
                    for r in (range(mode)):
                        region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                        new_columns.update({
                            f'tof{mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                            f'tof{mode}_{i}_region_{r}_std': region_data.std(axis=1),
                            f'tof{mode}_{i}_region_{r}_min': region_data.min(axis=1),
                            f'tof{mode}_{i}_region_{r}_max': region_data.max(axis=1)
                        })
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,14:] = 0.
            
            for model in self.models:
                model.eval()
                p = model(x)
                if outputs is None: outputs = p
                else: outputs += p
            outputs /= len(self.models)

        return self.gesture_classes, outputs.cpu().numpy()

In [63]:
model_zhou_inference_v17_1 = model_zhou_v17('/kaggle/input/cmi3v63/42', 42, './42')
if VALIDATION:
    MMM = model_zhou_inference_v17_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, ]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_17_1, pred_imuonly_17_1, targets, idlist_17_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_17_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_17_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [64]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented
        
transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v20:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4,]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=1, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()

                self.gru = nn.GRU(out_channels, out_channels//2, batch_first=True, bidirectional=True)
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                
                out += shortcut
                
                # out = self.act(out)

                out = out + self.gru(out.transpose(1, 2))[0].transpose(1, 2)
                
                out = self.act(out)
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context

        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.feature_engineering = feature_engineering
                if feature_engineering:
                    self.imu_fe = ImuFeatureExtractor(**kwargs)
                    imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7
                else:
                    self.imu_fe = nn.Identity()
                    imu_dim = imu_dim_raw   
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7 + 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (5 + 3 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(256, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(256, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.emb_all = 256 + 256 + 32 + 64
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 256+128
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.GRU(self.emb_all, self.dim_lstm, num_layers=1, bidirectional=True, batch_first=True)

                self.encoder = ResidualSECNNBlock(self.emb_all, self.dim_lstm, 5)
                
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 256, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(256)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(256, 5)
                self.classifier = nn.Linear(256, n_classes)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
        
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan//2].transpose(1, 2)  # (batch, imu_dim, seq_len)
                imu2 = x[:, :, self.fir_nchan//2:self.fir_nchan].transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale3[:,:thm.shape[1],:]  + self.bias3[:,:thm.shape[1],:] ) * mask_all.transpose(1, 2)
                tof = (tof * self.scale4[:,:tof.shape[1],:]  + self.bias4[:,:tof.shape[1],:] ) * mask_all.transpose(1, 2)

                
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)

                 # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
                
        
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                        
                x1 = torch.cat((x1, x12, thm, tof), 1)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)

                # lstm_out = self.encoder(merged).transpose(1,2)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits, logits3
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()
        
        if np.any(norms < 1e-8):
            mask = norms < 1e-8
            tmp[mask] = [0.0, 0.0, 0.0, 0.0]
        
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr

    
    def train(self, ):
        tof_mode = self.tof_mode
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':

                    X_ = np.concatenate([X, y2], -1)
    
                    X_ = transforms_custom(X_)
    
                    X = X_[:,:X.shape[-1]]
                    y2 = X_[:,X.shape[-1]:]

                    ratio = np.random.random() *0.85+0.3
                    
                    #X[:,:3] = X[:,:3] * ratio
                    #X[:,7:7+3] = X[:,7:7+3] * ratio
                    #X[:,14:14+5] = X[:,14:14+5] * ratio

                    #X[:,14+10+5:] = X[:,14+10+5:] * ratio
        
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {'Seated Lean Non Dom - FACE DOWN': 1,
                         'Lie on Side - Non Dominant': 2,
                         'Seated Straight': 3,
                         'Lie on Back': 4}
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        predorientationall = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=True, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA 
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)
            model2 = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)

            hidden_weights2 = [p for p in model2.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases2 = [p for p in model2.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups2 = [
                dict(params=hidden_weights2, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases2, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer2 = SingleDeviceMuonWithAuxAdam(param_groups2)

            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                model2.train()
                
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
                    X0 = X.clone()

                    X[BS//8:,:,14:] = 0.

                    if np.random.random() > 0.5:
                        X[:BS//8*2,:,3:7] = 0.
                        X[:BS//8*2,:,7+3:7+7] = 0.
                    else:
                        X[:BS//8*2,:,:3] = 0.
                        X[:BS//8*2,:,7+0:7+3] = 0.
                    
                    X, y = X.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    X0 = X0.float().to(device)
                    
                    optimizer.zero_grad()
                    optimizer2.zero_grad()
                    
                    LEN = 1200 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
                    
                    logits_, logits2_, logits3_ = model2(X0[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.5

                    clipval = 0.6
                    loss2 = clipped_cross_entropy(logits_, y, clipval, 18) 
                    loss2 += clipped_cross_entropy(logits2_, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss2 += clipped_cross_entropy(logits3_, y3, clipval, 5)  * 0.5

                    if epoch > 40:
                        loss = loss + loss2 + criterion_rdrop(logits, logits_)
                    else:
                        loss = loss + loss2 
                    
                    loss.backward()
                    
                    optimizer.step()
                    optimizer2.step()
                    
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                model2.eval()
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds_accmask, val_targets_accmask = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_rotmask, val_targets_rotmask = [], []

                val_preds_mother, val_targets_rotmask = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []

                val_preds_orientation, val_targets_orientation = [], []
                val_preds_orientation_logits = []
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 14:] = 0.0

                        X_accmask = X[:].clone()
                        X_accmask[:, :, 14:] = 0.0
                        X_accmask[:, :, 0:3] = 0.0
                        X_accmask[:, :, 7+0:7+3] = 0.0

                        X_rotmask = X[:].clone()
                        X_rotmask[:, :, 14:] = 0.0
                        X_rotmask[:, :, 3:7] = 0.0
                        X_rotmask[:, :, 7+3:7+7] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        X_accmask = X_accmask.float().to(device)
                        X_rotmask = X_rotmask.float().to(device)

                        logits_mother, _ = model(X)
                        
                        logits_rotmask, _  = model(X_rotmask)
                        logits_accmask, _  = model(X_accmask)
                        logits_imuonly, logits_orientation  = model(X_imuonly)

                        val_preds_mother.append(logits_mother.argmax(dim=1).cpu().numpy())
                        val_preds_accmask.append(logits_accmask.argmax(dim=1).cpu().numpy())
                        val_preds_rotmask.append(logits_rotmask.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_preds_logits.append(logits_mother.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_targets_imuonly.append(y.argmax(dim=1).cpu().numpy())


                        val_preds_orientation_logits.append(logits_orientation.cpu().numpy())
                        val_preds_orientation.append(logits_orientation.argmax(dim=1).cpu().numpy())
                        val_targets_orientation.append(y3.argmax(dim=1).cpu().numpy())
    
    
                        loss = F.cross_entropy(logits_imuonly, y)
                        val_loss += loss.item()
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc_rotmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_rotmask)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_accmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_accmask)]})
                    )
                    val_acc_mother = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_mother)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    val_loss /= len(val_loader)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ROTMASK : ', round(val_acc_rotmask, 4),  '| ACCMASK : ', round(val_acc_accmask, 4),  '| MOTHER : ', round(val_acc_mother, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc_imuonly # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets_imuonly))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                predorientationall.append(np.concatenate(val_preds_orientation_logits, 0))
                
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall),
                'pred_orientation' : np.concatenate(predorientationall, 0)
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall), np.concatenate(predorientationall, 0)
        
    def get_new_features(self, df):
        # 优化后的函数
        def remove_gravity_from_acc(acc_data, rot_data):
            if isinstance(acc_data, pd.DataFrame):
                acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
            else:
                acc_values = np.asarray(acc_data)
                
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = acc_values.shape[0]
            linear_accel = np.zeros_like(acc_values)
            gravity_world = np.array([0, 0, 9.81])
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            
            valid_quats = quat_values[valid_mask]
            valid_quats_normalized = valid_quats / quat_norms[valid_mask, np.newaxis]
            try:
                rotations = R.from_quat(valid_quats_normalized)
                    
                gravity_sensor_frame = rotations.apply(gravity_world, inverse=True)
            
                linear_accel[valid_mask] = acc_values[valid_mask] - gravity_sensor_frame
            except:
                linear_accel[valid_mask] = acc_values[valid_mask]
            linear_accel[~valid_mask] = acc_values[~valid_mask]
            
            return linear_accel
        
        
        def calculate_angular_velocity_from_quat(rot_data, time_delta=1/20):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_vel = np.zeros((num_samples, 3))
            
            if num_samples < 2:
                return angular_vel
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q_t = quat_values[:-1][valid_pairs_mask]
                q_t_plus_dt = quat_values[1:][valid_pairs_mask]
                
                q_t_norms = quat_norms[:-1][valid_pairs_mask]
                q_t_plus_dt_norms = quat_norms[1:][valid_pairs_mask]
                
                q_t_norm = q_t / q_t_norms[:, np.newaxis]
                q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms[:, np.newaxis]
                
                rot_t = R.from_quat(q_t_norm)
                rot_t_plus_dt = R.from_quat(q_t_plus_dt_norm)
                delta_rot = rot_t.inv() * rot_t_plus_dt
                
                angular_vel[:-1][valid_pairs_mask] = delta_rot.as_rotvec() / time_delta
            
            angular_vel[-1, :] = angular_vel[-2, :] if num_samples > 1 else 0
                    
            return angular_vel
        
        
        def calculate_angular_distance(rot_data, cumulative=False):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_dist = np.zeros(num_samples)
            
            if num_samples < 2:
                return angular_dist
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q1 = quat_values[:-1][valid_pairs_mask]
                q2 = quat_values[1:][valid_pairs_mask]
                
                q1_norms = quat_norms[:-1][valid_pairs_mask]
                q2_norms = quat_norms[1:][valid_pairs_mask]
                
                q1_norm = q1 / q1_norms[:, np.newaxis]
                q2_norm = q2 / q2_norms[:, np.newaxis]
                
                r1 = R.from_quat(q1_norm)
                r2 = R.from_quat(q2_norm)
                relative_rotation = r1.inv() * r2
                
                angular_dist[:-1][valid_pairs_mask] = np.linalg.norm(relative_rotation.as_rotvec(), axis=1)
            
            if num_samples > 1:
                angular_dist[-1] = angular_dist[-2] if cumulative else 0
                
            if cumulative:
                angular_dist = np.cumsum(angular_dist)
                    
            return angular_dist
        
        linear_accel_list = []
        angular_vel_list = []
        angular_distance_list = []
        
        for _, group in (df.groupby('sequence_id')):
            acc_data_group = group[['acc_x', 'acc_y', 'acc_z']].values
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            linear_accel_group = remove_gravity_from_acc(acc_data_group, rot_data_group)
            linear_accel_list.append(pd.DataFrame(linear_accel_group, columns=['f1', 'f2', 'f3'], index=group.index))

            ot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_vel_group = calculate_angular_velocity_from_quat(rot_data_group)
            angular_vel_list.append(pd.DataFrame(angular_vel_group, columns=['f4', 'f5', 'f6'], index=group.index))

            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_dist_group = calculate_angular_distance(rot_data_group)
            angular_distance_list.append(pd.DataFrame(angular_dist_group, columns=['f7'], index=group.index))
        
        df_linear_accel = pd.concat(linear_accel_list)    
        df_angular_vel = pd.concat(angular_vel_list)    
        df_angular_distance = pd.concat(angular_distance_list)
        df = pd.concat([df, df_linear_accel, df_angular_vel, df_angular_distance], axis=1)
    
        feature_names = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']

        df[['acc_x', 'acc_y', 'acc_z']] = df[['acc_x', 'acc_y', 'acc_z']] / 5.6
        df[['f1', 'f2', 'f3',]] = df[['f1', 'f2', 'f3',]] / 5.6
    
        thm_mean = df[['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']].mean(1)
    
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = (np.where(df[col].isna(), thm_mean, df[col]) ) / 3.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 4 - df[pixel_cols].replace(-1, 255.) / 82. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                # f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                # f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        # f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })

        
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)


        # cols_extra = ['acc_x', 'acc_y', 'acc_z', 'f1', 'f2', 'f3'] + [col for col in df.columns if 'thm_' in col ] + [col for col in df.columns if 'tof' in col and '_isna' not in col ]
        
        # group_list = []
        # dict_sid = {}
        # for sid, group in tqdm(df.groupby('sequence_id')):
        #     std_all = group[['acc_x', 'acc_y', 'acc_z',]].std(0).mean()
        #     dict_sid[sid] = std_all

        # df['sid_std'] = df['sequence_id'].map(dict_sid)
        # for col in tqdm(cols_extra):
        #     df[col] = df[col].values / (df['sid_std'] + 1e-6)
        # del df['sid_std']

        tof_cols = list(df_tof.columns)
        tof_cols_head = tof_cols[:: (self.tof_mode * 2 + 4) ] +  tof_cols[1:: (self.tof_mode * 2 + 4) ]
        tof_cols = tof_cols_head + [item for item in tof_cols if item not in tof_cols_head]
        
        return df, feature_names, tof_cols
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            outputs2 = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,14:] = 0.
            
            for model in self.models:
                model.eval()
                p, p2 = model(x)
                if outputs is None: 
                    outputs = p
                    outputs2 = p2
                else: 
                    outputs += p
                    outputs2 += p2
                    
            outputs /= len(self.models)
            outputs2 /= len(self.models)

        return self.gesture_classes, outputs.cpu().numpy()

In [65]:
model_zhou_inference_v20_1 = model_zhou_v20('/kaggle/input/cmi3v75/42', 42, './42')
if VALIDATION:
    MMM = model_zhou_inference_v20_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, ]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_20_1, pred_imuonly_20_1, targets, idlist_20_1, pred_orientation_20_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_20_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_20_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [66]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented
        
transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v21:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4,]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()

                self.gru = nn.GRU(out_channels, out_channels//2, batch_first=True, bidirectional=True)
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                
                out += shortcut
                
                # out = self.act(out)

                out = out + self.gru(out.transpose(1, 2))[0].transpose(1, 2)
                out = self.act(out)
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context

        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.feature_engineering = feature_engineering
                if feature_engineering:
                    self.imu_fe = ImuFeatureExtractor(**kwargs)
                    imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7
                else:
                    self.imu_fe = nn.Identity()
                    imu_dim = imu_dim_raw   
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7 + 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (5 + 3 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 320, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(320, 512, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 320, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(320, 512, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 128, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(128, 128, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.emb_all = 512 + 512 + 64 + 128
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 512
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.GRU(self.emb_all, self.dim_lstm, num_layers=1, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 256, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(256)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(256, 5)
                self.classifier = nn.Linear(256, n_classes)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
        
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan//2].transpose(1, 2)  # (batch, imu_dim, seq_len)
                imu2 = x[:, :, self.fir_nchan//2:self.fir_nchan].transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale3[:,:thm.shape[1],:]  + self.bias3[:,:thm.shape[1],:] ) * mask_all.transpose(1, 2)
                tof = (tof * self.scale4[:,:tof.shape[1],:]  + self.bias4[:,:tof.shape[1],:] ) * mask_all.transpose(1, 2)

                
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
        
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
        
                x1 = torch.cat((x1, x12, thm, tof), 1)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits, logits3
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()

        if np.any(norms < 1e-8):
            mask = norms < 1e-8
            tmp[mask] = [0.0, 0.0, 0.0, 0.0]
            
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
    
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':

                    X_ = np.concatenate([X, y2], -1)
    
                    X_ = transforms_custom(X_)
    
                    X = X_[:,:X.shape[-1]]
                    y2 = X_[:,X.shape[-1]:]
        
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {'Seated Lean Non Dom - FACE DOWN': 1,
                         'Lie on Side - Non Dominant': 2,
                         'Seated Straight': 3,
                         'Lie on Back': 4}
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        predorientationall = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA 
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)
            model2 = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)

            hidden_weights2 = [p for p in model2.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases2 = [p for p in model2.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups2 = [
                dict(params=hidden_weights2, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases2, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer2 = SingleDeviceMuonWithAuxAdam(param_groups2)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                model2.train()
                
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
                    X0 = X.clone()

                    X[BS//8:,:,14:] = 0.

                    if np.random.random() > 0.5:
                        X[:BS//8*2,:,3:7] = 0.
                        X[:BS//8*2,:,7+3:7+7] = 0.
                    else:
                        X[:BS//8*2,:,:3] = 0.
                        X[:BS//8*2,:,7+0:7+3] = 0.
                    
                    X, y = X.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    X0 = X0.float().to(device)
                    
                    optimizer.zero_grad()
                    optimizer2.zero_grad()
                    
                    LEN = 1200 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
                    
                    logits_, logits2_, logits3_ = model2(X0[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.5

                    clipval = 0.6
                    loss2 = clipped_cross_entropy(logits_, y, clipval, 18) 
                    loss2 += clipped_cross_entropy(logits2_, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss2 += clipped_cross_entropy(logits3_, y3, clipval, 5)  * 0.5

                    if epoch > 40:
                        loss = loss + loss2 + criterion_rdrop(logits, logits_)
                    else:
                        loss = loss + loss2 
                    
                    loss.backward()
                    
                    optimizer.step()
                    optimizer2.step()
                    
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                model2.eval()
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds_accmask, val_targets_accmask = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_rotmask, val_targets_rotmask = [], []

                val_preds_mother, val_targets_rotmask = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []

                val_preds_orientation, val_targets_orientation = [], []
                val_preds_orientation_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 14:] = 0.0

                        X_accmask = X[:].clone()
                        X_accmask[:, :, 14:] = 0.0
                        X_accmask[:, :, 0:3] = 0.0
                        X_accmask[:, :, 7+0:7+3] = 0.0

                        X_rotmask = X[:].clone()
                        X_rotmask[:, :, 14:] = 0.0
                        X_rotmask[:, :, 3:7] = 0.0
                        X_rotmask[:, :, 7+3:7+7] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        X_accmask = X_accmask.float().to(device)
                        X_rotmask = X_rotmask.float().to(device)

                        logits_mother, _ = model(X)
                        
                        logits_rotmask, _  = model(X_rotmask)
                        logits_accmask, _  = model(X_accmask)
                        logits_imuonly, logits_orientation  = model(X_imuonly)

                        val_preds_mother.append(logits_mother.argmax(dim=1).cpu().numpy())
                        val_preds_accmask.append(logits_accmask.argmax(dim=1).cpu().numpy())
                        val_preds_rotmask.append(logits_rotmask.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_preds_logits.append(logits_mother.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_targets_imuonly.append(y.argmax(dim=1).cpu().numpy())


                        val_preds_orientation_logits.append(logits_orientation.cpu().numpy())
                        val_preds_orientation.append(logits_orientation.argmax(dim=1).cpu().numpy())
                        val_targets_orientation.append(y3.argmax(dim=1).cpu().numpy())
    
    
                        loss = F.cross_entropy(logits_imuonly, y)
                        val_loss += loss.item()
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc_rotmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_rotmask)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_accmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_accmask)]})
                    )
                    val_acc_mother = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_mother)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    val_loss /= len(val_loader)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ROTMASK : ', round(val_acc_rotmask, 4),  '| ACCMASK : ', round(val_acc_accmask, 4),  '| MOTHER : ', round(val_acc_mother, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc_imuonly # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets_imuonly))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                predorientationall.append(np.concatenate(val_preds_orientation_logits, 0))
                
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall),
                'pred_orientation' : np.concatenate(predorientationall, 0)
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall), np.concatenate(predorientationall, 0)
        
    def get_new_features(self, df):
        # 优化后的函数
        def remove_gravity_from_acc(acc_data, rot_data):
            if isinstance(acc_data, pd.DataFrame):
                acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
            else:
                acc_values = np.asarray(acc_data)
                
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = acc_values.shape[0]
            linear_accel = np.zeros_like(acc_values)
            gravity_world = np.array([0, 0, 9.81])
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            
            valid_quats = quat_values[valid_mask]
            valid_quats_normalized = valid_quats / quat_norms[valid_mask, np.newaxis]
            try:
                rotations = R.from_quat(valid_quats_normalized)
                    
                gravity_sensor_frame = rotations.apply(gravity_world, inverse=True)
            
                linear_accel[valid_mask] = acc_values[valid_mask] - gravity_sensor_frame
            except:
                linear_accel[valid_mask] = acc_values[valid_mask]
            linear_accel[~valid_mask] = acc_values[~valid_mask]
            
            return linear_accel
        
        
        def calculate_angular_velocity_from_quat(rot_data, time_delta=1/200):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_vel = np.zeros((num_samples, 3))
            
            if num_samples < 2:
                return angular_vel
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q_t = quat_values[:-1][valid_pairs_mask]
                q_t_plus_dt = quat_values[1:][valid_pairs_mask]
                
                q_t_norms = quat_norms[:-1][valid_pairs_mask]
                q_t_plus_dt_norms = quat_norms[1:][valid_pairs_mask]
                
                q_t_norm = q_t / q_t_norms[:, np.newaxis]
                q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms[:, np.newaxis]
                
                rot_t = R.from_quat(q_t_norm)
                rot_t_plus_dt = R.from_quat(q_t_plus_dt_norm)
                delta_rot = rot_t.inv() * rot_t_plus_dt
                
                angular_vel[:-1][valid_pairs_mask] = delta_rot.as_rotvec() / time_delta
            
            angular_vel[-1, :] = angular_vel[-2, :] if num_samples > 1 else 0
                    
            return angular_vel
        
        
        def calculate_angular_distance(rot_data, cumulative=False):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_dist = np.zeros(num_samples)
            
            if num_samples < 2:
                return angular_dist
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q1 = quat_values[:-1][valid_pairs_mask]
                q2 = quat_values[1:][valid_pairs_mask]
                
                q1_norms = quat_norms[:-1][valid_pairs_mask]
                q2_norms = quat_norms[1:][valid_pairs_mask]
                
                q1_norm = q1 / q1_norms[:, np.newaxis]
                q2_norm = q2 / q2_norms[:, np.newaxis]
                
                r1 = R.from_quat(q1_norm)
                r2 = R.from_quat(q2_norm)
                relative_rotation = r1.inv() * r2
                
                angular_dist[:-1][valid_pairs_mask] = np.linalg.norm(relative_rotation.as_rotvec(), axis=1)
            
            if num_samples > 1:
                angular_dist[-1] = angular_dist[-2] if cumulative else 0
                
            if cumulative:
                angular_dist = np.cumsum(angular_dist)
                    
            return angular_dist
    
    
        
        linear_accel_list = []
        angular_vel_list = []
        angular_distance_list = []
        
        for _, group in (df.groupby('sequence_id')):
            acc_data_group = group[['acc_x', 'acc_y', 'acc_z']].values
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            linear_accel_group = remove_gravity_from_acc(acc_data_group, rot_data_group)
            linear_accel_list.append(pd.DataFrame(linear_accel_group, columns=['f1', 'f2', 'f3'], index=group.index))

            ot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_vel_group = calculate_angular_velocity_from_quat(rot_data_group)
            angular_vel_list.append(pd.DataFrame(angular_vel_group, columns=['f4', 'f5', 'f6'], index=group.index))

            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_dist_group = calculate_angular_distance(rot_data_group)
            angular_distance_list.append(pd.DataFrame(angular_dist_group, columns=['f7'], index=group.index))
        
        df_linear_accel = pd.concat(linear_accel_list)    
        df_angular_vel = pd.concat(angular_vel_list)    
        df_angular_distance = pd.concat(angular_distance_list)
        df = pd.concat([df, df_linear_accel, df_angular_vel, df_angular_distance], axis=1)
    
        feature_names = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
    
    
        thm_mean = df[['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']].mean(1)
    
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = (np.where(df[col].isna(), thm_mean, df[col]) ) / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                # f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                # f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        # f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            outputs2 = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,14:] = 0.
            
            for model in self.models:
                model.eval()
                p, p2 = model(x)
                if outputs is None: 
                    outputs = p
                    outputs2 = p2
                else: 
                    outputs += p
                    outputs2 += p2
                    
            outputs /= len(self.models)
            outputs2 /= len(self.models)

        return self.gesture_classes, outputs.cpu().numpy()

In [67]:
model_zhou_inference_v21_1 = model_zhou_v21('/kaggle/input/cmi3v78/42', 42, './42')
if VALIDATION:
    MMM = model_zhou_inference_v21_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, ]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_21_1, pred_imuonly_21_1, targets, idlist_21_1, pred_orientation_21_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_21_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_21_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [68]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented
        
transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v22:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4,]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()

                self.gru = nn.GRU(out_channels, out_channels//2, batch_first=True, bidirectional=True)
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                
                out += shortcut
                
                # out = self.act(out)

                out = out + self.gru(out.transpose(1, 2))[0].transpose(1, 2)
                out = self.act(out)
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context

        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.feature_engineering = feature_engineering
                if feature_engineering:
                    self.imu_fe = ImuFeatureExtractor(**kwargs)
                    imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7
                else:
                    self.imu_fe = nn.Identity()
                    imu_dim = imu_dim_raw   
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7 + 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (5 + 3 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 320, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(320, 512, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 320, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(320, 512, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 128, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(128, 128, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.emb_all = 512 + 512 + 64 + 128
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 512
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.GRU(self.emb_all, self.dim_lstm, num_layers=1, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 256, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(256)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(256, 5)
                self.classifier = nn.Linear(256, n_classes)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
        
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan//2].transpose(1, 2)  # (batch, imu_dim, seq_len)
                imu2 = x[:, :, self.fir_nchan//2:self.fir_nchan].transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale3[:,:thm.shape[1],:]  + self.bias3[:,:thm.shape[1],:] ) * mask_all.transpose(1, 2)
                tof = (tof * self.scale4[:,:tof.shape[1],:]  + self.bias4[:,:tof.shape[1],:] ) * mask_all.transpose(1, 2)

                
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
        
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
        
                x1 = torch.cat((x1, x12, thm, tof), 1)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits, logits3
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()

        if np.any(norms < 1e-8):
            mask = norms < 1e-8
            tmp[mask] = [0.0, 0.0, 0.0, 0.0]
            
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
    
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':

                    X_ = np.concatenate([X, y2], -1)
    
                    X_ = transforms_custom(X_)
    
                    X = X_[:,:X.shape[-1]]
                    y2 = X_[:,X.shape[-1]:]
        
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {'Seated Lean Non Dom - FACE DOWN': 1,
                         'Lie on Side - Non Dominant': 2,
                         'Seated Straight': 3,
                         'Lie on Back': 4}
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        predorientationall = []
        targetsorientations = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA 
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)
            model2 = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)

            hidden_weights2 = [p for p in model2.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases2 = [p for p in model2.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups2 = [
                dict(params=hidden_weights2, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases2, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer2 = SingleDeviceMuonWithAuxAdam(param_groups2)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                model2.train()
                
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
                    X0 = X.clone()

                    X[BS//8:,:,14:] = 0.

                    if np.random.random() > 0.5:
                        X[:BS//8*2,:,3:7] = 0.
                        X[:BS//8*2,:,7+3:7+7] = 0.
                    else:
                        X[:BS//8*2,:,:3] = 0.
                        X[:BS//8*2,:,7+0:7+3] = 0.
                    
                    X, y = X.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    X0 = X0.float().to(device)
                    
                    optimizer.zero_grad()
                    optimizer2.zero_grad()
                    
                    LEN = 1200 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
                    
                    logits_, logits2_, logits3_ = model2(X0[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.5

                    clipval = 0.6
                    loss2 = clipped_cross_entropy(logits_, y, clipval, 18) 
                    loss2 += clipped_cross_entropy(logits2_, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss2 += clipped_cross_entropy(logits3_, y3, clipval, 5)  * 0.5

                    if epoch > 40:
                        loss = loss + loss2 + criterion_rdrop(logits, logits_)
                    else:
                        loss = loss + loss2 
                    
                    loss.backward()
                    
                    optimizer.step()
                    optimizer2.step()
                    
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                model2.eval()
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds_accmask, val_targets_accmask = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_rotmask, val_targets_rotmask = [], []

                val_preds_mother, val_targets_rotmask = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []

                val_preds_orientation, val_targets_orientation = [], []
                val_preds_orientation_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 14:] = 0.0

                        X_accmask = X[:].clone()
                        X_accmask[:, :, 14:] = 0.0
                        X_accmask[:, :, 0:3] = 0.0
                        X_accmask[:, :, 7+0:7+3] = 0.0

                        X_rotmask = X[:].clone()
                        X_rotmask[:, :, 14:] = 0.0
                        X_rotmask[:, :, 3:7] = 0.0
                        X_rotmask[:, :, 7+3:7+7] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        X_accmask = X_accmask.float().to(device)
                        X_rotmask = X_rotmask.float().to(device)

                        logits_mother, _ = model(X)
                        
                        logits_rotmask, _  = model(X_rotmask)
                        logits_accmask, _  = model(X_accmask)
                        logits_imuonly, logits_orientation  = model(X_imuonly)

                        val_preds_mother.append(logits_mother.argmax(dim=1).cpu().numpy())
                        val_preds_accmask.append(logits_accmask.argmax(dim=1).cpu().numpy())
                        val_preds_rotmask.append(logits_rotmask.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_preds_logits.append(logits_mother.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_targets_imuonly.append(y.argmax(dim=1).cpu().numpy())


                        val_preds_orientation_logits.append(logits_orientation.cpu().numpy())
                        val_preds_orientation.append(logits_orientation.argmax(dim=1).cpu().numpy())
                        val_targets_orientation.append(y3.argmax(dim=1).cpu().numpy())
    
    
                        loss = F.cross_entropy(logits_imuonly, y)
                        val_loss += loss.item()
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc_rotmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_rotmask)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_accmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_accmask)]})
                    )
                    val_acc_mother = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_mother)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    val_loss /= len(val_loader)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ROTMASK : ', round(val_acc_rotmask, 4),  '| ACCMASK : ', round(val_acc_accmask, 4),  '| MOTHER : ', round(val_acc_mother, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc_imuonly # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets_imuonly))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                predorientationall.append(np.concatenate(val_preds_orientation_logits, 0))
                targetsorientations.append(np.concatenate(val_targets_orientation))
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall),
                'pred_orientation' : np.concatenate(predorientationall, 0),
                'target_orientation' : np.concatenate(targetsorientations), 
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall), np.concatenate(predorientationall, 0), np.concatenate(targetsorientations)
        
    def get_new_features(self, df):
        # 优化后的函数
        def remove_gravity_from_acc(acc_data, rot_data):
            if isinstance(acc_data, pd.DataFrame):
                acc_values = acc_data[['acc_x', 'acc_y', 'acc_z']].values
            else:
                acc_values = np.asarray(acc_data)
                
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = acc_values.shape[0]
            linear_accel = np.zeros_like(acc_values)
            gravity_world = np.array([0, 0, 9.81])
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            
            valid_quats = quat_values[valid_mask]
            valid_quats_normalized = valid_quats / quat_norms[valid_mask, np.newaxis]
            try:
                rotations = R.from_quat(valid_quats_normalized)
                    
                gravity_sensor_frame = rotations.apply(gravity_world, inverse=True)
            
                linear_accel[valid_mask] = acc_values[valid_mask] - gravity_sensor_frame
            except:
                linear_accel[valid_mask] = acc_values[valid_mask]
            linear_accel[~valid_mask] = acc_values[~valid_mask]
            
            return linear_accel
        
        
        def calculate_angular_velocity_from_quat(rot_data, time_delta=1/200):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_vel = np.zeros((num_samples, 3))
            
            if num_samples < 2:
                return angular_vel
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q_t = quat_values[:-1][valid_pairs_mask]
                q_t_plus_dt = quat_values[1:][valid_pairs_mask]
                
                q_t_norms = quat_norms[:-1][valid_pairs_mask]
                q_t_plus_dt_norms = quat_norms[1:][valid_pairs_mask]
                
                q_t_norm = q_t / q_t_norms[:, np.newaxis]
                q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms[:, np.newaxis]
                
                rot_t = R.from_quat(q_t_norm)
                rot_t_plus_dt = R.from_quat(q_t_plus_dt_norm)
                delta_rot = rot_t.inv() * rot_t_plus_dt
                
                angular_vel[:-1][valid_pairs_mask] = delta_rot.as_rotvec() / time_delta
            
            angular_vel[-1, :] = angular_vel[-2, :] if num_samples > 1 else 0
                    
            return angular_vel
        
        
        def calculate_angular_distance(rot_data, cumulative=False):
            if isinstance(rot_data, pd.DataFrame):
                quat_values = rot_data[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            else:
                quat_values = np.asarray(rot_data)
            
            num_samples = quat_values.shape[0]
            angular_dist = np.zeros(num_samples)
            
            if num_samples < 2:
                return angular_dist
            
            quat_norms = np.linalg.norm(quat_values, axis=1)
            valid_mask = ~(np.isnan(quat_norms) | np.isclose(quat_norms, 0))
            valid_pairs_mask = valid_mask[:-1] & valid_mask[1:]
            
            if np.any(valid_pairs_mask):
                q1 = quat_values[:-1][valid_pairs_mask]
                q2 = quat_values[1:][valid_pairs_mask]
                
                q1_norms = quat_norms[:-1][valid_pairs_mask]
                q2_norms = quat_norms[1:][valid_pairs_mask]
                
                q1_norm = q1 / q1_norms[:, np.newaxis]
                q2_norm = q2 / q2_norms[:, np.newaxis]
                
                r1 = R.from_quat(q1_norm)
                r2 = R.from_quat(q2_norm)
                relative_rotation = r1.inv() * r2
                
                angular_dist[:-1][valid_pairs_mask] = np.linalg.norm(relative_rotation.as_rotvec(), axis=1)
            
            if num_samples > 1:
                angular_dist[-1] = angular_dist[-2] if cumulative else 0
                
            if cumulative:
                angular_dist = np.cumsum(angular_dist)
                    
            return angular_dist
    
    
        
        linear_accel_list = []
        angular_vel_list = []
        angular_distance_list = []
        
        for _, group in (df.groupby('sequence_id')):
            acc_data_group = group[['acc_x', 'acc_y', 'acc_z']].values
            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            linear_accel_group = remove_gravity_from_acc(acc_data_group, rot_data_group)
            linear_accel_list.append(pd.DataFrame(linear_accel_group, columns=['f1', 'f2', 'f3'], index=group.index))

            ot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_vel_group = calculate_angular_velocity_from_quat(rot_data_group)
            angular_vel_list.append(pd.DataFrame(angular_vel_group, columns=['f4', 'f5', 'f6'], index=group.index))

            rot_data_group = group[['rot_x', 'rot_y', 'rot_z', 'rot_w']].values
            angular_dist_group = calculate_angular_distance(rot_data_group)
            angular_distance_list.append(pd.DataFrame(angular_dist_group, columns=['f7'], index=group.index))
        
        df_linear_accel = pd.concat(linear_accel_list)    
        df_angular_vel = pd.concat(angular_vel_list)    
        df_angular_distance = pd.concat(angular_distance_list)
        df = pd.concat([df, df_linear_accel, df_angular_vel, df_angular_distance], axis=1)
    
        feature_names = ['f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7']
    
    
        thm_mean = df[['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']].mean(1)
    
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = (np.where(df[col].isna(), thm_mean, df[col]) ) / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                # f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                # f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        # f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            outputs2 = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,14:] = 0.
            
            for model in self.models:
                model.eval()
                p, p2 = model(x)
                if outputs is None: 
                    outputs = p
                    outputs2 = p2
                else: 
                    outputs += p
                    outputs2 += p2
                    
            outputs /= len(self.models)
            outputs2 /= len(self.models)

        return self.gesture_classes, outputs.cpu().numpy()

In [69]:
model_zhou_inference_v22_1 = model_zhou_v22('/kaggle/input/cmi3v79', 42, './42')
if VALIDATION:
    MMM = model_zhou_inference_v22_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, ]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_22_1, pred_imuonly_22_1, targets, idlist_22_1, pred_orientation_22_1, target_orientation_22_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_22_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_22_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [70]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented

transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v25:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4,]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):        
        def get_gravity(quat):
            g=9.81
             # 四元数旋转
            x = quat[..., 0:1]  # rot_x
            y = quat[..., 1:2]  # rot_y
            z = quat[..., 2:3]  # rot_z
            w = quat[..., 3:4]  # rot_w
            g_x = 2 * g * (x * z - w * y)
            g_y = 2 * g * (y * z + w * x)
            g_z = g * (w**2 - x**2 - y**2 + z**2)
            gravity = np.concatenate([g_x, g_y, g_z], -1)
            return gravity
            
        def rotate_quaternion_np(quat: np.ndarray, angle_range: tuple = [(-10, 10), (-10, 10), (-10, 10), ]) -> np.ndarray:
            # 确保输入形状正确
            if quat.ndim != 2 or quat.shape[1] != 4:
                raise ValueError("输入四元数必须是形状为[len, 4]的数组")
            
            seq_len = quat.shape[0]
            
            # 1. 生成绕x、y、z轴的随机旋转角度（度）
            angle_x = np.random.uniform(angle_range[0][0], angle_range[0][1])
            angle_y = np.random.uniform(angle_range[1][0], angle_range[1][1])
            angle_z = np.random.uniform(angle_range[2][0], angle_range[2][1])
            
            # 2. 将角度转换为弧度
            rad_x = math.pi * angle_x / 180.0
            rad_y = math.pi * angle_y / 180.0
            rad_z = math.pi * angle_z / 180.0
            
            # 3. 生成绕各轴旋转的四元数
            qx = np.array([math.sin(rad_x/2), 0, 0, math.cos(rad_x/2)], dtype=np.float32)  # x轴旋转
            qy = np.array([0, math.sin(rad_y/2), 0, math.cos(rad_y/2)], dtype=np.float32)  # y轴旋转
            qz = np.array([0, 0, math.sin(rad_z/2), math.cos(rad_z/2)], dtype=np.float32)  # z轴旋转
            
            # 4. 组合旋转四元数 (q_total = qz * qy * qx，注意乘法顺序)
            q_zy = _quat_mul(qz, qy)
            q_total = _quat_mul(q_zy, qx)
            
            # 5. 将组合旋转应用到每个四元数上
            # 扩展旋转四元数以匹配序列长度
            q_total_expanded = np.tile(q_total, (seq_len, 1))  # [len, 4]
            rotated_quat = _quat_mul(q_total_expanded, quat)
            
            # 6. 归一化确保四元数有效性
            norms = np.linalg.norm(rotated_quat, axis=1, keepdims=True)
            rotated_quat = rotated_quat / np.maximum(norms, 1e-8)  # 防止除零
            
            return rotated_quat
        
        def _quat_mul(q1: np.ndarray, q2: np.ndarray) -> np.ndarray:
            # 提取分量（利用广播机制自动适配维度）
            x1, y1, z1, w1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
            x2, y2, z2, w2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
            
            # 四元数乘法公式（完全向量化）
            x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
            y = w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2
            z = w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2
            w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
            
            # 堆叠结果（保持维度）
            return np.stack([x, y, z, w], axis=-1).astype(q1.dtype)
        
        
        def remove_average_rotation_optimized(q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
            """
            从四元数序列中移除平均旋转，忽略全零四元数，并旋转到指定方向
            Args:
                q: 输入四元数序列，形状 [bs, len, 4]，顺序 [rot_x, rot_y, rot_z, rot_w]
                eps: 数值稳定性阈值
            Returns:
                q_final: 移除平均旋转并旋转到[0.5,0.5,0.5,0.5]方向后的四元数序列，形状 [bs, len, 4]
                         原始全零的位置仍为零
            """
            def is_zero_quaternion(q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
                """
                判断四元数是否为全零（或接近全零）
                Args:
                    q: 四元数，形状 [..., 4]
                    eps: 接近零的阈值
                Returns:
                    布尔张量，形状 [...], 指示每个四元数是否为零
                """
                return torch.norm(q, dim=-1) < eps
            
            def safe_normalize(q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
                """
                安全地规范化四元数，避免除零错误
                Args:
                    q: 四元数，形状 [..., 4]
                    eps: 最小范数阈值
                Returns:
                    规范化后的四元数
                """
                norm = torch.norm(q, dim=-1, keepdim=True)
                # 添加最小范数保护
                norm = torch.clamp(norm, min=eps)
                return q / norm
            
            def quat_multiply(q1: torch.Tensor, q2: torch.Tensor) -> torch.Tensor:
                """
                批量四元数乘法 (q1 * q2)
                Args:
                    q1: 形状 [..., 4]，四元数 [x1,y1,z1,w1]
                    q2: 形状 [..., 4]，四元数 [x2,y2,z2,w2]
                Returns:
                    q: 形状 [..., 4]，乘积 q1*q2 [x,y,z,w]
                """
                x1, y1, z1, w1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
                x2, y2, z2, w2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
                
                x = w1*x2 + x1*w2 + y1*z2 - z1*y2
                y = w1*y2 - x1*z2 + y1*w2 + z1*x2
                z = w1*z2 + x1*y2 - y1*x2 + z1*w2
                w = w1*w2 - x1*x2 - y1*y2 - z1*z2
                
                return torch.stack([x, y, z, w], dim=-1)
            
            def find_first_valid_indices(mask: torch.Tensor) -> torch.Tensor:
                """
                向量化方法查找每个batch中第一个有效四元数的索引
                Args:
                    mask: 有效掩码，形状 [bs, len]
                Returns:
                    indices: 每个batch的第一个有效索引，形状 [bs]
                """
                bs, seq_len = mask.shape
                device = mask.device
                
                # 创建索引矩阵
                indices = torch.arange(seq_len, device=device).expand(bs, seq_len)
                # 将无效位置的索引设置为大数
                indices = torch.where(mask, indices, seq_len)
                # 找到最小索引（第一个有效位置）
                first_valid = torch.min(indices, dim=1)[0]
                # 处理全无效的情况
                all_invalid = first_valid == seq_len
                first_valid[all_invalid] = 0  # 设置为0，后续会处理
                
                return first_valid
            
            def quaternion_average_frechet_optimized(q: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
                """
                计算批量四元数的Frechet均值（优化版本）
                Args:
                    q: 输入四元数，形状 [bs, len, 4]，顺序 [rot_x, rot_y, rot_z, rot_w]
                    mask: 有效四元数掩码，形状 [bs, len]，1表示有效，0表示无效
                    eps: 数值稳定性阈值
                Returns:
                    q_avg: 平均四元数，形状 [bs, 4]
                """
                bs, len_q, _ = q.shape
                device = q.device
                
                # 步骤1：安全地单位化四元数
                q_normalized = safe_normalize(q, eps=eps)
                
                # 应用掩码：无效四元数置为零
                mask_keepdim = mask.unsqueeze(-1)  # [bs, len, 1]
                q_normalized = q_normalized * mask_keepdim
                
                # 步骤2：向量化查找参考四元数
                first_valid_indices = find_first_valid_indices(mask)
                q_ref = q_normalized[torch.arange(bs, device=device), first_valid_indices].unsqueeze(1)  # [bs, 1, 4]
                
                # 步骤3：改进的符号统一策略（添加阈值避免在接近正交时翻转）
                dot = (q_normalized * q_ref).sum(dim=-1, keepdim=True)  # [bs, len, 1]
                q_unified = torch.where(dot < -0.01, -q_normalized, q_normalized)
                
                # 步骤4：构建协方差矩阵 M [bs, 4, 4]，考虑掩码
                q_reshaped = q_unified.unsqueeze(-1)  # [bs, len, 4, 1]
                q_transposed = q_reshaped.transpose(2, 3)  # [bs, len, 1, 4]
                outer_products = q_reshaped @ q_transposed  # [bs, len, 4, 4]
                
                # 应用掩码并求和
                mask_4d = mask.unsqueeze(-1).unsqueeze(-1)  # [bs, len, 1, 1]
                outer_products = outer_products * mask_4d
                M = outer_products.sum(dim=1)  # [bs, 4, 4]
                
                # 处理所有四元数都无效的情况
                valid_count = mask.sum(dim=1)  # [bs]
                all_invalid = valid_count == 0
                if all_invalid.any():
                    # 对全无效的batch，使用单位四元数作为平均
                    M[all_invalid] = torch.eye(4, device=device)
                
                # 步骤5：使用更稳定的特征值求解
                # 添加小单位矩阵增强数值稳定性
                M_stable = M + eps * torch.eye(4, device=device).unsqueeze(0)
                w, v = torch.linalg.eigh(M_stable)  # w: [bs,4], v: [bs,4,4]
                q_avg = v[:, :, -1]  # 最大特征值对应特征向量
                
                # 确保单位化 + 实部(rot_w)为正
                q_avg = safe_normalize(q_avg, eps=eps)
                q_avg = torch.where(q_avg[:, 3:4] < 0, -q_avg, q_avg)
                
                return q_avg
            
            bs, len_q, _ = q.shape
            device = q.device
            
            # 1. 识别全零四元数
            zero_mask = is_zero_quaternion(q, eps)  # [bs, len]
            valid_mask = ~zero_mask  # [bs, len]，1表示有效四元数
            
            # 2. 计算有效四元数的平均旋转
            q_avg = quaternion_average_frechet_optimized(q, valid_mask, eps)  # [bs, 4]
            
            # 3. 创建目标四元数 [0.5, 0.5, 0.5, 0.5] 并规范化
            target_quat = torch.tensor([0, 0, 0, 1], device=device, dtype=q.dtype)
            target_quat = safe_normalize(target_quat)
            
            # 4. 将目标旋转整合到平均旋转中
            # 计算目标旋转与平均旋转的复合旋转
            # q_composite = target_quat * q_avg_conj
            # 这等价于先应用平均旋转的逆，然后应用目标旋转
            
            # 计算平均四元数的共轭（逆）
            q_avg_conj = q_avg.clone()
            q_avg_conj[:, :3] *= -1  # 虚部取反：[x,y,z,w] → [-x,-y,-z,w]
            
            # 将目标旋转与平均旋转的共轭相乘
            q_composite = quat_multiply(
                target_quat.unsqueeze(0).expand(bs, 4),  # 扩展目标四元数以匹配批次大小
                q_avg_conj
            )  # [bs, 4]
            
            q_composite = safe_normalize(q_composite)  # 确保复合旋转是单位四元数
            q_composite = q_composite.unsqueeze(1)  # [bs, 1, 4]，便于广播
            
            # 5. 应用复合旋转：q_final = q_composite * q
            q_final = quat_multiply(q_composite, q)  # [bs, len, 4]
            
            # 6. 保持原始全零位置为零
            q_final = q_final * valid_mask.unsqueeze(-1)  # [bs, len, 4]
            
            return q_final

        
        class MotionFeatureExtractor(nn.Module):
            """
            支持CUDA/CPU输入的运动特征提取模块
            输入: [bs, len, 7] 张量（[acc_x, acc_y, acc_z, rot_x, rot_y, rot_z, rot_w]）
            输出: [bs, len, 7] 张量（[线性加速度x/y/z, 角速度x/y/z, 角距离]）
            """
            def __init__(self, time_delta=1/200):
                super().__init__()
                self.time_delta = time_delta
                # ！修改1：不提前固定gravity_world的设备，改为使用时动态匹配输入设备
                self.gravity_world_val = torch.tensor([0.0, 0.0, 9.81], dtype=torch.float32)
                
            def quat_to_rot_matrix(self, quat):
                """四元数（[x,y,z,w]）转旋转矩阵 [bs, len, 3, 3]"""
                x, y, z, w = quat[..., 0], quat[..., 1], quat[..., 2], quat[..., 3]
                
                xx = x * x
                yy = y * y
                zz = z * z
                xy = x * y
                xz = x * z
                yz = y * z
                xw = x * w
                yw = y * w
                zw = z * w
                ww = w * w
                
                # 旋转矩阵计算（基于输入quat的设备，自动在CUDA/CPU上运算）
                rot_mat = torch.stack([
                    xx + ww - (yy + zz), 2 * (xy - zw), 2 * (xz + yw),
                    2 * (xy + zw), yy + ww - (xx + zz), 2 * (yz - xw),
                    2 * (xz - yw), 2 * (yz + xw), zz + ww - (xx + yy)
                ], dim=-1).view(*quat.shape[:-1], 3, 3)
                
                return rot_mat
            
            def rot_matrix_inverse(self, rot_mat):
                """旋转矩阵求逆（逆=转置，设备与输入一致）"""
                return rot_mat.transpose(-2, -1)
            
            def rot_matrix_multiply(self, rot_mat1, rot_mat2):
                """旋转矩阵乘法（设备自动匹配）"""
                return torch.matmul(rot_mat1, rot_mat2)
            
            def apply_rotation(self, rot_mat, vec):
                """旋转矩阵应用于向量（设备自动匹配）"""
                return torch.matmul(rot_mat, vec.unsqueeze(-1)).squeeze(-1)
            
            def remove_gravity_from_acc(self, acc_data, rot_data):
                """从加速度中移除重力（核心：设备动态匹配）"""
                bs, seq_len, _ = acc_data.shape
                device = acc_data.device  # ！修改2：获取输入设备（CUDA/CPU）
                
                valid_quats_normalized = rot_data / torch.norm(rot_data, dim=-1, keepdim=True).clamp(min=1e-8)
                rotations = self.quat_to_rot_matrix(valid_quats_normalized)
                
                # ！修改3：gravity_world动态匹配输入设备，避免CPU/CUDA不兼容
                gravity_world = self.gravity_world_val.to(device).expand(bs, seq_len, 3)
                # 重力向量从世界坐标系转到传感器坐标系
                gravity_sensor_frame = self.apply_rotation(
                    self.rot_matrix_inverse(rotations),
                    gravity_world
                )
                
                # 线性加速度计算（设备与输入一致）
                linear_accel = acc_data - gravity_sensor_frame
                return linear_accel
            
            def calculate_angular_velocity_from_quat(self, rot_data):
                """从四元数计算角速度（设备动态匹配）"""
                bs, seq_len, _ = rot_data.shape
                device = rot_data.device  # ！修改4：获取输入设备
                angular_vel = torch.zeros(bs, seq_len, 3, device=device)  # 零张量指定设备
                
                if seq_len < 2:
                    return angular_vel
                
                # 四元数有效性判断（torch.tensor(0.0)改为匹配设备）
                quat_norms = torch.norm(rot_data, dim=-1)
                # ！修改5：用rot_data.new_zeros(1)创建同设备的0张量，替代硬编码的torch.tensor(0.0)
                valid_mask = ~(torch.isnan(quat_norms) | torch.isclose(quat_norms, rot_data.new_zeros(1)))
                valid_pairs_mask = valid_mask[:, :-1] & valid_mask[:, 1:]
                
                if torch.any(valid_pairs_mask):
                    # 提取有效四元数对
                    q_t = rot_data[:, :-1][valid_pairs_mask]
                    q_t_plus_dt = rot_data[:, 1:][valid_pairs_mask]
                    
                    # 归一化
                    q_t_norms = quat_norms[:, :-1][valid_pairs_mask].unsqueeze(-1)
                    q_t_plus_dt_norms = quat_norms[:, 1:][valid_pairs_mask].unsqueeze(-1)
                    q_t_norm = q_t / q_t_norms.clamp(min=1e-8)
                    q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms.clamp(min=1e-8)
                    
                    # 旋转矩阵与相对旋转计算
                    rot_t = self.quat_to_rot_matrix(q_t_norm)
                    rot_t_plus_dt = self.quat_to_rot_matrix(q_t_plus_dt_norm)
                    delta_rot = self.rot_matrix_multiply(self.rot_matrix_inverse(rot_t), rot_t_plus_dt)
                    
                    # 罗德里格斯公式求旋转向量（设备自动匹配）
                    trace = delta_rot[..., 0, 0] + delta_rot[..., 1, 1] + delta_rot[..., 2, 2]
                    theta = torch.acos(torch.clamp((trace - 1) / 2, -1.0, 1.0))
                    sin_theta = torch.sin(theta)
                    
                    # 旋转向量计算（零张量指定设备）
                    rot_vec = torch.zeros_like(q_t_norm[..., :3], device=device)
                    # ！修改6：同修改5，避免设备不兼容
                    mask = ~torch.isclose(sin_theta, rot_data.new_zeros(1))
                    
                    if torch.any(mask):
                        factor = theta / (2 * sin_theta)
                        rot_vec[mask] = factor[mask, None] * torch.stack([
                            delta_rot[mask, 2, 1] - delta_rot[mask, 1, 2],
                            delta_rot[mask, 0, 2] - delta_rot[mask, 2, 0],
                            delta_rot[mask, 1, 0] - delta_rot[mask, 0, 1]
                        ], dim=1)
                    
                    # 角速度计算
                    angular_vel[:, :-1][valid_pairs_mask] = rot_vec / self.time_delta
                
                # 最后一个时间步复制前一个值
                if seq_len > 1:
                    angular_vel[:, -1] = angular_vel[:, -2]
                
                return angular_vel
            
            def calculate_angular_distance(self, rot_data, cumulative=False):
                """计算角距离（设备动态匹配）"""
                bs, seq_len, _ = rot_data.shape
                device = rot_data.device  # ！修改7：获取输入设备
                angular_dist = torch.zeros(bs, seq_len, 1, device=device)  # 零张量指定设备
                
                if seq_len < 2:
                    return angular_dist
                
                # 四元数归一化
                quat_norms = torch.norm(rot_data, dim=-1)
                q1 = rot_data[:, :-1]
                q2 = rot_data[:, 1:]
                q1_norms = quat_norms[:, :-1].unsqueeze(-1)
                q2_norms = quat_norms[:, 1:].unsqueeze(-1)
                q1_norm = q1 / q1_norms.clamp(min=1e-8)
                q2_norm = q2 / q2_norms.clamp(min=1e-8)
                
                # 相对旋转与角距离计算
                r1 = self.quat_to_rot_matrix(q1_norm)
                r2 = self.quat_to_rot_matrix(q2_norm)
                relative_rotation = self.rot_matrix_multiply(self.rot_matrix_inverse(r1), r2)
                
                trace = relative_rotation[..., 0, 0] + relative_rotation[..., 1, 1] + relative_rotation[..., 2, 2]
                theta = torch.acos(torch.clamp((trace - 1) / 2, -1.0, 1.0))
                angular_dist[:, :-1, 0] = theta
                
                # 最后一个时间步处理
                if seq_len > 1:
                    angular_dist[:, -1] = angular_dist[:, -2] if cumulative else 0.0
                
                # 累积角距离（设备一致）
                if cumulative:
                    angular_dist = torch.cumsum(angular_dist, dim=1)
                
                return angular_dist
            
            def forward(self, x):
                """前向传播（自动适配CUDA/CPU输入）"""
                # 分离加速度（前3维）和四元数（后4维）
                acc_data = x[..., :3]
                rot_data = x[..., 3:]
                
                # 计算三大特征（设备自动匹配输入）
                linear_accel = self.remove_gravity_from_acc(acc_data, rot_data)
                angular_vel = self.calculate_angular_velocity_from_quat(rot_data)
                angular_dist = self.calculate_angular_distance(rot_data)
                
                # 拼接输出（设备与输入一致）
                features = torch.cat([linear_accel, angular_vel, angular_dist], dim=-1)
                return features
        

        
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128,
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()

                self.gru = nn.GRU(out_channels, out_channels//2, batch_first=True, bidirectional=True)
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                
                out += shortcut
                
                out = out + self.gru(out.transpose(1, 2))[0].transpose(1, 2)
                out = self.act(out)

                out = self.act(out)
                
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context


        def get_gravity_torch(quat):
            g=9.81
             # 四元数旋转
            x = quat[..., 0:1]  # rot_x
            y = quat[..., 1:2]  # rot_y
            z = quat[..., 2:3]  # rot_z
            w = quat[..., 3:4]  # rot_w
            g_x = 2 * g * (x * z - w * y)
            g_y = 2 * g * (y * z + w * x)
            g_z = g * (w**2 - x**2 - y**2 + z**2)
            gravity = torch.concat([g_x, g_y, g_z], -1)
            return gravity

        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.imu_fe1 = ImuFeatureExtractor(**kwargs)
                self.imu_fe2 = ImuFeatureExtractor(**kwargs)
                self.imu_fe3 = ImuFeatureExtractor(**kwargs)
                
                imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7 

                self.new_feat_module = MotionFeatureExtractor()
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (5 + 3 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 160, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(160, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 160, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(160, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)

                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.emb_all = 256
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 512
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.LSTM(self.emb_all, self.dim_lstm, num_layers=2, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 512, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(512)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(512, 5)
                self.classifier = nn.Linear(512, n_classes * 4)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))

                self.scale5 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias5 = nn.Parameter(torch.zeros((1, 512, 1)))

                self.scale6 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias6 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan] # (batch, imu_dim, seq_len)
                imu2 = self.new_feat_module(imu)
                imu2 = imu2[:,:,:7]

                imu = imu.transpose(1, 2)
                imu2 = imu2.transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe1(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe2(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale5[:,:thm.shape[1],:]  + self.bias5[:,:thm.shape[1],:] ) * mask_all.transpose(1, 2)
                tof = (tof * self.scale6[:,:tof.shape[1],:]  + self.bias6[:,:tof.shape[1],:] ) * mask_all.transpose(1, 2)
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
                
                x1 = (x1 + x12 + thm + tof)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits.reshape(logits.shape[0], 4, 18).mean(1), logits.reshape(logits.shape[0], 4, 18).mean(2)
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()
        
        if np.any(norms < 1e-8):
            mask = norms < 1e-8
            tmp[mask] = [0.0, 0.0, 0.0, 0.0]
        
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
        
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':
                        X_ = np.concatenate([X, y2], -1)
        
                        X_ = transforms_custom(X_)
        
                        X = X_[:,:X.shape[-1]]
                        y2 = X_[:,X.shape[-1]:]
    
                        X[:,3:7] = X[:,3:7]/(np.sqrt((X[:,3:7]**2).sum(-1))+1e-6)[:,None]
            
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {
                            'Lie on Back': 0,
                            'Lie on Side - Non Dominant': 1,
                            'Seated Lean Non Dom - FACE DOWN': 2,
                            'Seated Straight': 3,
                           }
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])


        df['gesture_int'] = df['gesture_int'] + df['oid'] * 18

        
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(4 * len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 

            # return 1 - (logits.softmax(-1) * y).mean()
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        predorientationall = []
        targetsorientations = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA 
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)
            model2 = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)

            hidden_weights2 = [p for p in model2.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases2 = [p for p in model2.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups2 = [
                dict(params=hidden_weights2, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases2, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer2 = SingleDeviceMuonWithAuxAdam(param_groups2)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                model2.train()
                
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
                    X0 = X.clone()

                    X[BS//8:,:,7:] = 0.

                    if np.random.random() > 0.5:
                        X[:BS//8*2,:,3:7] = 0.
                    else:
                        X[:BS//8*2,:,:3] = 0.
                    
                    X, y = X.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    X0 = X0.float().to(device)
                    
                    optimizer.zero_grad()
                    optimizer2.zero_grad()
                    
                    LEN = 1200 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
                    
                    logits_, logits2_, logits3_ = model2(X0[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.5

                    clipval = 0.6
                    loss2 = clipped_cross_entropy(logits_, y, clipval, 18) 
                    loss2 += clipped_cross_entropy(logits2_, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss2 += clipped_cross_entropy(logits3_, y3, clipval, 5)  * 0.5

                    if epoch > 40:
                        loss = loss + loss2 + criterion_rdrop(logits, logits_)
                    else:
                        loss = loss + loss2 
                    
                    loss.backward()
                    
                    optimizer.step()
                    optimizer2.step()
                    
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                model2.eval()
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds_accmask, val_targets_accmask = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_rotmask, val_targets_rotmask = [], []

                val_preds_mother, val_targets_rotmask = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []

                val_preds_orientation, val_targets_orientation = [], []
                val_preds_orientation_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 7:] = 0.0

                        X_accmask = X[:].clone()
                        X_accmask[:, :, 7:] = 0.0
                        X_accmask[:, :, 0:3] = 0.0

                        X_rotmask = X[:].clone()
                        X_rotmask[:, :, 7:] = 0.0
                        X_rotmask[:, :, 3:7] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        X_accmask = X_accmask.float().to(device)
                        X_rotmask = X_rotmask.float().to(device)

                        logits_mother, _ = model(X)
                        
                        logits_rotmask, _  = model(X_rotmask)
                        logits_accmask, _  = model(X_accmask)
                        logits_imuonly, logits_orientation  = model(X_imuonly)

                        val_preds_mother.append(logits_mother.argmax(dim=1).cpu().numpy())
                        val_preds_accmask.append(logits_accmask.argmax(dim=1).cpu().numpy())
                        val_preds_rotmask.append(logits_rotmask.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_preds_logits.append(logits_mother.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_targets_imuonly.append(y.reshape(y.shape[0], 4, 18).sum(1).argmax(dim=1).cpu().numpy())


                        val_preds_orientation_logits.append(logits_orientation.cpu().numpy())
                        val_preds_orientation.append(logits_orientation.argmax(dim=1).cpu().numpy())
                        val_targets_orientation.append(y3.argmax(dim=1).cpu().numpy())
    
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc_rotmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_rotmask)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_accmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_accmask)]})
                    )
                    val_acc_mother = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_mother)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ROTMASK : ', round(val_acc_rotmask, 4),  '| ACCMASK : ', round(val_acc_accmask, 4),  '| MOTHER : ', round(val_acc_mother, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc_imuonly # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets_imuonly))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                predorientationall.append(np.concatenate(val_preds_orientation_logits, 0))
                targetsorientations.append(np.concatenate(val_targets_orientation))
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall),
                'pred_orientation' : np.concatenate(predorientationall, 0),
                'target_orientation' : np.concatenate(targetsorientations), 
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall), np.concatenate(predorientationall, 0), np.concatenate(targetsorientations)
        
        
    def get_new_features(self, df):
        feature_names = []
        
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = df[col] / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                # f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                # f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        # f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            outputs2 = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,7:] = 0.
            
            for model in self.models:
                model.eval()
                p, p2 = model(x)
                if outputs is None: 
                    outputs = p
                    outputs2 = p2
                else: 
                    outputs += p
                    outputs2 += p2
                    
            outputs /= len(self.models)
            outputs2 /= len(self.models)

        return self.gesture_classes, outputs.cpu().numpy()

In [71]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented

transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v24:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4,]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):        
        def get_gravity(quat):
            g=9.81
             # 四元数旋转
            x = quat[..., 0:1]  # rot_x
            y = quat[..., 1:2]  # rot_y
            z = quat[..., 2:3]  # rot_z
            w = quat[..., 3:4]  # rot_w
            g_x = 2 * g * (x * z - w * y)
            g_y = 2 * g * (y * z + w * x)
            g_z = g * (w**2 - x**2 - y**2 + z**2)
            gravity = np.concatenate([g_x, g_y, g_z], -1)
            return gravity
            
        def rotate_quaternion_np(quat: np.ndarray, angle_range: tuple = [(-10, 10), (-10, 10), (-10, 10), ]) -> np.ndarray:
            # 确保输入形状正确
            if quat.ndim != 2 or quat.shape[1] != 4:
                raise ValueError("输入四元数必须是形状为[len, 4]的数组")
            
            seq_len = quat.shape[0]
            
            # 1. 生成绕x、y、z轴的随机旋转角度（度）
            angle_x = np.random.uniform(angle_range[0][0], angle_range[0][1])
            angle_y = np.random.uniform(angle_range[1][0], angle_range[1][1])
            angle_z = np.random.uniform(angle_range[2][0], angle_range[2][1])
            
            # 2. 将角度转换为弧度
            rad_x = math.pi * angle_x / 180.0
            rad_y = math.pi * angle_y / 180.0
            rad_z = math.pi * angle_z / 180.0
            
            # 3. 生成绕各轴旋转的四元数
            qx = np.array([math.sin(rad_x/2), 0, 0, math.cos(rad_x/2)], dtype=np.float32)  # x轴旋转
            qy = np.array([0, math.sin(rad_y/2), 0, math.cos(rad_y/2)], dtype=np.float32)  # y轴旋转
            qz = np.array([0, 0, math.sin(rad_z/2), math.cos(rad_z/2)], dtype=np.float32)  # z轴旋转
            
            # 4. 组合旋转四元数 (q_total = qz * qy * qx，注意乘法顺序)
            q_zy = _quat_mul(qz, qy)
            q_total = _quat_mul(q_zy, qx)
            
            # 5. 将组合旋转应用到每个四元数上
            # 扩展旋转四元数以匹配序列长度
            q_total_expanded = np.tile(q_total, (seq_len, 1))  # [len, 4]
            rotated_quat = _quat_mul(q_total_expanded, quat)
            
            # 6. 归一化确保四元数有效性
            norms = np.linalg.norm(rotated_quat, axis=1, keepdims=True)
            rotated_quat = rotated_quat / np.maximum(norms, 1e-8)  # 防止除零
            
            return rotated_quat
        
        def _quat_mul(q1: np.ndarray, q2: np.ndarray) -> np.ndarray:
            # 提取分量（利用广播机制自动适配维度）
            x1, y1, z1, w1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
            x2, y2, z2, w2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
            
            # 四元数乘法公式（完全向量化）
            x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
            y = w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2
            z = w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2
            w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
            
            # 堆叠结果（保持维度）
            return np.stack([x, y, z, w], axis=-1).astype(q1.dtype)
        
        
        def remove_average_rotation_optimized(q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
            """
            从四元数序列中移除平均旋转，忽略全零四元数，并旋转到指定方向
            Args:
                q: 输入四元数序列，形状 [bs, len, 4]，顺序 [rot_x, rot_y, rot_z, rot_w]
                eps: 数值稳定性阈值
            Returns:
                q_final: 移除平均旋转并旋转到[0.5,0.5,0.5,0.5]方向后的四元数序列，形状 [bs, len, 4]
                         原始全零的位置仍为零
            """
            def is_zero_quaternion(q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
                """
                判断四元数是否为全零（或接近全零）
                Args:
                    q: 四元数，形状 [..., 4]
                    eps: 接近零的阈值
                Returns:
                    布尔张量，形状 [...], 指示每个四元数是否为零
                """
                return torch.norm(q, dim=-1) < eps
            
            def safe_normalize(q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
                """
                安全地规范化四元数，避免除零错误
                Args:
                    q: 四元数，形状 [..., 4]
                    eps: 最小范数阈值
                Returns:
                    规范化后的四元数
                """
                norm = torch.norm(q, dim=-1, keepdim=True)
                # 添加最小范数保护
                norm = torch.clamp(norm, min=eps)
                return q / norm
            
            def quat_multiply(q1: torch.Tensor, q2: torch.Tensor) -> torch.Tensor:
                """
                批量四元数乘法 (q1 * q2)
                Args:
                    q1: 形状 [..., 4]，四元数 [x1,y1,z1,w1]
                    q2: 形状 [..., 4]，四元数 [x2,y2,z2,w2]
                Returns:
                    q: 形状 [..., 4]，乘积 q1*q2 [x,y,z,w]
                """
                x1, y1, z1, w1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
                x2, y2, z2, w2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
                
                x = w1*x2 + x1*w2 + y1*z2 - z1*y2
                y = w1*y2 - x1*z2 + y1*w2 + z1*x2
                z = w1*z2 + x1*y2 - y1*x2 + z1*w2
                w = w1*w2 - x1*x2 - y1*y2 - z1*z2
                
                return torch.stack([x, y, z, w], dim=-1)
            
            def find_first_valid_indices(mask: torch.Tensor) -> torch.Tensor:
                """
                向量化方法查找每个batch中第一个有效四元数的索引
                Args:
                    mask: 有效掩码，形状 [bs, len]
                Returns:
                    indices: 每个batch的第一个有效索引，形状 [bs]
                """
                bs, seq_len = mask.shape
                device = mask.device
                
                # 创建索引矩阵
                indices = torch.arange(seq_len, device=device).expand(bs, seq_len)
                # 将无效位置的索引设置为大数
                indices = torch.where(mask, indices, seq_len)
                # 找到最小索引（第一个有效位置）
                first_valid = torch.min(indices, dim=1)[0]
                # 处理全无效的情况
                all_invalid = first_valid == seq_len
                first_valid[all_invalid] = 0  # 设置为0，后续会处理
                
                return first_valid
            
            def quaternion_average_frechet_optimized(q: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
                """
                计算批量四元数的Frechet均值（优化版本）
                Args:
                    q: 输入四元数，形状 [bs, len, 4]，顺序 [rot_x, rot_y, rot_z, rot_w]
                    mask: 有效四元数掩码，形状 [bs, len]，1表示有效，0表示无效
                    eps: 数值稳定性阈值
                Returns:
                    q_avg: 平均四元数，形状 [bs, 4]
                """
                bs, len_q, _ = q.shape
                device = q.device
                
                # 步骤1：安全地单位化四元数
                q_normalized = safe_normalize(q, eps=eps)
                
                # 应用掩码：无效四元数置为零
                mask_keepdim = mask.unsqueeze(-1)  # [bs, len, 1]
                q_normalized = q_normalized * mask_keepdim
                
                # 步骤2：向量化查找参考四元数
                first_valid_indices = find_first_valid_indices(mask)
                q_ref = q_normalized[torch.arange(bs, device=device), first_valid_indices].unsqueeze(1)  # [bs, 1, 4]
                
                # 步骤3：改进的符号统一策略（添加阈值避免在接近正交时翻转）
                dot = (q_normalized * q_ref).sum(dim=-1, keepdim=True)  # [bs, len, 1]
                q_unified = torch.where(dot < -0.01, -q_normalized, q_normalized)
                
                # 步骤4：构建协方差矩阵 M [bs, 4, 4]，考虑掩码
                q_reshaped = q_unified.unsqueeze(-1)  # [bs, len, 4, 1]
                q_transposed = q_reshaped.transpose(2, 3)  # [bs, len, 1, 4]
                outer_products = q_reshaped @ q_transposed  # [bs, len, 4, 4]
                
                # 应用掩码并求和
                mask_4d = mask.unsqueeze(-1).unsqueeze(-1)  # [bs, len, 1, 1]
                outer_products = outer_products * mask_4d
                M = outer_products.sum(dim=1)  # [bs, 4, 4]
                
                # 处理所有四元数都无效的情况
                valid_count = mask.sum(dim=1)  # [bs]
                all_invalid = valid_count == 0
                if all_invalid.any():
                    # 对全无效的batch，使用单位四元数作为平均
                    M[all_invalid] = torch.eye(4, device=device)
                
                # 步骤5：使用更稳定的特征值求解
                # 添加小单位矩阵增强数值稳定性
                M_stable = M + eps * torch.eye(4, device=device).unsqueeze(0)
                w, v = torch.linalg.eigh(M_stable)  # w: [bs,4], v: [bs,4,4]
                q_avg = v[:, :, -1]  # 最大特征值对应特征向量
                
                # 确保单位化 + 实部(rot_w)为正
                q_avg = safe_normalize(q_avg, eps=eps)
                q_avg = torch.where(q_avg[:, 3:4] < 0, -q_avg, q_avg)
                
                return q_avg
            
            bs, len_q, _ = q.shape
            device = q.device
            
            # 1. 识别全零四元数
            zero_mask = is_zero_quaternion(q, eps)  # [bs, len]
            valid_mask = ~zero_mask  # [bs, len]，1表示有效四元数
            
            # 2. 计算有效四元数的平均旋转
            q_avg = quaternion_average_frechet_optimized(q, valid_mask, eps)  # [bs, 4]
            
            # 3. 创建目标四元数 [0.5, 0.5, 0.5, 0.5] 并规范化
            target_quat = torch.tensor([0, 0, 0, 1], device=device, dtype=q.dtype)
            target_quat = safe_normalize(target_quat)
            
            # 4. 将目标旋转整合到平均旋转中
            # 计算目标旋转与平均旋转的复合旋转
            # q_composite = target_quat * q_avg_conj
            # 这等价于先应用平均旋转的逆，然后应用目标旋转
            
            # 计算平均四元数的共轭（逆）
            q_avg_conj = q_avg.clone()
            q_avg_conj[:, :3] *= -1  # 虚部取反：[x,y,z,w] → [-x,-y,-z,w]
            
            # 将目标旋转与平均旋转的共轭相乘
            q_composite = quat_multiply(
                target_quat.unsqueeze(0).expand(bs, 4),  # 扩展目标四元数以匹配批次大小
                q_avg_conj
            )  # [bs, 4]
            
            q_composite = safe_normalize(q_composite)  # 确保复合旋转是单位四元数
            q_composite = q_composite.unsqueeze(1)  # [bs, 1, 4]，便于广播
            
            # 5. 应用复合旋转：q_final = q_composite * q
            q_final = quat_multiply(q_composite, q)  # [bs, len, 4]
            
            # 6. 保持原始全零位置为零
            q_final = q_final * valid_mask.unsqueeze(-1)  # [bs, len, 4]
            
            return q_final

        
        class MotionFeatureExtractor(nn.Module):
            """
            支持CUDA/CPU输入的运动特征提取模块
            输入: [bs, len, 7] 张量（[acc_x, acc_y, acc_z, rot_x, rot_y, rot_z, rot_w]）
            输出: [bs, len, 7] 张量（[线性加速度x/y/z, 角速度x/y/z, 角距离]）
            """
            def __init__(self, time_delta=1/200):
                super().__init__()
                self.time_delta = time_delta
                # ！修改1：不提前固定gravity_world的设备，改为使用时动态匹配输入设备
                self.gravity_world_val = torch.tensor([0.0, 0.0, 9.81], dtype=torch.float32)
                
            def quat_to_rot_matrix(self, quat):
                """四元数（[x,y,z,w]）转旋转矩阵 [bs, len, 3, 3]"""
                x, y, z, w = quat[..., 0], quat[..., 1], quat[..., 2], quat[..., 3]
                
                xx = x * x
                yy = y * y
                zz = z * z
                xy = x * y
                xz = x * z
                yz = y * z
                xw = x * w
                yw = y * w
                zw = z * w
                ww = w * w
                
                # 旋转矩阵计算（基于输入quat的设备，自动在CUDA/CPU上运算）
                rot_mat = torch.stack([
                    xx + ww - (yy + zz), 2 * (xy - zw), 2 * (xz + yw),
                    2 * (xy + zw), yy + ww - (xx + zz), 2 * (yz - xw),
                    2 * (xz - yw), 2 * (yz + xw), zz + ww - (xx + yy)
                ], dim=-1).view(*quat.shape[:-1], 3, 3)
                
                return rot_mat
            
            def rot_matrix_inverse(self, rot_mat):
                """旋转矩阵求逆（逆=转置，设备与输入一致）"""
                return rot_mat.transpose(-2, -1)
            
            def rot_matrix_multiply(self, rot_mat1, rot_mat2):
                """旋转矩阵乘法（设备自动匹配）"""
                return torch.matmul(rot_mat1, rot_mat2)
            
            def apply_rotation(self, rot_mat, vec):
                """旋转矩阵应用于向量（设备自动匹配）"""
                return torch.matmul(rot_mat, vec.unsqueeze(-1)).squeeze(-1)
            
            def remove_gravity_from_acc(self, acc_data, rot_data):
                """从加速度中移除重力（核心：设备动态匹配）"""
                bs, seq_len, _ = acc_data.shape
                device = acc_data.device  # ！修改2：获取输入设备（CUDA/CPU）
                
                valid_quats_normalized = rot_data / torch.norm(rot_data, dim=-1, keepdim=True).clamp(min=1e-8)
                rotations = self.quat_to_rot_matrix(valid_quats_normalized)
                
                # ！修改3：gravity_world动态匹配输入设备，避免CPU/CUDA不兼容
                gravity_world = self.gravity_world_val.to(device).expand(bs, seq_len, 3)
                # 重力向量从世界坐标系转到传感器坐标系
                gravity_sensor_frame = self.apply_rotation(
                    self.rot_matrix_inverse(rotations),
                    gravity_world
                )
                
                # 线性加速度计算（设备与输入一致）
                linear_accel = acc_data - gravity_sensor_frame
                return linear_accel
            
            def calculate_angular_velocity_from_quat(self, rot_data):
                """从四元数计算角速度（设备动态匹配）"""
                bs, seq_len, _ = rot_data.shape
                device = rot_data.device  # ！修改4：获取输入设备
                angular_vel = torch.zeros(bs, seq_len, 3, device=device)  # 零张量指定设备
                
                if seq_len < 2:
                    return angular_vel
                
                # 四元数有效性判断（torch.tensor(0.0)改为匹配设备）
                quat_norms = torch.norm(rot_data, dim=-1)
                # ！修改5：用rot_data.new_zeros(1)创建同设备的0张量，替代硬编码的torch.tensor(0.0)
                valid_mask = ~(torch.isnan(quat_norms) | torch.isclose(quat_norms, rot_data.new_zeros(1)))
                valid_pairs_mask = valid_mask[:, :-1] & valid_mask[:, 1:]
                
                if torch.any(valid_pairs_mask):
                    # 提取有效四元数对
                    q_t = rot_data[:, :-1][valid_pairs_mask]
                    q_t_plus_dt = rot_data[:, 1:][valid_pairs_mask]
                    
                    # 归一化
                    q_t_norms = quat_norms[:, :-1][valid_pairs_mask].unsqueeze(-1)
                    q_t_plus_dt_norms = quat_norms[:, 1:][valid_pairs_mask].unsqueeze(-1)
                    q_t_norm = q_t / q_t_norms.clamp(min=1e-8)
                    q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms.clamp(min=1e-8)
                    
                    # 旋转矩阵与相对旋转计算
                    rot_t = self.quat_to_rot_matrix(q_t_norm)
                    rot_t_plus_dt = self.quat_to_rot_matrix(q_t_plus_dt_norm)
                    delta_rot = self.rot_matrix_multiply(self.rot_matrix_inverse(rot_t), rot_t_plus_dt)
                    
                    # 罗德里格斯公式求旋转向量（设备自动匹配）
                    trace = delta_rot[..., 0, 0] + delta_rot[..., 1, 1] + delta_rot[..., 2, 2]
                    theta = torch.acos(torch.clamp((trace - 1) / 2, -1.0, 1.0))
                    sin_theta = torch.sin(theta)
                    
                    # 旋转向量计算（零张量指定设备）
                    rot_vec = torch.zeros_like(q_t_norm[..., :3], device=device)
                    # ！修改6：同修改5，避免设备不兼容
                    mask = ~torch.isclose(sin_theta, rot_data.new_zeros(1))
                    
                    if torch.any(mask):
                        factor = theta / (2 * sin_theta)
                        rot_vec[mask] = factor[mask, None] * torch.stack([
                            delta_rot[mask, 2, 1] - delta_rot[mask, 1, 2],
                            delta_rot[mask, 0, 2] - delta_rot[mask, 2, 0],
                            delta_rot[mask, 1, 0] - delta_rot[mask, 0, 1]
                        ], dim=1)
                    
                    # 角速度计算
                    angular_vel[:, :-1][valid_pairs_mask] = rot_vec / self.time_delta
                
                # 最后一个时间步复制前一个值
                if seq_len > 1:
                    angular_vel[:, -1] = angular_vel[:, -2]
                
                return angular_vel
            
            def calculate_angular_distance(self, rot_data, cumulative=False):
                """计算角距离（设备动态匹配）"""
                bs, seq_len, _ = rot_data.shape
                device = rot_data.device  # ！修改7：获取输入设备
                angular_dist = torch.zeros(bs, seq_len, 1, device=device)  # 零张量指定设备
                
                if seq_len < 2:
                    return angular_dist
                
                # 四元数归一化
                quat_norms = torch.norm(rot_data, dim=-1)
                q1 = rot_data[:, :-1]
                q2 = rot_data[:, 1:]
                q1_norms = quat_norms[:, :-1].unsqueeze(-1)
                q2_norms = quat_norms[:, 1:].unsqueeze(-1)
                q1_norm = q1 / q1_norms.clamp(min=1e-8)
                q2_norm = q2 / q2_norms.clamp(min=1e-8)
                
                # 相对旋转与角距离计算
                r1 = self.quat_to_rot_matrix(q1_norm)
                r2 = self.quat_to_rot_matrix(q2_norm)
                relative_rotation = self.rot_matrix_multiply(self.rot_matrix_inverse(r1), r2)
                
                trace = relative_rotation[..., 0, 0] + relative_rotation[..., 1, 1] + relative_rotation[..., 2, 2]
                theta = torch.acos(torch.clamp((trace - 1) / 2, -1.0, 1.0))
                angular_dist[:, :-1, 0] = theta
                
                # 最后一个时间步处理
                if seq_len > 1:
                    angular_dist[:, -1] = angular_dist[:, -2] if cumulative else 0.0
                
                # 累积角距离（设备一致）
                if cumulative:
                    angular_dist = torch.cumsum(angular_dist, dim=1)
                
                return angular_dist
            
            def forward(self, x):
                """前向传播（自动适配CUDA/CPU输入）"""
                # 分离加速度（前3维）和四元数（后4维）
                acc_data = x[..., :3]
                rot_data = x[..., 3:]
                
                # 计算三大特征（设备自动匹配输入）
                linear_accel = self.remove_gravity_from_acc(acc_data, rot_data)
                angular_vel = self.calculate_angular_velocity_from_quat(rot_data)
                angular_dist = self.calculate_angular_distance(rot_data)
                
                # 拼接输出（设备与输入一致）
                features = torch.cat([linear_accel, angular_vel, angular_dist], dim=-1)
                return features
        

        
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128,
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()

                self.gru = nn.GRU(out_channels, out_channels//2, batch_first=True, bidirectional=True)
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                
                out += shortcut
                
                out = out + self.gru(out.transpose(1, 2))[0].transpose(1, 2)
                out = self.act(out)

                out = self.act(out)
                
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context


        def get_gravity_torch(quat):
            g=9.81
             # 四元数旋转
            x = quat[..., 0:1]  # rot_x
            y = quat[..., 1:2]  # rot_y
            z = quat[..., 2:3]  # rot_z
            w = quat[..., 3:4]  # rot_w
            g_x = 2 * g * (x * z - w * y)
            g_y = 2 * g * (y * z + w * x)
            g_z = g * (w**2 - x**2 - y**2 + z**2)
            gravity = torch.concat([g_x, g_y, g_z], -1)
            return gravity

        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.imu_fe1 = ImuFeatureExtractor(**kwargs)
                self.imu_fe2 = ImuFeatureExtractor(**kwargs)
                self.imu_fe3 = ImuFeatureExtractor(**kwargs)
                
                imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7 

                self.new_feat_module = MotionFeatureExtractor()
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (5 + 3 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 160, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(160, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 160, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(160, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)

                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.emb_all = 256
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 512
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.LSTM(self.emb_all, self.dim_lstm, num_layers=2, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 512, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(512)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(512, 5)
                self.classifier = nn.Linear(512, n_classes * 4)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))

                self.scale5 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias5 = nn.Parameter(torch.zeros((1, 512, 1)))

                self.scale6 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias6 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan] # (batch, imu_dim, seq_len)
                imu2 = self.new_feat_module(imu)
                imu2 = imu2[:,:,:7]

                imu = imu.transpose(1, 2)
                imu2 = imu2.transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe1(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe2(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale5[:,:thm.shape[1],:]  + self.bias5[:,:thm.shape[1],:] ) * mask_all.transpose(1, 2)
                tof = (tof * self.scale6[:,:tof.shape[1],:]  + self.bias6[:,:tof.shape[1],:] ) * mask_all.transpose(1, 2)
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
                
                x1 = (x1 + x12 + thm + tof)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits.reshape(logits.shape[0], 4, 18).mean(1), logits.reshape(logits.shape[0], 4, 18).mean(2)
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()
        
        if np.any(norms < 1e-8):
            mask = norms < 1e-8
            tmp[mask] = [0.0, 0.0, 0.0, 0.0]
        
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
        
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':
                        X_ = np.concatenate([X, y2], -1)
        
                        X_ = transforms_custom(X_)
        
                        X = X_[:,:X.shape[-1]]
                        y2 = X_[:,X.shape[-1]:]
    
                        X[:,3:7] = X[:,3:7]/(np.sqrt((X[:,3:7]**2).sum(-1))+1e-6)[:,None]
            
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {
                            'Lie on Back': 0,
                            'Lie on Side - Non Dominant': 1,
                            'Seated Lean Non Dom - FACE DOWN': 2,
                            'Seated Straight': 3,
                           }
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])


        df['gesture_int'] = df['gesture_int'] + df['oid'] * 18

        
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(4 * len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 

            # return 1 - (logits.softmax(-1) * y).mean()
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        predorientationall = []
        targetsorientations = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA 
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)
            model2 = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)

            hidden_weights2 = [p for p in model2.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases2 = [p for p in model2.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups2 = [
                dict(params=hidden_weights2, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases2, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer2 = SingleDeviceMuonWithAuxAdam(param_groups2)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                model2.train()
                
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
                    X0 = X.clone()

                    X[BS//8:,:,7:] = 0.

                    if np.random.random() > 0.5:
                        X[:BS//8*2,:,3:7] = 0.
                    else:
                        X[:BS//8*2,:,:3] = 0.
                    
                    X, y = X.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    X0 = X0.float().to(device)
                    
                    optimizer.zero_grad()
                    optimizer2.zero_grad()
                    
                    LEN = 1200 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
                    
                    logits_, logits2_, logits3_ = model2(X0[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.5

                    clipval = 0.6
                    loss2 = clipped_cross_entropy(logits_, y, clipval, 18) 
                    loss2 += clipped_cross_entropy(logits2_, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss2 += clipped_cross_entropy(logits3_, y3, clipval, 5)  * 0.5

                    if epoch > 40:
                        loss = loss + loss2 + criterion_rdrop(logits, logits_)
                    else:
                        loss = loss + loss2 
                    
                    loss.backward()
                    
                    optimizer.step()
                    optimizer2.step()
                    
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                model2.eval()
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds_accmask, val_targets_accmask = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_rotmask, val_targets_rotmask = [], []

                val_preds_mother, val_targets_rotmask = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []

                val_preds_orientation, val_targets_orientation = [], []
                val_preds_orientation_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 7:] = 0.0

                        X_accmask = X[:].clone()
                        X_accmask[:, :, 7:] = 0.0
                        X_accmask[:, :, 0:3] = 0.0

                        X_rotmask = X[:].clone()
                        X_rotmask[:, :, 7:] = 0.0
                        X_rotmask[:, :, 3:7] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        X_accmask = X_accmask.float().to(device)
                        X_rotmask = X_rotmask.float().to(device)

                        logits_mother, _ = model(X)
                        
                        logits_rotmask, _  = model(X_rotmask)
                        logits_accmask, _  = model(X_accmask)
                        logits_imuonly, logits_orientation  = model(X_imuonly)

                        val_preds_mother.append(logits_mother.argmax(dim=1).cpu().numpy())
                        val_preds_accmask.append(logits_accmask.argmax(dim=1).cpu().numpy())
                        val_preds_rotmask.append(logits_rotmask.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_preds_logits.append(logits_mother.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_targets_imuonly.append(y.reshape(y.shape[0], 4, 18).sum(1).argmax(dim=1).cpu().numpy())


                        val_preds_orientation_logits.append(logits_orientation.cpu().numpy())
                        val_preds_orientation.append(logits_orientation.argmax(dim=1).cpu().numpy())
                        val_targets_orientation.append(y3.argmax(dim=1).cpu().numpy())
    
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc_rotmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_rotmask)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_accmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_accmask)]})
                    )
                    val_acc_mother = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_mother)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ROTMASK : ', round(val_acc_rotmask, 4),  '| ACCMASK : ', round(val_acc_accmask, 4),  '| MOTHER : ', round(val_acc_mother, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc_imuonly # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets_imuonly))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                predorientationall.append(np.concatenate(val_preds_orientation_logits, 0))
                targetsorientations.append(np.concatenate(val_targets_orientation))
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall),
                'pred_orientation' : np.concatenate(predorientationall, 0),
                'target_orientation' : np.concatenate(targetsorientations), 
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall), np.concatenate(predorientationall, 0), np.concatenate(targetsorientations)
        
        
    def get_new_features(self, df):
        feature_names = []
        
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = df[col] / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                # f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                # f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        # f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            outputs2 = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,7:] = 0.
            
            for model in self.models:
                model.eval()
                p, p2 = model(x)
                if outputs is None: 
                    outputs = p
                    outputs2 = p2
                else: 
                    outputs += p
                    outputs2 += p2
                    
            outputs /= len(self.models)
            outputs2 /= len(self.models)

        return self.gesture_classes, (outputs.cpu().numpy(), outputs2.cpu().numpy())

In [72]:
model_zhou_inference_v24_1 = model_zhou_v24('/kaggle/input/cmi3v86/42', 42, './42')
if VALIDATION:
    MMM = model_zhou_inference_v24_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, ]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_24_1, pred_imuonly_24_1, targets, idlist_24_1, pred_orientation_24_1, target_orientation_24_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_24_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_24_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [73]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented

transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v23:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4,]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):        
        def get_gravity(quat):
            g=9.81
             # 四元数旋转
            x = quat[..., 0:1]  # rot_x
            y = quat[..., 1:2]  # rot_y
            z = quat[..., 2:3]  # rot_z
            w = quat[..., 3:4]  # rot_w
            g_x = 2 * g * (x * z - w * y)
            g_y = 2 * g * (y * z + w * x)
            g_z = g * (w**2 - x**2 - y**2 + z**2)
            gravity = np.concatenate([g_x, g_y, g_z], -1)
            return gravity
            
        def rotate_quaternion_np(quat: np.ndarray, angle_range: tuple = [(-10, 10), (-10, 10), (-10, 10), ]) -> np.ndarray:
            # 确保输入形状正确
            if quat.ndim != 2 or quat.shape[1] != 4:
                raise ValueError("输入四元数必须是形状为[len, 4]的数组")
            
            seq_len = quat.shape[0]
            
            # 1. 生成绕x、y、z轴的随机旋转角度（度）
            angle_x = np.random.uniform(angle_range[0][0], angle_range[0][1])
            angle_y = np.random.uniform(angle_range[1][0], angle_range[1][1])
            angle_z = np.random.uniform(angle_range[2][0], angle_range[2][1])
            
            # 2. 将角度转换为弧度
            rad_x = math.pi * angle_x / 180.0
            rad_y = math.pi * angle_y / 180.0
            rad_z = math.pi * angle_z / 180.0
            
            # 3. 生成绕各轴旋转的四元数
            qx = np.array([math.sin(rad_x/2), 0, 0, math.cos(rad_x/2)], dtype=np.float32)  # x轴旋转
            qy = np.array([0, math.sin(rad_y/2), 0, math.cos(rad_y/2)], dtype=np.float32)  # y轴旋转
            qz = np.array([0, 0, math.sin(rad_z/2), math.cos(rad_z/2)], dtype=np.float32)  # z轴旋转
            
            # 4. 组合旋转四元数 (q_total = qz * qy * qx，注意乘法顺序)
            q_zy = _quat_mul(qz, qy)
            q_total = _quat_mul(q_zy, qx)
            
            # 5. 将组合旋转应用到每个四元数上
            # 扩展旋转四元数以匹配序列长度
            q_total_expanded = np.tile(q_total, (seq_len, 1))  # [len, 4]
            rotated_quat = _quat_mul(q_total_expanded, quat)
            
            # 6. 归一化确保四元数有效性
            norms = np.linalg.norm(rotated_quat, axis=1, keepdims=True)
            rotated_quat = rotated_quat / np.maximum(norms, 1e-8)  # 防止除零
            
            return rotated_quat
        
        def _quat_mul(q1: np.ndarray, q2: np.ndarray) -> np.ndarray:
            # 提取分量（利用广播机制自动适配维度）
            x1, y1, z1, w1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
            x2, y2, z2, w2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
            
            # 四元数乘法公式（完全向量化）
            x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
            y = w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2
            z = w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2
            w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
            
            # 堆叠结果（保持维度）
            return np.stack([x, y, z, w], axis=-1).astype(q1.dtype)
        
        
        def remove_average_rotation_optimized(q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
            """
            从四元数序列中移除平均旋转，忽略全零四元数，并旋转到指定方向
            Args:
                q: 输入四元数序列，形状 [bs, len, 4]，顺序 [rot_x, rot_y, rot_z, rot_w]
                eps: 数值稳定性阈值
            Returns:
                q_final: 移除平均旋转并旋转到[0.5,0.5,0.5,0.5]方向后的四元数序列，形状 [bs, len, 4]
                         原始全零的位置仍为零
            """
            def is_zero_quaternion(q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
                """
                判断四元数是否为全零（或接近全零）
                Args:
                    q: 四元数，形状 [..., 4]
                    eps: 接近零的阈值
                Returns:
                    布尔张量，形状 [...], 指示每个四元数是否为零
                """
                return torch.norm(q, dim=-1) < eps
            
            def safe_normalize(q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
                """
                安全地规范化四元数，避免除零错误
                Args:
                    q: 四元数，形状 [..., 4]
                    eps: 最小范数阈值
                Returns:
                    规范化后的四元数
                """
                norm = torch.norm(q, dim=-1, keepdim=True)
                # 添加最小范数保护
                norm = torch.clamp(norm, min=eps)
                return q / norm
            
            def quat_multiply(q1: torch.Tensor, q2: torch.Tensor) -> torch.Tensor:
                """
                批量四元数乘法 (q1 * q2)
                Args:
                    q1: 形状 [..., 4]，四元数 [x1,y1,z1,w1]
                    q2: 形状 [..., 4]，四元数 [x2,y2,z2,w2]
                Returns:
                    q: 形状 [..., 4]，乘积 q1*q2 [x,y,z,w]
                """
                x1, y1, z1, w1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
                x2, y2, z2, w2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
                
                x = w1*x2 + x1*w2 + y1*z2 - z1*y2
                y = w1*y2 - x1*z2 + y1*w2 + z1*x2
                z = w1*z2 + x1*y2 - y1*x2 + z1*w2
                w = w1*w2 - x1*x2 - y1*y2 - z1*z2
                
                return torch.stack([x, y, z, w], dim=-1)
            
            def find_first_valid_indices(mask: torch.Tensor) -> torch.Tensor:
                """
                向量化方法查找每个batch中第一个有效四元数的索引
                Args:
                    mask: 有效掩码，形状 [bs, len]
                Returns:
                    indices: 每个batch的第一个有效索引，形状 [bs]
                """
                bs, seq_len = mask.shape
                device = mask.device
                
                # 创建索引矩阵
                indices = torch.arange(seq_len, device=device).expand(bs, seq_len)
                # 将无效位置的索引设置为大数
                indices = torch.where(mask, indices, seq_len)
                # 找到最小索引（第一个有效位置）
                first_valid = torch.min(indices, dim=1)[0]
                # 处理全无效的情况
                all_invalid = first_valid == seq_len
                first_valid[all_invalid] = 0  # 设置为0，后续会处理
                
                return first_valid
            
            def quaternion_average_frechet_optimized(q: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
                """
                计算批量四元数的Frechet均值（优化版本）
                Args:
                    q: 输入四元数，形状 [bs, len, 4]，顺序 [rot_x, rot_y, rot_z, rot_w]
                    mask: 有效四元数掩码，形状 [bs, len]，1表示有效，0表示无效
                    eps: 数值稳定性阈值
                Returns:
                    q_avg: 平均四元数，形状 [bs, 4]
                """
                bs, len_q, _ = q.shape
                device = q.device
                
                # 步骤1：安全地单位化四元数
                q_normalized = safe_normalize(q, eps=eps)
                
                # 应用掩码：无效四元数置为零
                mask_keepdim = mask.unsqueeze(-1)  # [bs, len, 1]
                q_normalized = q_normalized * mask_keepdim
                
                # 步骤2：向量化查找参考四元数
                first_valid_indices = find_first_valid_indices(mask)
                q_ref = q_normalized[torch.arange(bs, device=device), first_valid_indices].unsqueeze(1)  # [bs, 1, 4]
                
                # 步骤3：改进的符号统一策略（添加阈值避免在接近正交时翻转）
                dot = (q_normalized * q_ref).sum(dim=-1, keepdim=True)  # [bs, len, 1]
                q_unified = torch.where(dot < -0.01, -q_normalized, q_normalized)
                
                # 步骤4：构建协方差矩阵 M [bs, 4, 4]，考虑掩码
                q_reshaped = q_unified.unsqueeze(-1)  # [bs, len, 4, 1]
                q_transposed = q_reshaped.transpose(2, 3)  # [bs, len, 1, 4]
                outer_products = q_reshaped @ q_transposed  # [bs, len, 4, 4]
                
                # 应用掩码并求和
                mask_4d = mask.unsqueeze(-1).unsqueeze(-1)  # [bs, len, 1, 1]
                outer_products = outer_products * mask_4d
                M = outer_products.sum(dim=1)  # [bs, 4, 4]
                
                # 处理所有四元数都无效的情况
                valid_count = mask.sum(dim=1)  # [bs]
                all_invalid = valid_count == 0
                if all_invalid.any():
                    # 对全无效的batch，使用单位四元数作为平均
                    M[all_invalid] = torch.eye(4, device=device)
                
                # 步骤5：使用更稳定的特征值求解
                # 添加小单位矩阵增强数值稳定性
                M_stable = M + eps * torch.eye(4, device=device).unsqueeze(0)
                w, v = torch.linalg.eigh(M_stable)  # w: [bs,4], v: [bs,4,4]
                q_avg = v[:, :, -1]  # 最大特征值对应特征向量
                
                # 确保单位化 + 实部(rot_w)为正
                q_avg = safe_normalize(q_avg, eps=eps)
                q_avg = torch.where(q_avg[:, 3:4] < 0, -q_avg, q_avg)
                
                return q_avg
            
            bs, len_q, _ = q.shape
            device = q.device
            
            # 1. 识别全零四元数
            zero_mask = is_zero_quaternion(q, eps)  # [bs, len]
            valid_mask = ~zero_mask  # [bs, len]，1表示有效四元数
            
            # 2. 计算有效四元数的平均旋转
            q_avg = quaternion_average_frechet_optimized(q, valid_mask, eps)  # [bs, 4]
            
            # 3. 创建目标四元数 [0.5, 0.5, 0.5, 0.5] 并规范化
            target_quat = torch.tensor([0, 0, 0, 1], device=device, dtype=q.dtype)
            target_quat = safe_normalize(target_quat)
            
            # 4. 将目标旋转整合到平均旋转中
            # 计算目标旋转与平均旋转的复合旋转
            # q_composite = target_quat * q_avg_conj
            # 这等价于先应用平均旋转的逆，然后应用目标旋转
            
            # 计算平均四元数的共轭（逆）
            q_avg_conj = q_avg.clone()
            q_avg_conj[:, :3] *= -1  # 虚部取反：[x,y,z,w] → [-x,-y,-z,w]
            
            # 将目标旋转与平均旋转的共轭相乘
            q_composite = quat_multiply(
                target_quat.unsqueeze(0).expand(bs, 4),  # 扩展目标四元数以匹配批次大小
                q_avg_conj
            )  # [bs, 4]
            
            q_composite = safe_normalize(q_composite)  # 确保复合旋转是单位四元数
            q_composite = q_composite.unsqueeze(1)  # [bs, 1, 4]，便于广播
            
            # 5. 应用复合旋转：q_final = q_composite * q
            q_final = quat_multiply(q_composite, q)  # [bs, len, 4]
            
            # 6. 保持原始全零位置为零
            q_final = q_final * valid_mask.unsqueeze(-1)  # [bs, len, 4]
            
            return q_final

        
        class MotionFeatureExtractor(nn.Module):
            """
            支持CUDA/CPU输入的运动特征提取模块
            输入: [bs, len, 7] 张量（[acc_x, acc_y, acc_z, rot_x, rot_y, rot_z, rot_w]）
            输出: [bs, len, 7] 张量（[线性加速度x/y/z, 角速度x/y/z, 角距离]）
            """
            def __init__(self, time_delta=1/200):
                super().__init__()
                self.time_delta = time_delta
                # ！修改1：不提前固定gravity_world的设备，改为使用时动态匹配输入设备
                self.gravity_world_val = torch.tensor([0.0, 0.0, 9.81], dtype=torch.float32)
                
            def quat_to_rot_matrix(self, quat):
                """四元数（[x,y,z,w]）转旋转矩阵 [bs, len, 3, 3]"""
                x, y, z, w = quat[..., 0], quat[..., 1], quat[..., 2], quat[..., 3]
                
                xx = x * x
                yy = y * y
                zz = z * z
                xy = x * y
                xz = x * z
                yz = y * z
                xw = x * w
                yw = y * w
                zw = z * w
                ww = w * w
                
                # 旋转矩阵计算（基于输入quat的设备，自动在CUDA/CPU上运算）
                rot_mat = torch.stack([
                    xx + ww - (yy + zz), 2 * (xy - zw), 2 * (xz + yw),
                    2 * (xy + zw), yy + ww - (xx + zz), 2 * (yz - xw),
                    2 * (xz - yw), 2 * (yz + xw), zz + ww - (xx + yy)
                ], dim=-1).view(*quat.shape[:-1], 3, 3)
                
                return rot_mat
            
            def rot_matrix_inverse(self, rot_mat):
                """旋转矩阵求逆（逆=转置，设备与输入一致）"""
                return rot_mat.transpose(-2, -1)
            
            def rot_matrix_multiply(self, rot_mat1, rot_mat2):
                """旋转矩阵乘法（设备自动匹配）"""
                return torch.matmul(rot_mat1, rot_mat2)
            
            def apply_rotation(self, rot_mat, vec):
                """旋转矩阵应用于向量（设备自动匹配）"""
                return torch.matmul(rot_mat, vec.unsqueeze(-1)).squeeze(-1)
            
            def remove_gravity_from_acc(self, acc_data, rot_data):
                """从加速度中移除重力（核心：设备动态匹配）"""
                bs, seq_len, _ = acc_data.shape
                device = acc_data.device  # ！修改2：获取输入设备（CUDA/CPU）
                
                valid_quats_normalized = rot_data / torch.norm(rot_data, dim=-1, keepdim=True).clamp(min=1e-8)
                rotations = self.quat_to_rot_matrix(valid_quats_normalized)
                
                # ！修改3：gravity_world动态匹配输入设备，避免CPU/CUDA不兼容
                gravity_world = self.gravity_world_val.to(device).expand(bs, seq_len, 3)
                # 重力向量从世界坐标系转到传感器坐标系
                gravity_sensor_frame = self.apply_rotation(
                    self.rot_matrix_inverse(rotations),
                    gravity_world
                )
                
                # 线性加速度计算（设备与输入一致）
                linear_accel = acc_data - gravity_sensor_frame
                return linear_accel
            
            def calculate_angular_velocity_from_quat(self, rot_data):
                """从四元数计算角速度（设备动态匹配）"""
                bs, seq_len, _ = rot_data.shape
                device = rot_data.device  # ！修改4：获取输入设备
                angular_vel = torch.zeros(bs, seq_len, 3, device=device)  # 零张量指定设备
                
                if seq_len < 2:
                    return angular_vel
                
                # 四元数有效性判断（torch.tensor(0.0)改为匹配设备）
                quat_norms = torch.norm(rot_data, dim=-1)
                # ！修改5：用rot_data.new_zeros(1)创建同设备的0张量，替代硬编码的torch.tensor(0.0)
                valid_mask = ~(torch.isnan(quat_norms) | torch.isclose(quat_norms, rot_data.new_zeros(1)))
                valid_pairs_mask = valid_mask[:, :-1] & valid_mask[:, 1:]
                
                if torch.any(valid_pairs_mask):
                    # 提取有效四元数对
                    q_t = rot_data[:, :-1][valid_pairs_mask]
                    q_t_plus_dt = rot_data[:, 1:][valid_pairs_mask]
                    
                    # 归一化
                    q_t_norms = quat_norms[:, :-1][valid_pairs_mask].unsqueeze(-1)
                    q_t_plus_dt_norms = quat_norms[:, 1:][valid_pairs_mask].unsqueeze(-1)
                    q_t_norm = q_t / q_t_norms.clamp(min=1e-8)
                    q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms.clamp(min=1e-8)
                    
                    # 旋转矩阵与相对旋转计算
                    rot_t = self.quat_to_rot_matrix(q_t_norm)
                    rot_t_plus_dt = self.quat_to_rot_matrix(q_t_plus_dt_norm)
                    delta_rot = self.rot_matrix_multiply(self.rot_matrix_inverse(rot_t), rot_t_plus_dt)
                    
                    # 罗德里格斯公式求旋转向量（设备自动匹配）
                    trace = delta_rot[..., 0, 0] + delta_rot[..., 1, 1] + delta_rot[..., 2, 2]
                    theta = torch.acos(torch.clamp((trace - 1) / 2, -1.0, 1.0))
                    sin_theta = torch.sin(theta)
                    
                    # 旋转向量计算（零张量指定设备）
                    rot_vec = torch.zeros_like(q_t_norm[..., :3], device=device)
                    # ！修改6：同修改5，避免设备不兼容
                    mask = ~torch.isclose(sin_theta, rot_data.new_zeros(1))
                    
                    if torch.any(mask):
                        factor = theta / (2 * sin_theta)
                        rot_vec[mask] = factor[mask, None] * torch.stack([
                            delta_rot[mask, 2, 1] - delta_rot[mask, 1, 2],
                            delta_rot[mask, 0, 2] - delta_rot[mask, 2, 0],
                            delta_rot[mask, 1, 0] - delta_rot[mask, 0, 1]
                        ], dim=1)
                    
                    # 角速度计算
                    angular_vel[:, :-1][valid_pairs_mask] = rot_vec / self.time_delta
                
                # 最后一个时间步复制前一个值
                if seq_len > 1:
                    angular_vel[:, -1] = angular_vel[:, -2]
                
                return angular_vel
            
            def calculate_angular_distance(self, rot_data, cumulative=False):
                """计算角距离（设备动态匹配）"""
                bs, seq_len, _ = rot_data.shape
                device = rot_data.device  # ！修改7：获取输入设备
                angular_dist = torch.zeros(bs, seq_len, 1, device=device)  # 零张量指定设备
                
                if seq_len < 2:
                    return angular_dist
                
                # 四元数归一化
                quat_norms = torch.norm(rot_data, dim=-1)
                q1 = rot_data[:, :-1]
                q2 = rot_data[:, 1:]
                q1_norms = quat_norms[:, :-1].unsqueeze(-1)
                q2_norms = quat_norms[:, 1:].unsqueeze(-1)
                q1_norm = q1 / q1_norms.clamp(min=1e-8)
                q2_norm = q2 / q2_norms.clamp(min=1e-8)
                
                # 相对旋转与角距离计算
                r1 = self.quat_to_rot_matrix(q1_norm)
                r2 = self.quat_to_rot_matrix(q2_norm)
                relative_rotation = self.rot_matrix_multiply(self.rot_matrix_inverse(r1), r2)
                
                trace = relative_rotation[..., 0, 0] + relative_rotation[..., 1, 1] + relative_rotation[..., 2, 2]
                theta = torch.acos(torch.clamp((trace - 1) / 2, -1.0, 1.0))
                angular_dist[:, :-1, 0] = theta
                
                # 最后一个时间步处理
                if seq_len > 1:
                    angular_dist[:, -1] = angular_dist[:, -2] if cumulative else 0.0
                
                # 累积角距离（设备一致）
                if cumulative:
                    angular_dist = torch.cumsum(angular_dist, dim=1)
                
                return angular_dist
            
            def forward(self, x):
                """前向传播（自动适配CUDA/CPU输入）"""
                # 分离加速度（前3维）和四元数（后4维）
                acc_data = x[..., :3]
                rot_data = x[..., 3:]
                
                # 计算三大特征（设备自动匹配输入）
                linear_accel = self.remove_gravity_from_acc(acc_data, rot_data)
                angular_vel = self.calculate_angular_velocity_from_quat(rot_data)
                angular_dist = self.calculate_angular_distance(rot_data)
                
                # 拼接输出（设备与输入一致）
                features = torch.cat([linear_accel, angular_vel, angular_dist], dim=-1)
                return features
        

        
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128,
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()

                self.gru = nn.GRU(out_channels, out_channels//2, batch_first=True, bidirectional=True)
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                
                out += shortcut
                
                out = out + self.gru(out.transpose(1, 2))[0].transpose(1, 2)
                out = self.act(out)

                out = self.act(out)
                
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context


        def get_gravity_torch(quat):
            g=9.81
             # 四元数旋转
            x = quat[..., 0:1]  # rot_x
            y = quat[..., 1:2]  # rot_y
            z = quat[..., 2:3]  # rot_z
            w = quat[..., 3:4]  # rot_w
            g_x = 2 * g * (x * z - w * y)
            g_y = 2 * g * (y * z + w * x)
            g_z = g * (w**2 - x**2 - y**2 + z**2)
            gravity = torch.concat([g_x, g_y, g_z], -1)
            return gravity

        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.imu_fe1 = ImuFeatureExtractor(**kwargs)
                self.imu_fe2 = ImuFeatureExtractor(**kwargs)
                self.imu_fe3 = ImuFeatureExtractor(**kwargs)
                
                imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7 

                self.new_feat_module = MotionFeatureExtractor()
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (5 + 3 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 160, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(160, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 160, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(160, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)

                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.emb_all = 256
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 512
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.LSTM(self.emb_all, self.dim_lstm, num_layers=2, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 512, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(512)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(512, 5)
                self.classifier = nn.Linear(512, n_classes * 4)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))

                self.scale5 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias5 = nn.Parameter(torch.zeros((1, 512, 1)))

                self.scale6 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias6 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan] # (batch, imu_dim, seq_len)
                imu2 = self.new_feat_module(imu)
                imu2 = imu2[:,:,:7]

                imu = imu.transpose(1, 2)
                imu2 = imu2.transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe1(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe2(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale5[:,:thm.shape[1],:]  + self.bias5[:,:thm.shape[1],:] ) * mask_all.transpose(1, 2)
                tof = (tof * self.scale6[:,:tof.shape[1],:]  + self.bias6[:,:tof.shape[1],:] ) * mask_all.transpose(1, 2)
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
                
                x1 = (x1 + x12 + thm + tof)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits.reshape(logits.shape[0], 4, 18).mean(1), logits.reshape(logits.shape[0], 4, 18).mean(2)
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()
        
        if np.any(norms < 1e-8):
            mask = norms < 1e-8
            tmp[mask] = [0.0, 0.0, 0.0, 0.0]
        
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
        
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':
                        X_ = np.concatenate([X, y2], -1)
        
                        X_ = transforms_custom(X_)
        
                        X = X_[:,:X.shape[-1]]
                        y2 = X_[:,X.shape[-1]:]
    
                        X[:,3:7] = X[:,3:7]/(np.sqrt((X[:,3:7]**2).sum(-1))+1e-6)[:,None]
            
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {
                            'Lie on Back': 0,
                            'Lie on Side - Non Dominant': 1,
                            'Seated Lean Non Dom - FACE DOWN': 2,
                            'Seated Straight': 3,
                           }
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])


        df['gesture_int'] = df['gesture_int'] + df['oid'] * 18

        
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(4 * len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 

            # return 1 - (logits.softmax(-1) * y).mean()
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        predorientationall = []
        targetsorientations = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA 
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)
            model2 = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)

            hidden_weights2 = [p for p in model2.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases2 = [p for p in model2.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups2 = [
                dict(params=hidden_weights2, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases2, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer2 = SingleDeviceMuonWithAuxAdam(param_groups2)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                model2.train()
                
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
                    X0 = X.clone()

                    X[BS//8:,:,7:] = 0.

                    if np.random.random() > 0.5:
                        X[:BS//8*2,:,3:7] = 0.
                    else:
                        X[:BS//8*2,:,:3] = 0.
                    
                    X, y = X.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    X0 = X0.float().to(device)
                    
                    optimizer.zero_grad()
                    optimizer2.zero_grad()
                    
                    LEN = 1200 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
                    
                    logits_, logits2_, logits3_ = model2(X0[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.5

                    clipval = 0.6
                    loss2 = clipped_cross_entropy(logits_, y, clipval, 18) 
                    loss2 += clipped_cross_entropy(logits2_, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss2 += clipped_cross_entropy(logits3_, y3, clipval, 5)  * 0.5

                    if epoch > 40:
                        loss = loss + loss2 + criterion_rdrop(logits, logits_)
                    else:
                        loss = loss + loss2 
                    
                    loss.backward()
                    
                    optimizer.step()
                    optimizer2.step()
                    
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                model2.eval()
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds_accmask, val_targets_accmask = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_rotmask, val_targets_rotmask = [], []

                val_preds_mother, val_targets_rotmask = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []

                val_preds_orientation, val_targets_orientation = [], []
                val_preds_orientation_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 7:] = 0.0

                        X_accmask = X[:].clone()
                        X_accmask[:, :, 7:] = 0.0
                        X_accmask[:, :, 0:3] = 0.0

                        X_rotmask = X[:].clone()
                        X_rotmask[:, :, 7:] = 0.0
                        X_rotmask[:, :, 3:7] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        X_accmask = X_accmask.float().to(device)
                        X_rotmask = X_rotmask.float().to(device)

                        logits_mother, _ = model(X)
                        
                        logits_rotmask, _  = model(X_rotmask)
                        logits_accmask, _  = model(X_accmask)
                        logits_imuonly, logits_orientation  = model(X_imuonly)

                        val_preds_mother.append(logits_mother.argmax(dim=1).cpu().numpy())
                        val_preds_accmask.append(logits_accmask.argmax(dim=1).cpu().numpy())
                        val_preds_rotmask.append(logits_rotmask.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_preds_logits.append(logits_mother.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_targets_imuonly.append(y.reshape(y.shape[0], 4, 18).sum(1).argmax(dim=1).cpu().numpy())


                        val_preds_orientation_logits.append(logits_orientation.cpu().numpy())
                        val_preds_orientation.append(logits_orientation.argmax(dim=1).cpu().numpy())
                        val_targets_orientation.append(y3.argmax(dim=1).cpu().numpy())
    
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc_rotmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_rotmask)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_accmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_accmask)]})
                    )
                    val_acc_mother = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_mother)]})
                    )

                        
                    train_loss = np.mean(train_loss)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ROTMASK : ', round(val_acc_rotmask, 4),  '| ACCMASK : ', round(val_acc_accmask, 4),  '| MOTHER : ', round(val_acc_mother, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc_imuonly # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets_imuonly))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                predorientationall.append(np.concatenate(val_preds_orientation_logits, 0))
                targetsorientations.append(np.concatenate(val_targets_orientation))
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall),
                'pred_orientation' : np.concatenate(predorientationall, 0),
                'target_orientation' : np.concatenate(targetsorientations), 
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall), np.concatenate(predorientationall, 0), np.concatenate(targetsorientations)
        
    def get_new_features(self, df):
        feature_names = []
        
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = df[col] / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                # f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                # f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        # f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            outputs2 = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,7:] = 0.
            
            for model in self.models:
                model.eval()
                p, p2 = model(x)
                if outputs is None: 
                    outputs = p
                    outputs2 = p2
                else: 
                    outputs += p
                    outputs2 += p2
                    
            outputs /= len(self.models)
            outputs2 /= len(self.models)

        return self.gesture_classes, outputs.cpu().numpy()

In [74]:
import os, json, joblib, numpy as np, pandas as pd
import random, math
from pathlib import Path
import warnings 
warnings.filterwarnings("ignore")

import sys
sys.path.append('/root/autodl-tmp/')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from timm.scheduler import CosineLRScheduler
from scipy.signal import firwin
import polars as pl
import numpy as np
import random


import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation as R
from tqdm.auto import tqdm 


class SignalTransform:
    def __init__(self, always_apply: bool = False, p: float = 0.5):
        self.always_apply = always_apply
        self.p = p

    def __call__(self, y: np.ndarray):
        if self.always_apply:
            return self.apply(y)
        else:
            if np.random.rand() < self.p:
                return self.apply(y)
            else:
                return y

    def apply(self, y: np.ndarray):
        raise NotImplementedError


class Compose:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        for trns in self.transforms:
            y = trns(y)
        return y


class OneOf:
    def __init__(self, transforms: list):
        self.transforms = transforms

    def __call__(self, y: np.ndarray):
        n_trns = len(self.transforms)
        trns_idx = np.random.choice(n_trns)
        trns = self.transforms[trns_idx]
        return trns(y)

class TimeStretch(SignalTransform):
    def __init__(self, max_rate=1.5, min_rate=0.5, always_apply=False, p=0.5):
        super().__init__(always_apply, p)
        self.max_rate = max_rate
        self.min_rate = min_rate
        self.always_apply = always_apply
        self.p = p

    def apply(self, x: np.ndarray):
        """
        Stretch a 1D or 2D array in time using linear interpolation.
        After stretching, pad or crop at the beginning to match the original length.
        Padding value is 0.
        - x: np.ndarray of shape (L,) or (L, N)
        """
        rate = np.random.uniform(self.min_rate, self.max_rate)
        L = x.shape[0]
        L_new = int(L / rate)
        orig_idx = np.linspace(0, L - 1, num=L)
        new_idx = np.linspace(0, L - 1, num=L_new)

        if x.ndim == 1:
            stretched = np.interp(new_idx, orig_idx, x)
            # Pad or crop at the beginning
            if L_new < L:
                # Pad at the beginning
                padded = np.zeros(L, dtype=stretched.dtype)
                padded[-L_new:] = stretched
                return padded
            elif L_new > L:
                # Crop at the beginning
                return stretched[-L:]
            else:
                return stretched
        elif x.ndim == 2:
            stretched = np.stack([
                np.interp(new_idx, orig_idx, x[:, i]) for i in range(x.shape[1])
            ], axis=1)
            if L_new < L:
                padded = np.zeros((L, x.shape[1]), dtype=stretched.dtype)
                padded[-L_new:, :] = stretched
                return padded
            elif L_new > L:
                return stretched[-L:, :]
            else:
                return stretched
        else:
            raise ValueError("Only 1D or 2D arrays are supported.")


class TimeShift(SignalTransform):
    def __init__(self, always_apply=False, p=0.5, max_shift_pct=0.25, padding_mode="replace"):
        super().__init__(always_apply, p)
        
        assert 0 <= max_shift_pct <= 1.0, "`max_shift_pct` must be between 0 and 1"
        assert padding_mode in ["replace", "zero"], "`padding_mode` must be either 'replace' or 'zero'"
        
        self.max_shift_pct = max_shift_pct
        self.padding_mode = padding_mode

    def apply(self, x: np.ndarray, **params):
        assert x.ndim == 2, "`x` must be a 2D array with shape (L, N)"
        
        L = x.shape[0]
        max_shift = int(L * self.max_shift_pct)
        shift = np.random.randint(-max_shift, max_shift + 1)

        # Roll along time axis (axis=0)
        augmented = np.roll(x, shift, axis=0)

        if self.padding_mode == "zero":
            if shift > 0:
                augmented[:shift, :] = 0
            elif shift < 0:
                augmented[shift:, :] = 0

        return augmented

transforms_custom = Compose([
    TimeShift(p=0.25, padding_mode="replace", max_shift_pct=0.25),
    TimeStretch(p=0.25, max_rate=1.5, min_rate=0.5),
])



if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
    test_path1 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv'
    test_path2 = '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv'
else:
    if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
        test_path1 = './cmi-detect-behavior-with-sensor-data/test.csv'
        test_path2 = './cmi-detect-behavior-with-sensor-data/test_demographics.csv'
    else:
        test_path1 = './test.csv'
        test_path2 = './test_demographics.csv'

class model_zhou_v26:
    def __init__(self, kaggle_input_path="/kaggle/input/cmi3v23", seed=42, save_path='./'):
        self.models = []

        import os
        if os.path.exists("../input/cmi-detect-behavior-with-sensor-data"):
            self.TRAIN = False                     
            self.RAW_DIR = Path("../input/cmi-detect-behavior-with-sensor-data")
            self.PRETRAINED_DIR = Path(kaggle_input_path) 
            self.EXPORT_DIR = Path(save_path)                                   
        else:
            if os.path.exists("./cmi-detect-behavior-with-sensor-data/"):
                self.TRAIN = True                     
                self.RAW_DIR = Path("./cmi-detect-behavior-with-sensor-data/")
                self.PRETRAINED_DIR = Path(kaggle_input_path) 
                self.EXPORT_DIR = Path(save_path)                                  
            else:
                self.TRAIN = True                    
                self.RAW_DIR = Path("./")
                self.EXPORT_DIR = Path(save_path)
                self.PRETRAINED_DIR = Path(kaggle_input_path) 

        if not os.path.exists(self.EXPORT_DIR):
            os.system(f'mkdir {self.EXPORT_DIR}')
            
        self.VALIDATION = False
        self.SEED = seed

        self.BATCH_SIZE = 64 * 1
        self.PAD_PERCENTILE = 128 + 64
        self.maxlen = self.PAD_PERCENTILE
        self.LR_INIT = 1e-3
        self.WD = 2e-1
        
        self.PATIENCE = 40
        self.random_state = self.SEED
        
        self.tof_mode = 4
        
        self.FOLDS = 5
        self.TRAIN_FOLDS = [0, 1, 2, 3, 4,]
        
        self.EPOCHS = 160

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"▶ imports ready · pytorch {torch.__version__} · device: {self.device}")
    
    def create_model(self, param_a, param_b, param_c, param_d):        
        def get_gravity(quat):
            g=9.81
             # 四元数旋转
            x = quat[..., 0:1]  # rot_x
            y = quat[..., 1:2]  # rot_y
            z = quat[..., 2:3]  # rot_z
            w = quat[..., 3:4]  # rot_w
            g_x = 2 * g * (x * z - w * y)
            g_y = 2 * g * (y * z + w * x)
            g_z = g * (w**2 - x**2 - y**2 + z**2)
            gravity = np.concatenate([g_x, g_y, g_z], -1)
            return gravity
            
        def rotate_quaternion_np(quat: np.ndarray, angle_range: tuple = [(-10, 10), (-10, 10), (-10, 10), ]) -> np.ndarray:
            # 确保输入形状正确
            if quat.ndim != 2 or quat.shape[1] != 4:
                raise ValueError("输入四元数必须是形状为[len, 4]的数组")
            
            seq_len = quat.shape[0]
            
            # 1. 生成绕x、y、z轴的随机旋转角度（度）
            angle_x = np.random.uniform(angle_range[0][0], angle_range[0][1])
            angle_y = np.random.uniform(angle_range[1][0], angle_range[1][1])
            angle_z = np.random.uniform(angle_range[2][0], angle_range[2][1])
            
            # 2. 将角度转换为弧度
            rad_x = math.pi * angle_x / 180.0
            rad_y = math.pi * angle_y / 180.0
            rad_z = math.pi * angle_z / 180.0
            
            # 3. 生成绕各轴旋转的四元数
            qx = np.array([math.sin(rad_x/2), 0, 0, math.cos(rad_x/2)], dtype=np.float32)  # x轴旋转
            qy = np.array([0, math.sin(rad_y/2), 0, math.cos(rad_y/2)], dtype=np.float32)  # y轴旋转
            qz = np.array([0, 0, math.sin(rad_z/2), math.cos(rad_z/2)], dtype=np.float32)  # z轴旋转
            
            # 4. 组合旋转四元数 (q_total = qz * qy * qx，注意乘法顺序)
            q_zy = _quat_mul(qz, qy)
            q_total = _quat_mul(q_zy, qx)
            
            # 5. 将组合旋转应用到每个四元数上
            # 扩展旋转四元数以匹配序列长度
            q_total_expanded = np.tile(q_total, (seq_len, 1))  # [len, 4]
            rotated_quat = _quat_mul(q_total_expanded, quat)
            
            # 6. 归一化确保四元数有效性
            norms = np.linalg.norm(rotated_quat, axis=1, keepdims=True)
            rotated_quat = rotated_quat / np.maximum(norms, 1e-8)  # 防止除零
            
            return rotated_quat
        
        def _quat_mul(q1: np.ndarray, q2: np.ndarray) -> np.ndarray:
            # 提取分量（利用广播机制自动适配维度）
            x1, y1, z1, w1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
            x2, y2, z2, w2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
            
            # 四元数乘法公式（完全向量化）
            x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2
            y = w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2
            z = w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2
            w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2
            
            # 堆叠结果（保持维度）
            return np.stack([x, y, z, w], axis=-1).astype(q1.dtype)
        
        
        def remove_average_rotation_optimized(q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
            """
            从四元数序列中移除平均旋转，忽略全零四元数，并旋转到指定方向
            Args:
                q: 输入四元数序列，形状 [bs, len, 4]，顺序 [rot_x, rot_y, rot_z, rot_w]
                eps: 数值稳定性阈值
            Returns:
                q_final: 移除平均旋转并旋转到[0.5,0.5,0.5,0.5]方向后的四元数序列，形状 [bs, len, 4]
                         原始全零的位置仍为零
            """
            def is_zero_quaternion(q: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
                """
                判断四元数是否为全零（或接近全零）
                Args:
                    q: 四元数，形状 [..., 4]
                    eps: 接近零的阈值
                Returns:
                    布尔张量，形状 [...], 指示每个四元数是否为零
                """
                return torch.norm(q, dim=-1) < eps
            
            def safe_normalize(q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
                """
                安全地规范化四元数，避免除零错误
                Args:
                    q: 四元数，形状 [..., 4]
                    eps: 最小范数阈值
                Returns:
                    规范化后的四元数
                """
                norm = torch.norm(q, dim=-1, keepdim=True)
                # 添加最小范数保护
                norm = torch.clamp(norm, min=eps)
                return q / norm
            
            def quat_multiply(q1: torch.Tensor, q2: torch.Tensor) -> torch.Tensor:
                """
                批量四元数乘法 (q1 * q2)
                Args:
                    q1: 形状 [..., 4]，四元数 [x1,y1,z1,w1]
                    q2: 形状 [..., 4]，四元数 [x2,y2,z2,w2]
                Returns:
                    q: 形状 [..., 4]，乘积 q1*q2 [x,y,z,w]
                """
                x1, y1, z1, w1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
                x2, y2, z2, w2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]
                
                x = w1*x2 + x1*w2 + y1*z2 - z1*y2
                y = w1*y2 - x1*z2 + y1*w2 + z1*x2
                z = w1*z2 + x1*y2 - y1*x2 + z1*w2
                w = w1*w2 - x1*x2 - y1*y2 - z1*z2
                
                return torch.stack([x, y, z, w], dim=-1)
            
            def find_first_valid_indices(mask: torch.Tensor) -> torch.Tensor:
                """
                向量化方法查找每个batch中第一个有效四元数的索引
                Args:
                    mask: 有效掩码，形状 [bs, len]
                Returns:
                    indices: 每个batch的第一个有效索引，形状 [bs]
                """
                bs, seq_len = mask.shape
                device = mask.device
                
                # 创建索引矩阵
                indices = torch.arange(seq_len, device=device).expand(bs, seq_len)
                # 将无效位置的索引设置为大数
                indices = torch.where(mask, indices, seq_len)
                # 找到最小索引（第一个有效位置）
                first_valid = torch.min(indices, dim=1)[0]
                # 处理全无效的情况
                all_invalid = first_valid == seq_len
                first_valid[all_invalid] = 0  # 设置为0，后续会处理
                
                return first_valid
            
            def quaternion_average_frechet_optimized(q: torch.Tensor, mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
                """
                计算批量四元数的Frechet均值（优化版本）
                Args:
                    q: 输入四元数，形状 [bs, len, 4]，顺序 [rot_x, rot_y, rot_z, rot_w]
                    mask: 有效四元数掩码，形状 [bs, len]，1表示有效，0表示无效
                    eps: 数值稳定性阈值
                Returns:
                    q_avg: 平均四元数，形状 [bs, 4]
                """
                bs, len_q, _ = q.shape
                device = q.device
                
                # 步骤1：安全地单位化四元数
                q_normalized = safe_normalize(q, eps=eps)
                
                # 应用掩码：无效四元数置为零
                mask_keepdim = mask.unsqueeze(-1)  # [bs, len, 1]
                q_normalized = q_normalized * mask_keepdim
                
                # 步骤2：向量化查找参考四元数
                first_valid_indices = find_first_valid_indices(mask)
                q_ref = q_normalized[torch.arange(bs, device=device), first_valid_indices].unsqueeze(1)  # [bs, 1, 4]
                
                # 步骤3：改进的符号统一策略（添加阈值避免在接近正交时翻转）
                dot = (q_normalized * q_ref).sum(dim=-1, keepdim=True)  # [bs, len, 1]
                q_unified = torch.where(dot < -0.01, -q_normalized, q_normalized)
                
                # 步骤4：构建协方差矩阵 M [bs, 4, 4]，考虑掩码
                q_reshaped = q_unified.unsqueeze(-1)  # [bs, len, 4, 1]
                q_transposed = q_reshaped.transpose(2, 3)  # [bs, len, 1, 4]
                outer_products = q_reshaped @ q_transposed  # [bs, len, 4, 4]
                
                # 应用掩码并求和
                mask_4d = mask.unsqueeze(-1).unsqueeze(-1)  # [bs, len, 1, 1]
                outer_products = outer_products * mask_4d
                M = outer_products.sum(dim=1)  # [bs, 4, 4]
                
                # 处理所有四元数都无效的情况
                valid_count = mask.sum(dim=1)  # [bs]
                all_invalid = valid_count == 0
                if all_invalid.any():
                    # 对全无效的batch，使用单位四元数作为平均
                    M[all_invalid] = torch.eye(4, device=device)
                
                # 步骤5：使用更稳定的特征值求解
                # 添加小单位矩阵增强数值稳定性
                M_stable = M + eps * torch.eye(4, device=device).unsqueeze(0)
                w, v = torch.linalg.eigh(M_stable)  # w: [bs,4], v: [bs,4,4]
                q_avg = v[:, :, -1]  # 最大特征值对应特征向量
                
                # 确保单位化 + 实部(rot_w)为正
                q_avg = safe_normalize(q_avg, eps=eps)
                q_avg = torch.where(q_avg[:, 3:4] < 0, -q_avg, q_avg)
                
                return q_avg
            
            bs, len_q, _ = q.shape
            device = q.device
            
            # 1. 识别全零四元数
            zero_mask = is_zero_quaternion(q, eps)  # [bs, len]
            valid_mask = ~zero_mask  # [bs, len]，1表示有效四元数
            
            # 2. 计算有效四元数的平均旋转
            q_avg = quaternion_average_frechet_optimized(q, valid_mask, eps)  # [bs, 4]
            
            # 3. 创建目标四元数 [0.5, 0.5, 0.5, 0.5] 并规范化
            target_quat = torch.tensor([0, 0, 0, 1], device=device, dtype=q.dtype)
            target_quat = safe_normalize(target_quat)
            
            # 4. 将目标旋转整合到平均旋转中
            # 计算目标旋转与平均旋转的复合旋转
            # q_composite = target_quat * q_avg_conj
            # 这等价于先应用平均旋转的逆，然后应用目标旋转
            
            # 计算平均四元数的共轭（逆）
            q_avg_conj = q_avg.clone()
            q_avg_conj[:, :3] *= -1  # 虚部取反：[x,y,z,w] → [-x,-y,-z,w]
            
            # 将目标旋转与平均旋转的共轭相乘
            q_composite = quat_multiply(
                target_quat.unsqueeze(0).expand(bs, 4),  # 扩展目标四元数以匹配批次大小
                q_avg_conj
            )  # [bs, 4]
            
            q_composite = safe_normalize(q_composite)  # 确保复合旋转是单位四元数
            q_composite = q_composite.unsqueeze(1)  # [bs, 1, 4]，便于广播
            
            # 5. 应用复合旋转：q_final = q_composite * q
            q_final = quat_multiply(q_composite, q)  # [bs, len, 4]
            
            # 6. 保持原始全零位置为零
            q_final = q_final * valid_mask.unsqueeze(-1)  # [bs, len, 4]
            
            return q_final

        
        class MotionFeatureExtractor(nn.Module):
            """
            支持CUDA/CPU输入的运动特征提取模块
            输入: [bs, len, 7] 张量（[acc_x, acc_y, acc_z, rot_x, rot_y, rot_z, rot_w]）
            输出: [bs, len, 7] 张量（[线性加速度x/y/z, 角速度x/y/z, 角距离]）
            """
            def __init__(self, time_delta=1/200):
                super().__init__()
                self.time_delta = time_delta
                # ！修改1：不提前固定gravity_world的设备，改为使用时动态匹配输入设备
                self.gravity_world_val = torch.tensor([0.0, 0.0, 9.81], dtype=torch.float32)
                
            def quat_to_rot_matrix(self, quat):
                """四元数（[x,y,z,w]）转旋转矩阵 [bs, len, 3, 3]"""
                x, y, z, w = quat[..., 0], quat[..., 1], quat[..., 2], quat[..., 3]
                
                xx = x * x
                yy = y * y
                zz = z * z
                xy = x * y
                xz = x * z
                yz = y * z
                xw = x * w
                yw = y * w
                zw = z * w
                ww = w * w
                
                # 旋转矩阵计算（基于输入quat的设备，自动在CUDA/CPU上运算）
                rot_mat = torch.stack([
                    xx + ww - (yy + zz), 2 * (xy - zw), 2 * (xz + yw),
                    2 * (xy + zw), yy + ww - (xx + zz), 2 * (yz - xw),
                    2 * (xz - yw), 2 * (yz + xw), zz + ww - (xx + yy)
                ], dim=-1).view(*quat.shape[:-1], 3, 3)
                
                return rot_mat
            
            def rot_matrix_inverse(self, rot_mat):
                """旋转矩阵求逆（逆=转置，设备与输入一致）"""
                return rot_mat.transpose(-2, -1)
            
            def rot_matrix_multiply(self, rot_mat1, rot_mat2):
                """旋转矩阵乘法（设备自动匹配）"""
                return torch.matmul(rot_mat1, rot_mat2)
            
            def apply_rotation(self, rot_mat, vec):
                """旋转矩阵应用于向量（设备自动匹配）"""
                return torch.matmul(rot_mat, vec.unsqueeze(-1)).squeeze(-1)
            
            def remove_gravity_from_acc(self, acc_data, rot_data):
                """从加速度中移除重力（核心：设备动态匹配）"""
                bs, seq_len, _ = acc_data.shape
                device = acc_data.device  # ！修改2：获取输入设备（CUDA/CPU）
                
                valid_quats_normalized = rot_data / torch.norm(rot_data, dim=-1, keepdim=True).clamp(min=1e-8)
                rotations = self.quat_to_rot_matrix(valid_quats_normalized)
                
                # ！修改3：gravity_world动态匹配输入设备，避免CPU/CUDA不兼容
                gravity_world = self.gravity_world_val.to(device).expand(bs, seq_len, 3)
                # 重力向量从世界坐标系转到传感器坐标系
                gravity_sensor_frame = self.apply_rotation(
                    self.rot_matrix_inverse(rotations),
                    gravity_world
                )
                
                # 线性加速度计算（设备与输入一致）
                linear_accel = acc_data - gravity_sensor_frame
                return linear_accel
            
            def calculate_angular_velocity_from_quat(self, rot_data):
                """从四元数计算角速度（设备动态匹配）"""
                bs, seq_len, _ = rot_data.shape
                device = rot_data.device  # ！修改4：获取输入设备
                angular_vel = torch.zeros(bs, seq_len, 3, device=device)  # 零张量指定设备
                
                if seq_len < 2:
                    return angular_vel
                
                # 四元数有效性判断（torch.tensor(0.0)改为匹配设备）
                quat_norms = torch.norm(rot_data, dim=-1)
                # ！修改5：用rot_data.new_zeros(1)创建同设备的0张量，替代硬编码的torch.tensor(0.0)
                valid_mask = ~(torch.isnan(quat_norms) | torch.isclose(quat_norms, rot_data.new_zeros(1)))
                valid_pairs_mask = valid_mask[:, :-1] & valid_mask[:, 1:]
                
                if torch.any(valid_pairs_mask):
                    # 提取有效四元数对
                    q_t = rot_data[:, :-1][valid_pairs_mask]
                    q_t_plus_dt = rot_data[:, 1:][valid_pairs_mask]
                    
                    # 归一化
                    q_t_norms = quat_norms[:, :-1][valid_pairs_mask].unsqueeze(-1)
                    q_t_plus_dt_norms = quat_norms[:, 1:][valid_pairs_mask].unsqueeze(-1)
                    q_t_norm = q_t / q_t_norms.clamp(min=1e-8)
                    q_t_plus_dt_norm = q_t_plus_dt / q_t_plus_dt_norms.clamp(min=1e-8)
                    
                    # 旋转矩阵与相对旋转计算
                    rot_t = self.quat_to_rot_matrix(q_t_norm)
                    rot_t_plus_dt = self.quat_to_rot_matrix(q_t_plus_dt_norm)
                    delta_rot = self.rot_matrix_multiply(self.rot_matrix_inverse(rot_t), rot_t_plus_dt)
                    
                    # 罗德里格斯公式求旋转向量（设备自动匹配）
                    trace = delta_rot[..., 0, 0] + delta_rot[..., 1, 1] + delta_rot[..., 2, 2]
                    theta = torch.acos(torch.clamp((trace - 1) / 2, -1.0, 1.0))
                    sin_theta = torch.sin(theta)
                    
                    # 旋转向量计算（零张量指定设备）
                    rot_vec = torch.zeros_like(q_t_norm[..., :3], device=device)
                    # ！修改6：同修改5，避免设备不兼容
                    mask = ~torch.isclose(sin_theta, rot_data.new_zeros(1))
                    
                    if torch.any(mask):
                        factor = theta / (2 * sin_theta)
                        rot_vec[mask] = factor[mask, None] * torch.stack([
                            delta_rot[mask, 2, 1] - delta_rot[mask, 1, 2],
                            delta_rot[mask, 0, 2] - delta_rot[mask, 2, 0],
                            delta_rot[mask, 1, 0] - delta_rot[mask, 0, 1]
                        ], dim=1)
                    
                    # 角速度计算
                    angular_vel[:, :-1][valid_pairs_mask] = rot_vec / self.time_delta
                
                # 最后一个时间步复制前一个值
                if seq_len > 1:
                    angular_vel[:, -1] = angular_vel[:, -2]
                
                return angular_vel
            
            def calculate_angular_distance(self, rot_data, cumulative=False):
                """计算角距离（设备动态匹配）"""
                bs, seq_len, _ = rot_data.shape
                device = rot_data.device  # ！修改7：获取输入设备
                angular_dist = torch.zeros(bs, seq_len, 1, device=device)  # 零张量指定设备
                
                if seq_len < 2:
                    return angular_dist
                
                # 四元数归一化
                quat_norms = torch.norm(rot_data, dim=-1)
                q1 = rot_data[:, :-1]
                q2 = rot_data[:, 1:]
                q1_norms = quat_norms[:, :-1].unsqueeze(-1)
                q2_norms = quat_norms[:, 1:].unsqueeze(-1)
                q1_norm = q1 / q1_norms.clamp(min=1e-8)
                q2_norm = q2 / q2_norms.clamp(min=1e-8)
                
                # 相对旋转与角距离计算
                r1 = self.quat_to_rot_matrix(q1_norm)
                r2 = self.quat_to_rot_matrix(q2_norm)
                relative_rotation = self.rot_matrix_multiply(self.rot_matrix_inverse(r1), r2)
                
                trace = relative_rotation[..., 0, 0] + relative_rotation[..., 1, 1] + relative_rotation[..., 2, 2]
                theta = torch.acos(torch.clamp((trace - 1) / 2, -1.0, 1.0))
                angular_dist[:, :-1, 0] = theta
                
                # 最后一个时间步处理
                if seq_len > 1:
                    angular_dist[:, -1] = angular_dist[:, -2] if cumulative else 0.0
                
                # 累积角距离（设备一致）
                if cumulative:
                    angular_dist = torch.cumsum(angular_dist, dim=1)
                
                return angular_dist
            
            def forward(self, x):
                """前向传播（自动适配CUDA/CPU输入）"""
                # 分离加速度（前3维）和四元数（后4维）
                acc_data = x[..., :3]
                rot_data = x[..., 3:]
                
                # 计算三大特征（设备自动匹配输入）
                linear_accel = self.remove_gravity_from_acc(acc_data, rot_data)
                angular_vel = self.calculate_angular_velocity_from_quat(rot_data)
                angular_dist = self.calculate_angular_distance(rot_data)
                
                # 拼接输出（设备与输入一致）
                features = torch.cat([linear_accel, angular_vel, angular_dist], dim=-1)
                return features
        

        
        class ImuFeatureExtractor(nn.Module):
            def __init__(self, ):
                super().__init__()
                k = 15
                self.lpf_all = nn.Conv1d(7, 7, kernel_size=7, padding=7//2, groups=7, bias=False)
        
                self.lpf_acc  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
                self.lpf_acc2  = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
                self.lpf_gyro2 = nn.Conv1d(3, 3, k, padding=k//2, groups=3, bias=False)
        
            def forward(self, imu):
                # imu: 
                B, C, T = imu.shape
                acc  = imu[:, 0:3, :]                 # acc_x, acc_y, acc_z
                gyro = imu[:, 3:6, :]                 # gyro_x, gyro_y, gyro_z
                extra = imu[:, 6:, :]
        
                # 1) magnitude
                acc_mag  = torch.norm(acc,  dim=1, p=2, keepdim=True)          # (B,1,T)
                gyro_mag = torch.norm(gyro, dim=1, p=2, keepdim=True)
        
                # 1.2) magnitude
                acc2 = acc/acc_mag.clip(1e-12)
                gyro2 = gyro/gyro_mag.clip(1e-12)
        
        
                acc_lpf2  = self.lpf_acc2(acc2)
                acc_hpf2  = acc2 - acc_lpf2
                gyro_lpf2 = self.lpf_gyro2(gyro2)
                gyro_hpf2 = gyro2 - gyro_lpf2
                
        
                # 1.3) magnitude
                acc_mag2  = torch.norm(acc,  dim=1, p=1, keepdim=True)          # (B,1,T)
                gyro_mag2 = torch.norm(gyro, dim=1, p=1, keepdim=True)
        
                # 1.4) magnitude
                acc3 = acc/acc_mag2.clip(1e-12)
                gyro3 = gyro/gyro_mag2.clip(1e-12)
        
                # 2) jerk 
                jerk = F.pad(acc[:, :, 1:] - acc[:, :, :-1], (1,0))       # (B,3,T)
                gyro_delta = F.pad(gyro[:, :, 1:] - gyro[:, :, :-1], (1,0))
        
                # 2) jerk level2
                jerk2 = F.pad(acc[:, :, 2:] + acc[:, :, :-2] - acc[:, :, 1:-1] * 2, (1,1))       # (B,3,T)
                gyro_delta2 = F.pad(gyro[:, :, 2:] + gyro[:, :, :-2] - gyro[:, :, 1:-1] * 2, (1,1))
        
                # 3) energy
                acc_pow  = acc ** 2
                gyro_pow = gyro ** 2
        
                # 4) LPF / HPF 
                acc_lpf  = self.lpf_acc(acc)
                acc_hpf  = acc - acc_lpf
                gyro_lpf = self.lpf_gyro(gyro)
                gyro_hpf = gyro - gyro_lpf
        
        
                imu_hpf = imu - self.lpf_all(imu)
        
                features = [
                    acc, gyro,
                    
                    # acc2, gyro2,
                    acc_mag, gyro_mag,
                    
                    acc3, gyro3,
                    acc_mag2, gyro_mag2,
                    
                    jerk, gyro_delta,
                    jerk2, gyro_delta2, 
                    
                    acc_pow, gyro_pow,
                    
                    acc_lpf, acc_hpf,
                    gyro_lpf, gyro_hpf,
        
                    acc_lpf2, acc_hpf2,
                    gyro_lpf2, gyro_hpf2,
        
                    imu_hpf,
        
                    extra, imu.argsort(-1).argsort(-1)/128,
                    
                ]
        
                return torch.cat(features, dim=1)  # (B, C_out, T)
                
        class SEBlock(nn.Module):
            def __init__(self, channels, reduction=8):
                super().__init__()
                self.squeeze = nn.AdaptiveAvgPool1d(1)
                self.excitation = nn.Sequential(
                    nn.Linear(channels, channels // reduction, bias=False),
                    nn.ReLU(inplace=True),
                    nn.Linear(channels // reduction, channels, bias=False),
                    nn.Sigmoid()
                )
            
            def forward(self, x):
                b, c, _ = x.size()
                y = self.squeeze(x).view(b, c)
                y = self.excitation(y).view(b, c, 1)
                return x * y.expand_as(x)

        class ResidualSECNNBlock(nn.Module):
            def __init__(self, in_channels, out_channels, kernel_size, pool_size=2, dropout=0.3, weight_decay=1e-4):
                super().__init__()
                
                # First conv block
                self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn1 = nn.BatchNorm1d(out_channels)
                
                # Second conv block
                self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size, padding=kernel_size//2, bias=False)
                self.bn2 = nn.BatchNorm1d(out_channels)
                
                # SE block
                self.se = SEBlock(out_channels)
                
                # Shortcut connection
                self.shortcut = nn.Sequential()
                if in_channels != out_channels:
                    self.shortcut = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, 1, bias=False),
                        nn.BatchNorm1d(out_channels)
                    )
                self.pool_size = pool_size
                self.pool = nn.MaxPool1d(pool_size)
                self.dropout = nn.Dropout(dropout)
        
                self.act = nn.ReLU()

                self.gru = nn.GRU(out_channels, out_channels//2, batch_first=True, bidirectional=True)
                
            def forward(self, x):
                shortcut = self.shortcut(x)
                
                # First conv
                out = self.act(self.bn1(self.conv1(x)))
                # Second conv
                out = self.bn2(self.conv2(out))
                
                # SE block
                out = self.se(out)
                
                # Add shortcut
                
                out += shortcut
                
                out = out + self.gru(out.transpose(1, 2))[0].transpose(1, 2)
                out = self.act(out)

                out = self.act(out)
                
                
                # Pool and dropout
                if self.pool_size>1:
                    out = self.pool(out)
                out = self.dropout(out)
                
                return out

        class AttentionLayer(nn.Module):
            def __init__(self, hidden_dim):
                super().__init__()
                self.attention = nn.Linear(hidden_dim, 1)
                
            def forward(self, x):
                # x shape: (batch, seq_len, hidden_dim)
                scores = torch.tanh(self.attention(x))  # (batch, seq_len, 1)
                weights = F.softmax(scores.squeeze(-1), dim=1)  # (batch, seq_len)
                context = torch.sum(x * weights.unsqueeze(-1), dim=1)  # (batch, hidden_dim)
                return context


        def get_gravity_torch(quat):
            g=9.81
             # 四元数旋转
            x = quat[..., 0:1]  # rot_x
            y = quat[..., 1:2]  # rot_y
            z = quat[..., 2:3]  # rot_z
            w = quat[..., 3:4]  # rot_w
            g_x = 2 * g * (x * z - w * y)
            g_y = 2 * g * (y * z + w * x)
            g_z = g * (w**2 - x**2 - y**2 + z**2)
            gravity = torch.concat([g_x, g_y, g_z], -1)
            return gravity

        tof_mode = self.tof_mode
        
        class TwoBranchModel(nn.Module):
            def __init__(self, pad_len, imu_dim_raw, tof_dim, n_classes, dropouts=[0.3, 0.3, 0.3, 0.3, 0.4, 0.5, 0.3], feature_engineering=True, **kwargs):
                super().__init__()
                self.imu_fe1 = ImuFeatureExtractor(**kwargs)
                self.imu_fe2 = ImuFeatureExtractor(**kwargs)
                self.imu_fe3 = ImuFeatureExtractor(**kwargs)
                
                imu_dim = 32 + 1 + 14 + 7 + 6 + 6 + 7 

                self.new_feat_module = MotionFeatureExtractor()
                
                self.imu_dim = imu_dim
                self.tof_dim = tof_dim
        
                self.fir_nchan = 7
                self.thm_nchan = 5
                self.tof_nchan = 5 * (5 + 3 * tof_mode)
        
                weight_decay = 3e-3
        
                
                # IMU deep branch
                self.imu_block1 = ResidualSECNNBlock(imu_dim * 1, 160, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block2 = ResidualSECNNBlock(160, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)
        
                self.imu_block12 = ResidualSECNNBlock(imu_dim * 1, 160, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                self.imu_block22 = ResidualSECNNBlock(160, 256, 5, dropout=dropouts[1], pool_size=1, weight_decay=weight_decay)

                self.thm_block = nn.Sequential(
                    ResidualSECNNBlock(self.thm_nchan * 1, 32, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(32, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.tof_block = nn.Sequential(
                    ResidualSECNNBlock(self.tof_nchan * 1, 64, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay),
                    ResidualSECNNBlock(64, 256, 5, dropout=dropouts[0], pool_size=1, weight_decay=weight_decay)
                )
                
                self.emb_all = 256
        
                self.dropout1d = nn.Dropout1d(0.15)
                
                # BiLSTM
                self.dim_lstm = 512
                self.dim_encoder = self.dim_lstm * 2
        
                self.bilstm = nn.LSTM(self.emb_all, self.dim_lstm, num_layers=2, bidirectional=True, batch_first=True)
            
                self.lstm_dropout = nn.Dropout(dropouts[4])
        
                self.output2 = nn.Linear(self.dim_encoder, 5)
                
                # Dense layers
            
                self.dense1 = nn.Linear(self.dim_encoder * 2, 512, bias=False)
                self.bn_dense1 = nn.BatchNorm1d(512)
                self.drop1 = nn.Dropout(dropouts[5])
                
                self.dense2 = nn.Linear(512, 512, bias=False)
                self.bn_dense2 = nn.BatchNorm1d(512)
                self.drop2 = nn.Dropout(dropouts[6])
        
                self.output3 = nn.Linear(512, 5)
                self.classifier = nn.Linear(512, n_classes * 4)
        
                self.scale = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale2 = nn.Parameter(torch.ones((1, 256, 1)))
                self.bias2 = nn.Parameter(torch.zeros((1, 256, 1)))
        
                self.scale3 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias3 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.scale4 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias4 = nn.Parameter(torch.zeros((1, 512, 1)))

                self.scale5 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias5 = nn.Parameter(torch.zeros((1, 512, 1)))

                self.scale6 = nn.Parameter(torch.ones((1, 512, 1)))
                self.bias6 = nn.Parameter(torch.zeros((1, 512, 1)))
        
                self.act = nn.GELU()
                
            def forward(self, x, ):
                # Split input
                mask_all = (x.abs().max(-1)[0]!=0).float()[:,:,None]
                mask_all_trans = mask_all.transpose(1, 2)
                
                imu = x[:, :, :self.fir_nchan] # (batch, imu_dim, seq_len)
                imu2 = self.new_feat_module(imu)
                imu2 = imu2[:,:,:7]

                imu = imu.transpose(1, 2)
                imu2 = imu2.transpose(1, 2)
                
                thm = x[:, :, self.fir_nchan:self.fir_nchan + self.thm_nchan].transpose(1, 2)  # (batch, thm_dim, seq_len)
                tof = x[:, :, self.fir_nchan + self.thm_nchan:].transpose(1, 2)  # (batch, tof_dim, seq_len)
        
                imu = self.imu_fe1(imu)   # (B, imu_dim, T)
                imu2 = self.imu_fe2(imu2)   # (B, imu_dim, T)
                
                imu = (imu * self.scale[:,:imu.shape[1],:] + self.bias[:,:imu.shape[1],:]) * mask_all.transpose(1, 2)
                imu2 = (imu2 * self.scale2[:,:imu2.shape[1],:] + self.bias2[:,:imu2.shape[1],:]) * mask_all.transpose(1, 2)
                
                thm = (thm * self.scale5[:,:thm.shape[1],:]  + self.bias5[:,:thm.shape[1],:] ) * mask_all.transpose(1, 2)
                tof = (tof * self.scale6[:,:tof.shape[1],:]  + self.bias6[:,:tof.shape[1],:] ) * mask_all.transpose(1, 2)
        
                thm = self.dropout1d(thm)
                tof = self.dropout1d(tof)
                imu = self.dropout1d(imu)
                imu2 = self.dropout1d(imu2)
                
                thm = self.thm_block(thm)
                tof = self.tof_block(tof)
        
                
                # IMU branch
                x1 = self.imu_block1(imu)
                x1 = self.imu_block2(x1)
        
                x12 = self.imu_block12(imu2)
                x12 = self.imu_block22(x12)
                
                x1 = (x1 + x12 + thm + tof)
        
                merged = x1.transpose(1, 2)  # (batch, seq_len, 256)
                
                # BiLSTM
                lstm_out, _ = self.bilstm(merged)
        
                logits2 = self.output2(self.lstm_dropout(lstm_out))
                
                attended = torch.cat(((lstm_out).max(1)[0], (lstm_out).mean(1), ), -1)
                attended = self.lstm_dropout(attended)
                
                
                # Dense layers
                x = self.act(self.bn_dense1(self.dense1(attended)))
                x = self.drop1(x)
                x = self.act(self.bn_dense2(self.dense2(x)))
                x = self.drop2(x)
                
                # Classification
                logits = (self.classifier(x))
                logits3 = (self.output3(x))
        
                if not self.training:
                    return logits.reshape(logits.shape[0], 4, 18).mean(1), logits3
                
                return logits, logits2, logits3


        return TwoBranchModel(param_a, param_b, param_c, param_d)

    def preprocess_sequence(self, df_seq: pd.DataFrame, feature_cols: list, scaler: StandardScaler):
        """Normalizes and cleans the time series sequence"""
        mat = df_seq[feature_cols].ffill().bfill().fillna(0).values
        return (mat).astype('float32')
        
    def pad_sequences_torch(self, sequences, maxlen, padding='pre', truncating='pre', value=0.0):
        """PyTorch equivalent of Keras pad_sequences"""
        result = []
        for seq in sequences:
            if len(seq) >= maxlen:
                if truncating == 'post':
                    seq = seq[:maxlen]
                else:  # 'pre'
                    seq = seq[-maxlen:]
            else:
                pad_len = maxlen - len(seq)
                if padding == 'post':
                    seq = np.concatenate([seq, np.full((pad_len, seq.shape[1]), value)])
                else:  # 'pre'
                    seq = np.concatenate([np.full((pad_len, seq.shape[1]), value), seq])
            result.append(seq)
        return np.array(result, dtype=np.float32)
        
    def preprocess_left_handed(self, l_tr):
        def handle_quaternion_missing_values(rot_data: np.ndarray) -> np.ndarray:
            """
            Handle missing values in quaternion data intelligently
            
            Key insight: Quaternions must have unit length |q| = 1
            If one component is missing, we can reconstruct it from the others
            """
            rot_cleaned = rot_data.copy()
            
            for i in range(len(rot_data)):
                row = rot_data[i]
                missing_count = np.isnan(row).sum()
                
                if missing_count == 0:
                    # No missing values, normalize to unit quaternion
                    norm = np.linalg.norm(row)
                    if norm > 1e-8:
                        rot_cleaned[i] = row / norm
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]  # Identity quaternion
                        
                elif missing_count == 1:
                    # One missing value, reconstruct using unit quaternion constraint
                    # |w|² + |x|² + |y|² + |z|² = 1
                    missing_idx = np.where(np.isnan(row))[0][0]
                    valid_values = row[~np.isnan(row)]
                    
                    sum_squares = np.sum(valid_values**2)
                    if sum_squares <= 1.0:
                        missing_value = np.sqrt(max(0, 1.0 - sum_squares))
                        # Choose sign for continuity with previous quaternion
                        if i > 0 and not np.isnan(rot_cleaned[i-1, missing_idx]):
                            if rot_cleaned[i-1, missing_idx] < 0:
                                missing_value = -missing_value
                        rot_cleaned[i, missing_idx] = missing_value
                        rot_cleaned[i, ~np.isnan(row)] = valid_values
                    else:
                        rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
                else:
                    # More than one missing value, use identity quaternion
                    rot_cleaned[i] = [0.0, 0.0, 0.0, 0.0]
            
            return rot_cleaned
    
        rot_cleaned = handle_quaternion_missing_values(l_tr[["rot_w","rot_x", "rot_y", "rot_z"]].to_numpy())
        rot_scipy = rot_cleaned[:, [1, 2, 3, 0]]
        
        norms = np.linalg.norm(rot_scipy, axis=1)
        if np.any(norms < 1e-8):
            # Replace problematic quaternions with identity
            mask = norms < 1e-8
            rot_scipy[mask] = [0.0, 0.0, 0.0, 1.0]  # Identity quaternion in scipy format
        
        r = R.from_quat(rot_scipy)
        tmp = r.as_euler("xyz")
        tmp[:,1] = - tmp[:,1]
        tmp[:,2] = - tmp[:,2] 
        r = R.from_euler("xyz", tmp)
        tmp = r.as_quat()
        
        if np.any(norms < 1e-8):
            mask = norms < 1e-8
            tmp[mask] = [0.0, 0.0, 0.0, 0.0]
        
        l_tr = l_tr.with_columns(pl.DataFrame(tmp, schema=["rot_x", "rot_y", "rot_z", "rot_w"]))
        l_tr = l_tr.with_columns(-pl.col("acc_x"))
        
        tmp = l_tr[["thm_3", "thm_5"]]
        tmp.columns = ["thm_5", "thm_3"]
        l_tr = l_tr.with_columns(tmp)
        
        swap_1_2_4_base = [[0,7],[1,6],[2,5],[3,4], [4,3], [5,2],[6,1],[7,0]]
        swap_3_5_base = [[0,56],[8,48],[16,40], [24,32],[32,24], [40,16],[48,8], [56,0]]
        
        swap_1_2_4 = list()
        for i in range(0,64,8):
            ll = list()
            for (k,l) in swap_1_2_4_base:
                ll.append([k+i, l+i])
            swap_1_2_4 += ll
        
        swap_3_5 = list()
        for i in range(8):
            ll = list()
            for (k,l) in swap_3_5_base:
                ll.append([k+i, l+i])
            swap_3_5 += ll
        
        l_df = l_tr
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[k].alias(l))
        
        for (k,l) in zip(["tof_3_v" + str(x) for x in range(64)], ["tof_5_v" + str(x) for x in range(64)]):
            l_tr = l_tr.with_columns(l_df[l].alias(k))
        
        l_df = l_tr
        
        for i in [1,2,4]:
            for (k, l) in swap_1_2_4:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        
        for i in [3,5]:
            for (k, l) in swap_3_5:
                l_tr = l_tr.with_columns(l_df["tof_" + str(i) + "_v"+str(k)].alias("tof_" + str(i) + "_v"+str(l)))
        return l_tr
        
    def train(self, ):
        class CMI3Dataset(Dataset):
            def __init__(self,
                         X_list,
                         y_list, y_list2, y_list3,
                         maxlen,
                         mode="train",
                         imu_dim=7,
                         augment=None,
                         epoch_multiplier=1,
                        ):
                self.X_list = X_list
                self.mode = mode
                self.y_list = y_list
                self.y_list2 = y_list2
                self.y_list3 = y_list3
                self.maxlen = maxlen
                self.imu_dim = imu_dim     
                self.augment = augment
                self.epoch_multiplier = epoch_multiplier
                
            def __getitem__(self, index):
                X = self.X_list[index//self.epoch_multiplier]
                y = self.y_list[index//self.epoch_multiplier]
                y2 = self.y_list2[index//self.epoch_multiplier]
                y3 = self.y_list3[index//self.epoch_multiplier]

                if self.mode=='train':
                        X_ = np.concatenate([X, y2], -1)
        
                        X_ = transforms_custom(X_)
        
                        X = X_[:,:X.shape[-1]]
                        y2 = X_[:,X.shape[-1]:]
    
                        X[:,3:7] = X[:,3:7]/(np.sqrt((X[:,3:7]**2).sum(-1))+1e-6)[:,None]
            
                return X, y, y2, y3
            
            def __len__(self):
                return len(self.X_list) * self.epoch_multiplier

        class EMA:
            def __init__(self, model, decay=0.999):
                self.decay = decay
                self.shadow = {}
                self.backup = {}
        
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.shadow[name] = param.data.clone()
        
            def update(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        assert name in self.shadow
                        new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                        self.shadow[name] = new_average.clone()
        
            def apply_shadow(self, model):
                self.backup = {}
                for name, param in model.named_parameters():
                    if param.requires_grad:
                        self.backup[name] = param.data.clone()
                        param.data = self.shadow[name]
        
            def restore(self, model):
                for name, param in model.named_parameters():
                    if param.requires_grad and name in self.backup:
                        param.data = self.backup[name]
                self.backup = {}

        def set_seed(seed: int = 42):
            import numpy as np
            
            random.seed(seed)
        
            os.environ['PYTHONHASHSEED'] = str(seed)
        
            np.random.seed(seed)
        
            torch.manual_seed(seed)
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed) 
            # torch.backends.cudnn.deterministic = True
            # torch.backends.cudnn.benchmark = False
            # torch.use_deterministic_algorithms(True)
        
        set_seed(self.SEED)

        class RDROPLoss(nn.Module):
            """
            RDROP损失函数实现
            结合原始损失和KL散度约束
            """
            def __init__(self, alpha=0.5):
                super(RDROPLoss, self).__init__()
                self.alpha = alpha  # KL损失的权重系数
                self.kl_div = nn.KLDivLoss(reduction='batchmean')
        
            def forward(self, logits1, logits2):
                # KL散度约束：两次次输出分布的一致性
                p1 = F.log_softmax(logits1, dim=1)
                p2 = F.softmax(logits2, dim=1)
                kl_loss1 = self.kl_div(p1, p2)
                
                p2 = F.log_softmax(logits2, dim=1)
                p1 = F.softmax(logits1, dim=1)
                kl_loss2 = self.kl_div(p2, p1)
                
                # 总损失 = 分类损失 + alpha * KL散度损失
                total_loss = self.alpha * (kl_loss1 + kl_loss2)
                return total_loss
                
        import numpy as np
        import pandas as pd
        from scipy.spatial.transform import Rotation as R
        from tqdm.auto import tqdm 
        
        print("▶ TRAIN MODE – loading dataset …")
        df = pd.read_csv(self.RAW_DIR / "train.csv")
        self.demo = pd.read_csv(self.RAW_DIR / "train_demographics.csv")



        
        group_list = []
        for _, group in tqdm(df.groupby('sequence_id')):
            index_old = group.index
            if self.demo[self.demo['subject']==group['subject'].values[0]]['handedness'].values[0] ==0:
                ###################################################################################
                group = self.preprocess_left_handed(pl.from_pandas(group)).to_pandas()
                group.index = index_old
                
                group_list.append(group)
        df.update(pd.concat(group_list))


        
    
        dict_ = dict(zip(df['subject'].unique(), list(range(df['subject'].nunique()))))
        df['subject_id_new'] = df['subject'].map(dict_)
    
        #df = df[~df['subject'].isin({'SUBJ_045235', 'SUBJ_019262'})]
    
        dict_behavior = {'Relaxes and moves hand to target location': 1,
                         'Hand at target location': 2,
                         'Performs gesture': 3,
                         'Moves hand to target location': 4}
        
        dict_orientation = {
                            'Lie on Back': 0,
                            'Lie on Side - Non Dominant': 1,
                            'Seated Lean Non Dom - FACE DOWN': 2,
                            'Seated Straight': 3,
                           }
    
        df['bid'] = df['behavior'].map(dict_behavior)
        df['oid'] = df['orientation'].map(dict_orientation)
    
        df, new_feat_cols, new_feat_cols2 = self.get_new_features(df)
    
        # Label encoding
        le = LabelEncoder()
        df['gesture_int'] = le.fit_transform(df['gesture'])


        df['gesture_int'] = df['gesture_int'] + df['oid'] * 18

        
        np.save(self.EXPORT_DIR / "gesture_classes.npy", le.classes_)
    
        # Feature list
        meta_cols = {'gesture', 'gesture_int', 'sequence_type', 'behavior', 'orientation',
                     'row_id', 'subject', 'phase', 'sequence_id', 'sequence_counter', 'subject_id_new'}
        feature_cols = [c for c in df.columns if c not in meta_cols]
        imu_cols = [c for c in feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        # tof_cols = [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        thm_cols = [c for c in feature_cols if c.startswith('thm_')]
        print(thm_cols)
        
        tof_cols = [] # [c for c in feature_cols if c.startswith('thm_') or c.startswith('tof_')]
    
        imu_cols = ['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_z', 'rot_w',]
        
        feature_cols = imu_cols + new_feat_cols + thm_cols + new_feat_cols2
        
        print(f"  IMU {len(imu_cols)+len(new_feat_cols)} | THM {len(thm_cols)} | TOF {len(new_feat_cols2)}  | total {len(feature_cols)} features")
    
        # Save feature_cols
        np.save(self.EXPORT_DIR / "feature_cols.npy", np.array(feature_cols))
        pad_len = self.PAD_PERCENTILE
        
        # Group sequences
        seq_gp = df.groupby('sequence_id')
        X_list_raw, y_list, id_list, subject_list, y2list, y3list = [], [], [], [], [], []
        for seq_id, seq in seq_gp:
            mat = seq[feature_cols].ffill().bfill().fillna(0).values
            X_list_raw.append(mat)
            y_list.append(seq['gesture_int'].iloc[0])
            id_list.append(seq_id)
            subject_list.append(seq['subject_id_new'].iloc[0])
    
            if len(seq) < pad_len:
                bid = np.zeros(pad_len)
                bid[-len(seq):] = seq['bid'].values.ravel()
            else:
                bid = seq['bid'].values.ravel()[-pad_len:]
            
            y2list.append(bid)
            y3list.append(seq['oid'].iloc[0])
    
        pad_len = self.PAD_PERCENTILE
        np.save(self.EXPORT_DIR / "sequence_maxlen.npy", pad_len)
    
        id_list = np.array(id_list)
        y_list_all = np.eye(4 * len(le.classes_))[y_list].astype(np.float32)  # one-hot
    
        y_list_all2 = np.vstack(y2list).astype(int)
        y_list_all2 = (np.eye(5)[y_list_all2.reshape(-1, 1)]).reshape(-1, pad_len, 5).astype(np.float32)
        y_list_all3 = np.eye(5)[y3list].astype(np.float32)
    
        augmenter = None
        metrics = []
    
        criterion_rdrop = RDROPLoss(alpha=0.5)

        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(
                         n_splits=self.FOLDS, 
                         shuffle=True, 
                         random_state=self.random_state
                        )
    
        def clipped_cross_entropy(logits, y, clipval=0.6, num=18):
            return -torch.sum(F.log_softmax(logits, dim=1) * y.clip((1-clipval)/num, clipval), dim=1).mean() 

            # return 1 - (logits.softmax(-1) * y).mean()
            
        idlistall = []
        targetfinalall = []
        predimuonlyall = []
        predallfeatall = []
        predorientationall = []
        targetsorientations = []
        foldlistall = []
        for fold, (train_idx, val_idx) in enumerate(gkf.split(id_list, id_list, groups=subject_list)):
            
        # skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=self.random_state)
        # for fold, (train_idx, val_idx) in enumerate(skf.split(id_list, np.argmax(y_list_all, axis=1))):  
            
            if fold not in self.TRAIN_FOLDS:
                continue
    
            print(f"\n▶ Fold {fold}")
            
            X_list_scaled = [x for x in X_list_raw]
            X_list_all = self.pad_sequences_torch(X_list_scaled, maxlen=pad_len, padding='pre', truncating='pre')
    
            # Prepare train/val sets
            train_list = X_list_all[train_idx]
            train_y_list = y_list_all[train_idx]
            train_y_list2 = y_list_all2[train_idx]
            train_y_list3 = y_list_all3[train_idx]
    
            
            val_list = X_list_all[val_idx]
            val_y_list = y_list_all[val_idx]
    
            val_y_list2 = y_list_all2[val_idx]
            val_y_list3 = y_list_all3[val_idx]

            id_list_valid = id_list[val_idx]

            print(len(id_list_valid))
            idlistall.append(id_list_valid)
            foldlistall.append([fold for _ in range(len(id_list_valid))])
    
            # Data loaders
            train_dataset = CMI3Dataset(train_list, train_y_list, train_y_list2, train_y_list3, pad_len, mode="train", imu_dim=len(imu_cols),
                                        augment=augmenter)
            train_loader = DataLoader(train_dataset, batch_size=self.BATCH_SIZE, shuffle=True, num_workers=6, drop_last=False, pin_memory=True)
            
            val_dataset = CMI3Dataset(val_list, val_y_list, val_y_list2, val_y_list3, pad_len, mode="val")
            val_loader = DataLoader(val_dataset, batch_size=self.BATCH_SIZE, shuffle=False, num_workers=6, drop_last=False, pin_memory=True)

            device = self.device
            
            # Model & EMA 
            model = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)
            model2 = self.create_model(pad_len, len(imu_cols), len(tof_cols), len(le.classes_)).to(device)

            if self.VALIDATION==True:
                checkpoint = torch.load(self.PRETRAINED_DIR / f'gesture_two_branch_fold{fold}.pth', map_location=device)
                model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            
            ema = EMA(model, decay=0.998)
    
            # Optimizer
            hidden_weights = [p for p in model.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases = [p for p in model.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups = [
                dict(params=hidden_weights, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer = SingleDeviceMuonWithAuxAdam(param_groups)

            hidden_weights2 = [p for p in model2.parameters() if p.ndim >= 2 and p.requires_grad]
            hidden_gains_biases2 = [p for p in model2.parameters() if p.ndim < 2 and p.requires_grad]
            param_groups2 = [
                dict(params=hidden_weights2, use_muon=True, lr=0.005, weight_decay=self.WD),
                dict(params=hidden_gains_biases2, use_muon=False, lr=self.LR_INIT, betas=(0.9, 0.95), weight_decay=self.WD),
            ]
            optimizer2 = SingleDeviceMuonWithAuxAdam(param_groups2)
    
            nbatch = len(train_loader)
            nsteps = self.EPOCHS * nbatch
    
            scheduler = ConstantCosineLR(optimizer, nsteps, 0.)    
            print("▶ Starting training...")
    
            best_val_acc = 0
            for epoch in tqdm(range(self.EPOCHS)):
                model.train()
                model2.train()
                
                train_preds, train_targets = [], []
                train_loss = 0.0
    
                for X, y, y2, y3 in train_loader:
                    if self.VALIDATION==True:
                        break
                        
                    BS = X.shape[0]
                    X0 = X.clone()

                    X[BS//8:,:,7:] = 0.

                    if np.random.random() > 0.5:
                        X[:BS//8*2,:,3:7] = 0.
                    else:
                        X[:BS//8*2,:,:3] = 0.
                    
                    X, y = X.float().to(device), y.to(device)
                    y2, y3 = y2.float().to(device), y3.to(device)
                    X0 = X0.float().to(device)
                    
                    optimizer.zero_grad()
                    optimizer2.zero_grad()
                    
                    LEN = 1200 + np.random.randint(80)
                    
                    logits, logits2, logits3 = model(X[:,-LEN:])
                    
                    logits_, logits2_, logits3_ = model2(X0[:,-LEN:])
    
                    clipval = 0.6
                    loss = clipped_cross_entropy(logits, y, clipval, 18) 
                    loss += clipped_cross_entropy(logits2, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss += clipped_cross_entropy(logits3, y3, clipval, 5)  * 0.5

                    clipval = 0.6
                    loss2 = clipped_cross_entropy(logits_, y, clipval, 18) 
                    loss2 += clipped_cross_entropy(logits2_, y2[:,-LEN:], clipval, 5)  * 0.5
                    loss2 += clipped_cross_entropy(logits3_, y3, clipval, 5)  * 0.5

                    if epoch > 40:
                        loss = loss + loss2 + criterion_rdrop(logits, logits_)
                    else:
                        loss = loss + loss2 
                    
                    loss.backward()
                    
                    optimizer.step()
                    optimizer2.step()
                    
                    ema.update(model)
    
                    train_preds.append(logits.argmax(dim=1).cpu().numpy())
                    train_targets.append(y.argmax(dim=1).cpu().numpy())
    
                    scheduler.step()
                    train_loss += loss.item()

                model2.eval()
                model.eval()
                ema.apply_shadow(model)
                
                val_loss = 0.0
                val_preds_accmask, val_targets_accmask = [], []
                val_preds_imuonly, val_targets_imuonly = [], []
                val_preds_rotmask, val_targets_rotmask = [], []

                val_preds_mother, val_targets_rotmask = [], []

                val_preds_logits = []
                val_preds_imuonly_logits = []

                val_preds_orientation, val_targets_orientation = [], []
                val_preds_orientation_logits = []
                
                
                with torch.inference_mode():
                    for X, y, y2, y3 in val_loader:
                        X_imuonly = X[:].clone()
                        X_imuonly[:, :, 7:] = 0.0

                        X_accmask = X[:].clone()
                        X_accmask[:, :, 7:] = 0.0
                        X_accmask[:, :, 0:3] = 0.0

                        X_rotmask = X[:].clone()
                        X_rotmask[:, :, 7:] = 0.0
                        X_rotmask[:, :, 3:7] = 0.0
                    
                        X, y = X.float().to(device), y.to(device)
                        X_imuonly = X_imuonly.float().to(device)
                        X_accmask = X_accmask.float().to(device)
                        X_rotmask = X_rotmask.float().to(device)

                        logits_mother, logits_orientation = model(X)
                        
                        logits_rotmask, _  = model(X_rotmask)
                        logits_accmask, _  = model(X_accmask)
                        logits_imuonly, _  = model(X_imuonly)

                        val_preds_mother.append(logits_mother.argmax(dim=1).cpu().numpy())
                        val_preds_accmask.append(logits_accmask.argmax(dim=1).cpu().numpy())
                        val_preds_rotmask.append(logits_rotmask.argmax(dim=1).cpu().numpy())
                        val_preds_imuonly.append(logits_imuonly.argmax(dim=1).cpu().numpy())
                        
                        val_preds_logits.append(logits_mother.cpu().numpy())
                        val_preds_imuonly_logits.append(logits_imuonly.cpu().numpy())
                        
                        val_targets_imuonly.append(y.reshape(y.shape[0], 4, 18).sum(1).argmax(dim=1).cpu().numpy())


                        val_preds_orientation_logits.append(logits_orientation.cpu().numpy())
                        val_preds_orientation.append(logits_orientation.argmax(dim=1).cpu().numpy())
                        val_targets_orientation.append(y3.argmax(dim=1).cpu().numpy())
    
    
                if len(train_targets) >= 0:
                    train_acc = 0.
                    try:
                        train_acc = CompetitionMetric().calculate_hierarchical_f1(
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_targets)]}),
                            pd.DataFrame({'gesture': le.classes_[np.concatenate(train_preds)]})
                        )
                    except:
                        pass
                    val_acc_rotmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_rotmask)]})
                    )
                    val_acc_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_imuonly)]})
                    )
                    val_acc_accmask = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_accmask)]})
                    )
                    val_acc_mother = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_targets_imuonly)]}),
                        pd.DataFrame({'gesture': le.classes_[np.concatenate(val_preds_mother)]})
                    )

                    val_acc_orientation = np.mean(
                        np.concatenate(val_preds_orientation)==np.concatenate(val_targets_orientation)
                    )
                    
                    train_loss = np.mean(train_loss)
                    print('epoch', epoch, 'loss : ', round(train_loss, 4), '| TRAIN : ', round(train_acc, 4), '| ORIENTATION: ', round(val_acc_orientation, 4), '| IMUONLY : ', round(val_acc_imuonly, 4), '| ROTMASK : ', round(val_acc_rotmask, 4),  '| ACCMASK : ', round(val_acc_accmask, 4),  '| MOTHER : ', round(val_acc_mother, 4), '| LR : ', optimizer.param_groups[0]['lr'])
    
                    metric = val_acc_orientation # val_acc
                    
                    if metric > best_val_acc:
                        best_val_acc = metric
                        # Save model
                        torch.save({
                            'model_state_dict': model.state_dict(),
                            'imu_dim': len(imu_cols),
                            'tof_dim': len(tof_cols),
                            'n_classes': len(le.classes_),
                            'pad_len': pad_len,
                            
                        }, self.EXPORT_DIR / f"gesture_two_branch_fold{fold}.pth")
                        print(f"fold: {fold} val_all_acc: {metric:.4f}")
                        print("✔ Training done – artefacts saved in", self.EXPORT_DIR)
                    
                    ema.restore(model)

                targetfinalall.append(np.concatenate(val_targets_imuonly))
                predimuonlyall.append(np.concatenate(val_preds_imuonly_logits, 0))
                predallfeatall.append(np.concatenate(val_preds_logits, 0))
                predorientationall.append(np.concatenate(val_preds_orientation_logits, 0))
                targetsorientations.append(np.concatenate(val_targets_orientation))
                if self.VALIDATION:
                    break
                
            
            ema.apply_shadow(model)
            metrics.append(best_val_acc)

        print(metrics, sum(metrics)/len(metrics))

        import joblib
        name = str(self.PRETRAINED_DIR).replace('/', '_')
        joblib.dump(
            {
                'guestures': le.classes_, 
                'pred_all': np.concatenate(predallfeatall, 0), 
                'pred_imuonly': np.concatenate(predimuonlyall, 0), 
                'targets': np.concatenate(targetfinalall), 
                'idlist': np.concatenate(idlistall),
                'fold': np.concatenate(foldlistall),
                'pred_orientation' : np.concatenate(predorientationall, 0),
                'target_orientation' : np.concatenate(targetsorientations), 
            }, 
                    f"oof_{name}.joblib", 
                   )

        return le.classes_, np.concatenate(predallfeatall, 0), np.concatenate(predimuonlyall, 0), np.concatenate(targetfinalall), np.concatenate(idlistall), np.concatenate(predorientationall, 0), np.concatenate(targetsorientations)
        
        
    def get_new_features(self, df):
        feature_names = []
        
        for col in ['thm_1', 'thm_2', 'thm_3', 'thm_4', 'thm_5']:
            df[col] = df[col] / 5.                                                                                                                                             
        
        new_columns = {} 
        for i in range(1, 6):
            pixel_cols = [f"tof_{i}_v{p}" for p in range(64)]
            
            new_columns.update({
                f'tof_{i}_isna_mean1': (df[pixel_cols]==-1).mean(axis=1),
                f'tof_{i}_isna_mean2': 1 - (df[pixel_cols].isna()).mean(axis=1),
            })
    
            df[pixel_cols] = 6 - df[pixel_cols].replace(-1, 255.) / 50. 
            
            tof_data = df[pixel_cols]
    
            new_columns.update({
                f'tof_{i}_mean': tof_data.mean(axis=1),
                f'tof_{i}_std': tof_data.std(axis=1),
                # f'tof_{i}_min': tof_data.min(axis=1),
                f'tof_{i}_max': tof_data.max(axis=1),
                
                # f'tof_{i}_median_norm': tof_data.median(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_max_norm': tof_data.min(axis=1)/tof_data.mean(axis=1).clip(1),
                # f'tof_{i}_min_norm': tof_data.max(axis=1)/tof_data.mean(axis=1).clip(1),
            })
            if self.tof_mode > 1:
                region_size = 64 // self.tof_mode
                for r in (range(self.tof_mode)):
                    region_data = tof_data.iloc[:, r*region_size : (r+1)*region_size]
                    new_columns.update({
                        f'tof{self.tof_mode}_{i}_region_{r}_mean': region_data.mean(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_std': region_data.std(axis=1),
                        # f'tof{self.tof_mode}_{i}_region_{r}_min': region_data.min(axis=1),
                        f'tof{self.tof_mode}_{i}_region_{r}_max': region_data.max(axis=1),
                    })
                    
        df_tof = pd.DataFrame(new_columns)
        df = pd.concat([df, df_tof], axis=1)
    
        
        return df, feature_names, list(df_tof.columns)
        

    def get_model(self, ):    
        print("▶ INFERENCE MODE – loading artefacts from", self.PRETRAINED_DIR)
        self.feature_cols = np.load(self.PRETRAINED_DIR / "feature_cols.npy", allow_pickle=True).tolist()
        self.pad_len = int(np.load(self.PRETRAINED_DIR / "sequence_maxlen.npy"))
        self.gesture_classes = np.load(self.PRETRAINED_DIR / "gesture_classes.npy", allow_pickle=True)
    
        self.imu_cols = [c for c in self.feature_cols if not (c.startswith('thm_') or c.startswith('tof_'))]
        self.tof_cols = [c for c in self.feature_cols if c.startswith('thm_') or c.startswith('tof_')]
        # Load model
        MODELS = [f'gesture_two_branch_fold{i}.pth' for i in range(self.FOLDS) if i in self.TRAIN_FOLDS]
        
        self.models = []
        for path in MODELS:
            checkpoint = torch.load(self.PRETRAINED_DIR / path, map_location=self.device)
            
            model = self.create_model(
                checkpoint['pad_len'], 
                checkpoint['imu_dim'], 
                checkpoint['tof_dim'], 
                checkpoint['n_classes']
                ).to(self.device)
    
            
            
            model.load_state_dict({k.replace('_orig_mod.', ''):v for k, v in checkpoint['model_state_dict'].items()})
            model.eval()
            self.models.append(model)
    
        print("  model, scaler, pads loaded – ready for evaluation")
    
        return 
    
    def predict(self, sequence: pl.DataFrame, demographics: pl.DataFrame, nanratio = 0):         
        if demographics[0, "handedness"] == 0: 
            sequence = self.preprocess_left_handed(sequence)
        
        df_seq = sequence.to_pandas()
        df_seq, _, _ = self.get_new_features(df_seq)
        
        with torch.no_grad():
            outputs = None
            outputs2 = None
            
            mat = self.preprocess_sequence(df_seq, self.feature_cols, None)
            pad = self.pad_sequences_torch([mat], maxlen=self.pad_len, padding='pre', truncating='pre')
            x = torch.FloatTensor(pad).float().to(self.device)

            if nanratio==1:
                x[:,:,7:] = 0.
            
            for model in self.models:
                model.eval()
                p, p2 = model(x)
                if outputs is None: 
                    outputs = p
                    outputs2 = p2
                else: 
                    outputs += p
                    outputs2 += p2
                    
            outputs /= len(self.models)
            outputs2 /= len(self.models)

        return self.gesture_classes, (outputs.cpu().numpy(), outputs2.cpu().numpy()[:,:4])

In [75]:
model_zhou_inference_v26_1 = model_zhou_v26('/kaggle/input/cmi3v90', 42, './42')
if VALIDATION:
    MMM = model_zhou_inference_v26_1
    
    MMM.TRAIN_FOLDS = [0, 1, 2, 3, 4, ]
    
    MMM.VALIDATION = VALIDATION
    guestures, pred_all_26_1, pred_imuonly_26_1, targets, idlist_26_1, pred_orientation_26_1, target_orientation_26_1 = MMM.train()

    val_score_imuonly = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_imuonly_26_1.argmax(1)]})
                    )
    val_score_all = CompetitionMetric().calculate_hierarchical_f1(
                        pd.DataFrame({'gesture': guestures[targets]}),
                        pd.DataFrame({'gesture': guestures[pred_all_26_1.argmax(1)]})
                    )

    print('score imuonly : ', val_score_imuonly)
    print('score allfeatures : ', val_score_all)
    
    MMM.VALIDATION = False

▶ imports ready · pytorch 2.6.0+cu124 · device: cuda


In [76]:
imuonly_models = [
    model_zhou_inference_v14_1,
    model_zhou_inference_v14_2,
    model_zhou_inference_v14_3,

    model_zhou_inference_v17_1,
    
    model_zhou_inference_v20_1,
    model_zhou_inference_v21_1,

    model_zhou_inference_v24_1,
]

all_feature_models = [
    model_zhou_inference_v15_1,
    model_zhou_inference_v15_2,
    model_zhou_inference_v15_3,
    
    model_zhou_inference_v16_1,

    
    model_zhou_inference_v1_1,
    model_zhou_inference_v1_2,

    model_zhou_inference_v22_1,
    model_zhou_inference_v26_1,
]

# PREDICTION

In [77]:
pad_lens_1branch = [np.load(imu_features_lstm_dir / f"sequence_maxlen_fold{i}.npy", 
                                allow_pickle = True).item() for i in range(5)]
pad_lens_1branch2 = [np.load(imu_features_lstm_dir2 / f"sequence_maxlen_fold{i}.npy", 
                                allow_pickle = True).item() for i in range(5)]
pad_lens_1branch_att = [np.load(imu_features_attention_dir / f"sequence_maxlen_fold{i}.npy", 
                                allow_pickle = True).item() for i in range(5)]
pad_lens_1branch_cnn = [np.load(imu_features_cnn_dir / f"sequence_maxlen_fold{i}.npy", 
                                allow_pickle = True).item() for i in range(5)]
pad_lens_1branch = pad_lens_1branch + pad_lens_1branch2 + pad_lens_1branch_att + \
        pad_lens_1branch_cnn

pad_lens_1branch = [i + 5 for i in pad_lens_1branch]
    
pad_lens_3branch = [np.load(all_features_lstm_dir / f"sequence_maxlen_fold{i}_3branch.npy", 
                                allow_pickle = True).item() for i in range(5)]
pad_lens_3branch2 = [np.load(all_features_lstm_dir2 / f"sequence_maxlen_fold{i}_3branch.npy", 
                                allow_pickle = True).item() for i in range(5)]
pad_lens_3branch_att = [np.load(all_features_attention_dir / f"sequence_maxlen_fold{i}_3branch.npy", 
                                allow_pickle = True).item() for i in range(5)]
pad_lens_3branch_cnn = [np.load(all_features_cnn_dir / f"sequence_maxlen_fold{i}_3branch.npy", 
                                allow_pickle = True).item() for i in range(5)]
pad_lens_3branch = pad_lens_3branch + pad_lens_3branch2 + pad_lens_3branch_att + pad_lens_3branch_cnn

pad_lens_3branch = [i + 5 for i in pad_lens_3branch]

feature_cols_ensemble_32 = np.load(all_features_cnn_dir / "feature_cols.npy", allow_pickle=True).tolist()
tof_cols_ensemble_32 = [c for c in feature_cols_ensemble_32 if c.startswith('tof_')]
imu_cols_ensemble_32 = np.load(imu_features_cnn_dir / "feature_cols_imu.npy", allow_pickle=True).tolist()
gesture_classes = np.load(all_features_cnn_dir / "gesture_classes.npy", allow_pickle=True)

In [78]:
dict_orientation2 = {
                     3: 'Seated Straight',
                        2: 'Seated Lean Non Dom - FACE DOWN',
                        1: 'Lie on Side - Non Dominant',
                     0: 'Lie on Back'
                    }

In [79]:
runi = 0

pp0 = PostProcessor0()
pp1 = PostProcessor0()

def predict(sequence: pl.DataFrame, demographics: pl.DataFrame) -> str:
    global runi
    global pp0, pp1
    
    if runi==0:
        print('start loading models zhou')
        for model in all_feature_models:
            model.get_model()

        for model in imuonly_models:
            model.get_model()
        
        runi += 1

    # predict for the orientations
    pred_orientation_list = []

    

    sequence_copy = sequence.clone()
    demographics_copy = demographics.clone()

    df_seq = sequence.to_pandas()
    dem = demographics.to_pandas()
    
    df_seq1 = sequence.to_pandas()
    dem1 = demographics.to_pandas()
    # <Devin's part>

    if demographics[0, "handedness"] == 0: 
        sequence = preprocess_left_handed(sequence)
        
    thm_tof = sequence[:,["thm_" + str(x) for x in range(1,6)] + 
    ["tof_" + str(x) + "_v"+str(y) for x in range(1,6) for y in range(64)]]

    inputs = torch.tensor(prep_data(sequence, imu_model_cols, scaler_imu).astype(np.float32)).to(device2)

    p_imu_cnngru = predict_inner(inputs, models_imu_cnngru)
    p_imu_cnn = predict_inner(inputs, models_imu_cnn)
    p_imu_densegru  = predict_inner(inputs, models_imu_densegru)
    p_imu_cnngruSE = predict_inner(inputs, models_imu_cnngruSE)
    p_imu_cnnSE = predict_inner(inputs, models_imu_cnnSE)
    inputs = inputs.to(device)
    p_imu_cnngru_resgru = predict_inner(inputs, models_imu_cnngru_resgru)
    p_imu_orientation = predict_inner(inputs, models_imu_orientation)
    p_imu_orientation = p_imu_orientation.detach().cpu().numpy()
    
    p_imu = 0.1*p_imu_cnngru + 0.1*p_imu_cnn + 0.05* p_imu_densegru + 0.1*p_imu_cnngruSE+0.1*p_imu_cnnSE
    
    if thm_tof.null_count().transpose().sum()[0,0] < (thm_tof.shape[1]*thm_tof.shape[0])/2:
        inputs = torch.tensor(prep_data(sequence, all_model_cols, scaler_all).astype(np.float32)).to(device2)

        p_all_cnngru = predict_inner(inputs, models_all_cnngru)
        p_all_cnn = predict_inner(inputs, models_all_cnn)           
        p_all_densegru = predict_inner(inputs, models_all_densegru)
        p_all_cnngruSE = predict_inner(inputs, models_all_cnngruSE)
        p_all_cnnSE = predict_inner(inputs, models_all_cnnSE) 
        inputs = inputs.to(device)
        p_all_cnngru_resgru = predict_inner(inputs, models_all_cnngru_resgru)
        p_all_orientation = predict_inner(inputs, models_all_orientation)
        p_all_orientation = p_all_orientation.detach().cpu().numpy() + p_imu_orientation
        
        p_all =  0.1*p_all_cnngru + 0.1*p_all_cnn + 0.05 * p_all_densegru + 0.1*p_all_cnngruSE + 0.1*p_all_cnnSE
        
        inputs = torch.tensor(prep_data(sequence, all_model_diff_cols, scaler_all_diff).astype(np.float32)).to(device2)
        p_all_cnngru_diff = predict_inner(inputs, models_all_cnngru_diff)
        p_all_cnn_diff = predict_inner(inputs, models_all_cnn_diff)           
        p_all_densegru_diff = predict_inner(inputs, models_all_densegru_diff)
        p_all_cnngruSE_diff = predict_inner(inputs, models_all_cnngruSE_diff)
        p_all_cnnSE_diff = predict_inner(inputs, models_all_cnnSE_diff) 
    
        p_all_diff = 0.1*p_all_cnngru_diff + 0.1*p_all_cnn_diff + 0.05 * p_all_densegru_diff + 0.1*p_all_cnngruSE_diff + 0.1*p_all_cnnSE_diff
        p_all = p_all + p_all_diff
        
        tof_inputs = torch.tensor(prep_tof(sequence, tof_mean, tof_std).astype(np.float32)).to(device2)
        tof_diff_inputs = torch.tensor(prep_tof_diff(sequence, tof_diff_mean, tof_diff_std).astype(np.float32)).to(device2)

        p_tof = predict_inner(tof_inputs, models_all_tof3dnet)
        p_tof_diff = predict_inner(tof_diff_inputs, models_all_tof3dnet_diff)
        p_tof_combined = p_tof+p_tof_diff
        
        final_prediction = (p_all * 0.3  + p_tof_combined * 0.2 + p_imu*0.4) * 0.95 + p_all_cnngru_resgru.to(device2)*0.05
    else:
        final_prediction = p_imu * 0.9 + p_imu_cnngru_resgru.to(device2) * 0.1 
    # </Devin's part>
   
    preds_imu = []
    preds_all = []

    df_seq[ROT_COLS] = handle_quaternion_missing_values(df_seq[ROT_COLS].to_numpy())

    if dem.loc[0, "handedness"] == 0:
        left_handed = df_seq
        cols_to_transform = ["acc_x", "acc_y", "acc_z", "rot_w", "rot_x", "rot_y", "rot_z"]
        left_handed_arr = left_handed[cols_to_transform].to_numpy()
        df_seq.loc[:, cols_to_transform] = mirror_data(left_handed_arr)
        
    # Condition from Devin's part
    if thm_tof.null_count().transpose().sum()[0,0] < (thm_tof.shape[1]*thm_tof.shape[0])/2: 
        df_seq = df_seq.ffill().bfill().fillna(0)
        mat = add_features(df_seq)[feature_cols_ensemble_32]
        mat = mat.to_numpy()
        for i, m in enumerate(models_all_features_old):
            pad = pad_sequences([mat], maxlen=pad_lens_3branch[i], padding='pre', truncating='pre', dtype='float32')
            pad = torch.from_numpy(pad).to(device)
            if i  == 0:
                preds_all = m(pad)
            else:
                preds_all += m(pad)
           
        preds_all_old = preds_all.detach().cpu().numpy()

        if dem1.loc[0, "handedness"] == 0:
            df_seq1 = preprocess_left_handed_pd(df_seq1, dem1)
        df_seq1[ROT_COLS] = handle_quaternion_missing_values(df_seq1[ROT_COLS].to_numpy())
        df_seq1 = df_seq1.ffill().bfill().fillna(0)
        mat = add_features(df_seq1)[feature_cols_ensemble_32]
        mat = mat.to_numpy()

        for i, m in enumerate(models_all_features):
            pad = pad_sequences([mat], maxlen=pad_lens_3branch[i], padding='pre', truncating='pre', dtype='float32')
            pad = torch.from_numpy(pad).to(device)
            if i  == 0:
                preds_all = m(pad)
            else:
                preds_all += m(pad)
           
        preds_all = preds_all.detach().cpu().numpy()

        preds_all = preds_all+preds_all_old
        
        # zhou
        pred_final_list_zhou = []
        for model in all_feature_models:
            gesture_classes_zhou, prediction_proba_zhou = model.predict(sequence_copy, demographics_copy, 0.)

            if type(prediction_proba_zhou) is tuple:
                pred_orientation_list.append(prediction_proba_zhou[1])
                prediction_proba_zhou = prediction_proba_zhou[0]
            else:
                pred_final_list_zhou.append(prediction_proba_zhou)

        pred_final_zhou_all = np.mean(pred_final_list_zhou, 0)

        pred_final_list_zhou = []
        for model in imuonly_models:
            gesture_classes_zhou, prediction_proba_zhou = model.predict(sequence_copy, demographics_copy, 1.)

            if type(prediction_proba_zhou) is tuple:
                pred_orientation_list.append(prediction_proba_zhou[1])
                prediction_proba_zhou = prediction_proba_zhou[0]
            else:
                pred_final_list_zhou.append(prediction_proba_zhou)
        pred_final_zhou_imu = np.mean(pred_final_list_zhou, 0)
        pred_final_zhou = pred_final_zhou_all + 0.2 * pred_final_zhou_imu
        
        pred_orientation_list = np.mean(pred_orientation_list, 0)
        pred_orientation_list = pred_orientation_list*0.975 + p_all_orientation*0.025
        orientation = dict_orientation2.get(pred_orientation_list[0].argmax(-1), 'Seated Lean Non Dom - FACE DOWN')
            
        # BLEND
        blend_preds = 0.43*preds_all + 0.57*final_prediction.cpu().numpy()
        # final_prediction = p_all * 0.3  + p_tof_combined * 0.2 + p_imu*0.4
        
        blend_preds = blend_preds * 0.025 + pred_final_zhou * 0.975

        blend_preds = blend_preds[0]

        print(blend_preds.shape)
        
        try:
            print('preprocessing for allfeat')
            result = pp0.process_prediction(blend_preds, demographics_copy[0, "subject"], orientation)
            print('finish')
            return result
        except:
            idx = int(blend_preds.argmax(1))
            return str(gesture_classes[idx])

    else:
        df_seq = df_seq.ffill().bfill().fillna(0)
        mat = add_features(df_seq)[imu_cols_ensemble_32]
        mat = mat.to_numpy()
        for i, m in enumerate(models_imu_features):
            pad = pad_sequences([mat], maxlen=pad_lens_1branch[i], padding='pre', truncating='pre', dtype='float32')
            pad = torch.from_numpy(pad).to(device)
            if i == 0:
                preds_imu = m(pad)
            else:
                preds_imu += m(pad)
        
        preds_imu = preds_imu.detach().cpu().numpy()

        pred_final_list_zhou = []
        for model in imuonly_models:
            gesture_classes_zhou, prediction_proba_zhou = model.predict(sequence_copy, demographics_copy, 1.)
            if type(prediction_proba_zhou) is tuple:
                pred_orientation_list.append(prediction_proba_zhou[1])
                prediction_proba_zhou = prediction_proba_zhou[0]
            else:
                pred_final_list_zhou.append(prediction_proba_zhou)   
        pred_final_zhou_imu = np.mean(pred_final_list_zhou, 0)
        
        pred_orientation_list = np.mean(pred_orientation_list, 0)
        pred_orientation_list = pred_orientation_list*0.975 + p_imu_orientation*0.025
        orientation = dict_orientation2.get(pred_orientation_list[0].argmax(-1), 'Seated Lean Non Dom - FACE DOWN')

        # BLEND
        blend_preds = 0.5*preds_imu + 0.5*final_prediction.cpu().numpy()
        blend_preds = 0.03*blend_preds + 0.97*pred_final_zhou_imu
        #blend_preds = 0.997*blend_preds + 0.003*pred_spec

        blend_preds = blend_preds[0]

        print(blend_preds.shape)
        
        try:
            print('preprocessing for imuonly')
            result = pp1.process_prediction(blend_preds, demographics_copy[0, "subject"], orientation)
            print('finish')
            return result
        except:
            idx = int(blend_preds.argmax(1))
            return str(gesture_classes[idx])

In [80]:
import kaggle_evaluation.cmi_inference_server
inference_server = kaggle_evaluation.cmi_inference_server.CMIInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(
        data_paths=(
            "/kaggle/input/cmi-detect-behavior/test_with_nulls.csv",
            '/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv',
            #"/kaggle/input/cmi-detect-behavior-with-sensor-data/test.csv",
            #"/kaggle/input/cmi-detect-behavior-with-sensor-data/test_demographics.csv",
        )
    )

start loading models zhou
▶ INFERENCE MODE – loading artefacts from /kaggle/input/cmi3v58/42
  model, scaler, pads loaded – ready for evaluation
▶ INFERENCE MODE – loading artefacts from /kaggle/input/cmi3v58/6665252
  model, scaler, pads loaded – ready for evaluation
▶ INFERENCE MODE – loading artefacts from /kaggle/input/cmi3v58/88885252
  model, scaler, pads loaded – ready for evaluation
▶ INFERENCE MODE – loading artefacts from /kaggle/input/cmi3v61
  model, scaler, pads loaded – ready for evaluation
▶ INFERENCE MODE – loading artefacts from /kaggle/input/cmi3v35
  model, scaler, pads loaded – ready for evaluation
▶ INFERENCE MODE – loading artefacts from /kaggle/input/cmi3v38
  model, scaler, pads loaded – ready for evaluation
▶ INFERENCE MODE – loading artefacts from /kaggle/input/cmi3v79
  model, scaler, pads loaded – ready for evaluation
▶ INFERENCE MODE – loading artefacts from /kaggle/input/cmi3v90
  model, scaler, pads loaded – ready for evaluation
▶ INFERENCE MODE – loading

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

(18,)
preprocessing for allfeat
finish
